In [1]:
import sys
sys.path.insert(0, 'DAGMA-DCE/src')


import time
import itertools
import math
import warnings
from typing import Dict, List, Set, Tuple

import numpy as np
import pandas as pd
import torch
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import networkx as nx
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

# ── DAGMA-DCE imports ─────────────────────────────────────────────
try:
    from DagmaDCE import nonlinear_dce
    HAS_DCE = True
    # linear_dce may not exist depending on version - try it
    try:
        from DagmaDCE import linear_dce
        HAS_DCE_LINEAR = True
    except ImportError:
        linear_dce = None
        HAS_DCE_LINEAR = False
    try:
        from DagmaDCE import utils as dce_utils
    except ImportError:
        dce_utils = None
    print("ok DagmaDCE available")
except ImportError as e:
    HAS_DCE = False
    HAS_DCE_LINEAR = False
    print(f"x DagmaDCE not found: {e}")
    print("  Run: git clone https://github.com/DanWaxman/DAGMA-DCE && cd DAGMA-DCE/src && pip install -e . && cd ../..")

# ── Standard DAGMA for comparison ────────────────────────────────
try:
    from dagma.linear import DagmaLinear
    HAS_DAGMA = True
    print("ok dagma (standard linear) available")
except ImportError:
    HAS_DAGMA = False
    print("x dagma not found — run: pip install dagma")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device: {device}")

# Config
CONFIG = {
    'SOIL_DATA_PATH':         '../../../../datasets/data_complete.csv',
    'SOIL_GROUND_TRUTH_PATH': '../../../../datasets/ground_truth.csv',
    'YIELD_DATA_PATH':        '../../../../datasets/yield_regression_preprocessed.npz',
    'TEST_SIZE':     0.20,
    'RANDOM_SEED':   42,
    # DAGMA-DCE hyperparameters (from paper defaults)
    'DCE_LAMBDA1':   3.5e-2,   # L1 sparsity on edges
    'DCE_LAMBDA2':   5e-3,     # L2 weight decay on MLP
    'DCE_LR':        2e-4,     # learning rate
    'DCE_MU_FACTOR': 0.1,
    'DCE_MU_INIT':   0.1,
    'DCE_T':         4,        # number of outer augmented Lagrangian steps
    'DCE_WARM_ITER': 7_000,
    'DCE_MAX_ITER':  8_000,
    # Causal SHAP
    'YIELD_MAX_PERM': 500,
    'YIELD_M':        128,
    'YIELD_K':        100,
    'SOIL_M':         128,
    'SOIL_K':         100,
}

SOIL_FEATURES   = ['Organic Carbon','Nitrogen','Phosphorus','Potassium',
                    'pH','Zinc','Electrical Conductivity']
SOIL_COL_MAP    = {'OC':'Organic Carbon','N':'Nitrogen','P':'Phosphorus',
                   'K':'Potassium','PH':'pH','ZN':'Zinc','EC':'Electrical Conductivity'}
SOIL_TRUE_EDGES = {('Organic Carbon','pH'), ('pH','Zinc')}

YIELD_FEATURES  = ['Nitrogen','Irrigation','Precipitation','Herbicide']
YIELD_TRUE_EDGES = set()
YIELD_TRANSLOG  = {
    'betaN':0.35,'betaI':0.50,'betaP':0.25,'betaH':0.05,
    'betaNN':-0.02,'betaII':-0.03,'betaPP':-0.01,'betaHH':-0.005,
    'betaNI':0.04,'betaNP':0.02,'betaNH':0.01,
    'betaIP':0.03,'betaIH':0.01,'betaPH':0.005,
}

np.random.seed(CONFIG['RANDOM_SEED'])
rng = np.random.default_rng(CONFIG['RANDOM_SEED'])
torch.manual_seed(CONFIG['RANDOM_SEED'])


# Ground-truth gamma
def compute_soil_gamma(path: str) -> Dict[str, float]:
    gt  = pd.read_csv(path)
    net = pd.DataFrame({
        'Organic Carbon':          gt['OC_base']  - gt['OC_penalty'],
        'Nitrogen':                gt['N_base']   - gt['N_penalty'],
        'Phosphorus':              gt['P_base']   - gt['P_penalty'],
        'Potassium':               gt['K_base']   - gt['K_penalty'],
        'pH':                     -gt['PH_penalty'],
        'Zinc':                    gt['ZN_base']  - gt['ZN_penalty'],
        'Electrical Conductivity': -gt['EC_penalty'],
    })
    m = net.abs().mean()
    return (m / m.sum()).to_dict()


def compute_yield_gamma(X_raw: pd.DataFrame) -> Dict[str, float]:
    c  = YIELD_TRANSLOG
    lm = {f: float(np.log(X_raw[f]).mean()) for f in YIELD_FEATURES}
    lnN, lnI, lnP, lnH = lm['Nitrogen'], lm['Irrigation'], lm['Precipitation'], lm['Herbicide']
    e = {
        'Nitrogen':      c['betaN']  + c['betaNN']*lnN + c['betaNI']*lnI + c['betaNP']*lnP + c['betaNH']*lnH,
        'Irrigation':    c['betaI']  + c['betaII']*lnI + c['betaNI']*lnN + c['betaIP']*lnP + c['betaIH']*lnH,
        'Precipitation': c['betaP']  + c['betaPP']*lnP + c['betaNP']*lnN + c['betaIP']*lnI + c['betaPH']*lnH,
        'Herbicide':     c['betaH']  + c['betaHH']*lnH + c['betaNH']*lnN + c['betaIH']*lnI + c['betaPH']*lnP,
    }
    tot = sum(e.values())
    return {f: v/tot for f, v in e.items()}


def normalise(g: Dict[str, float], features: List[str]) -> Dict[str, float]:
    g   = {f: max(v, 0.0) for f, v in g.items()}
    tot = sum(g.values())
    return {f: v/tot for f, v in g.items()} if tot > 1e-12 else {f: 1.0/len(features) for f in features}


# Dagma-dce runner
def run_dagma_dce_nonlinear(X: pd.DataFrame, y: pd.Series,
                             features: List[str], target_name: str,
                             cfg: dict) -> Tuple[np.ndarray, Dict[str, float], nx.DiGraph]:
    """
    Runs DAGMA-DCE with MLP backbone (non-parametric).
    The adjacency matrix W_est[i, j] = DCE of variable i on variable j
    = ∂E[Xⱼ]/∂Xᵢ  (differential causal effect, directly interpretable).

    For gamma: take W_est[:, target_idx] — the DCE of each feature on target.
    """
    if not HAS_DCE:
        print("  x DagmaDCE not available")
        return None, {f: 1.0/len(features) for f in features}, nx.DiGraph()

    all_cols   = features + [target_name]
    n_vars     = len(all_cols)
    X_full     = X.copy()
    X_full[target_name] = y.values if hasattr(y, 'values') else y
    data_np    = X_full[all_cols].values.astype(np.float64)
    # DagmaDCE model weights default to float64 (Double) - tensor must match
    data       = torch.tensor(data_np, dtype=torch.float64)  # CPU - matches DagmaDCE internal tensors

    print(f"  Running DAGMA-DCE (MLP, non-parametric) on {n_vars} variables...")

    # MLP dims: [d, hidden, 1] - hidden = 2×d is a good default
    hidden = max(10, 2 * n_vars)
    eq_model = nonlinear_dce.DagmaMLP_DCE(
        dims=[n_vars, hidden, 1], bias=True
    )  # keep on CPU - DagmaDCE internal identity matrix is on CPU

    model  = nonlinear_dce.DagmaDCE(eq_model, use_mse_loss=True)
    W_est  = model.fit(
        data,
        lambda1   = cfg['DCE_LAMBDA1'],
        lambda2   = cfg['DCE_LAMBDA2'],
        lr        = cfg['DCE_LR'],
        mu_factor = cfg['DCE_MU_FACTOR'],
        mu_init   = cfg['DCE_MU_INIT'],
        T         = cfg['DCE_T'],
        warm_iter = cfg['DCE_WARM_ITER'],
        max_iter  = cfg['DCE_MAX_ITER'],
    )
    # W_est shape: (n_vars, n_vars), W_est[i, j] = DCE of i -> j

    target_idx  = all_cols.index(target_name)
    raw_effects = {}
    for i, f in enumerate(features):
        raw_effects[f] = abs(float(W_est[i, target_idx]))

    dce_gamma = normalise(raw_effects, features)

    # Build inter-feature DAG (exclude target node)
    G = nx.DiGraph()
    G.add_nodes_from(features)
    threshold = 0.05
    for i, fi in enumerate(features):
        for j, fj in enumerate(features):
            if i != j and abs(W_est[i, j]) > threshold:
                G.add_edge(fi, fj, weight=float(W_est[i, j]))

    return W_est, dce_gamma, G


# run_dagma_dce_linear removed - linear_dce not in this DagmaDCE version


def run_standard_dagma(X: pd.DataFrame, y: pd.Series,
                        features: List[str], target_name: str) -> Tuple[np.ndarray, Dict[str, float], nx.DiGraph]:
    """Standard DAGMA linear — same as previous experiment, for direct comparison."""
    if not HAS_DAGMA:
        return None, {f: 1.0/len(features) for f in features}, nx.DiGraph()
    all_cols = features + [target_name]
    X_full   = X.copy()
    X_full[target_name] = y.values if hasattr(y, 'values') else y
    data     = X_full[all_cols].values.astype(float)
    model    = DagmaLinear(loss_type='l2')
    W        = model.fit(data, lambda1=0.02)
    target_idx  = all_cols.index(target_name)
    raw_effects = {f: abs(float(W[i, target_idx])) for i, f in enumerate(features)}
    dagma_gamma = normalise(raw_effects, features)
    G = nx.DiGraph()
    G.add_nodes_from(features)
    for i, fi in enumerate(features):
        for j, fj in enumerate(features):
            if i != j and abs(W[i, j]) > 0.05:
                G.add_edge(fi, fj, weight=float(W[i, j]))
    return W, dagma_gamma, G


# Graph + gamma comparison utils
def compare_graphs(recovered: nx.DiGraph, true_edges: set) -> dict:
    rec = set(recovered.edges())
    tp  = len(rec & true_edges); fp = len(rec - true_edges); fn = len(true_edges - rec)
    shd = fp + fn
    prec   = tp/(tp+fp) if (tp+fp) > 0 else 0.0
    recall = tp/(tp+fn) if (tp+fn) > 0 else 0.0
    f1     = 2*prec*recall/(prec+recall) if (prec+recall) > 0 else 0.0
    return {'TP':tp,'FP':fp,'FN':fn,'SHD':shd,
            'Precision':round(prec,3),'Recall':round(recall,3),'F1':round(f1,3),
            'Recovered':list(rec),'Missing':list(true_edges-rec),'Extra':list(rec-true_edges)}


def gamma_metrics(true_gamma: Dict[str,float], est_gamma: Dict[str,float],
                  features: List[str]) -> dict:
    gt = np.array([true_gamma[f] for f in features])
    eg = np.array([est_gamma[f]  for f in features])
    pr,_ = pearsonr(eg, gt); sr,_ = spearmanr(eg, gt)
    return {'Gamma_Pearson':round(float(pr),4),'Gamma_Spearman':round(float(sr),4),
            'Gamma_RMSE':round(float(np.sqrt(np.mean((eg-gt)**2))),4)}


def attribution_metrics(true_gamma: Dict[str,float], global_abs: pd.Series,
                        features: List[str]) -> dict:
    gt  = np.array([true_gamma[f]        for f in features])
    raw = np.array([global_abs.get(f,0)  for f in features])
    sv  = raw/raw.sum() if raw.sum() > 1e-12 else raw
    pr,_ = pearsonr(sv,gt); sr,_ = spearmanr(sv,gt)
    cos  = float(np.dot(sv,gt)/(np.linalg.norm(sv)*np.linalg.norm(gt)+1e-12))
    rmse = float(np.sqrt(np.mean((sv-gt)**2)))
    return {'Attr_Pearson':round(float(pr),4),'Attr_Spearman':round(float(sr),4),
            'Attr_Cosine':round(cos,4),'Attr_RMSE':round(rmse,4)}


class LinearMechanisms:
    def __init__(self, dag: nx.DiGraph, features: List[str]):
        self.dag = dag; self.features = features
        self.mech: Dict[str,dict] = {}
        self.topo = list(nx.topological_sort(dag))

    def fit(self, X: pd.DataFrame) -> 'LinearMechanisms':
        for var in self.features:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {'parents':[],'intercept':float(X[var].mean()),
                                  'coef':None,'sigma':float(X[var].std(ddof=0))}
            else:
                lr    = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mech[var] = {'parents':parents,'intercept':float(lr.intercept_),
                                  'coef':lr.coef_.astype(float),'sigma':float(np.std(resid,ddof=0))}
        return self

    def sample_batch(self, S: Set[str], x: pd.Series, M: int) -> np.ndarray:
        X_out = np.empty((M, len(self.features)))
        fi    = {f: i for i, f in enumerate(self.features)}
        sim   = {}
        for var in self.topo:
            vi = fi[var]
            if var in S:
                val=float(x[var]); X_out[:,vi]=val; sim[var]=np.full(M,val); continue
            m = self.mech[var]
            if not m['parents']:
                mu = np.full(M, m['intercept'])
            else:
                pm = np.stack([sim[p] if p in sim else np.full(M,float(x[p])) for p in m['parents']], axis=1)
                mu = m['intercept'] + pm @ m['coef']
            X_out[:,vi] = mu if m['sigma']<1e-8 else rng.normal(mu, m['sigma'], size=M)
            sim[var] = X_out[:,vi]
        return X_out


class PermutationEngine:
    def __init__(self, model, sem, gamma, X_train):
        self.model=model; self.features=list(X_train.columns)
        self.gamma=gamma; self.sem=sem
        self._Ef=float(model.predict(X_train.values).mean())

    def _vc(self, S, x, M): return float(np.mean(self.model.predict(self.sem.sample_batch(S,x,M))))

    def explain_instance(self, x, n_perms, M):
        fx=float(self.model.predict(x.values.reshape(1,-1))[0])
        phi=np.zeros(len(self.features))
        for _ in range(n_perms):
            perm=rng.permutation(self.features); S=set(); v_prev=self._vc(S,x,M)
            for feat in perm:
                Sn=S|{feat}; vc=self._vc(Sn,x,M)
                phi[self.features.index(feat)]+=self.gamma[feat]*(vc-v_prev)
                v_prev,S=vc,Sn
        phi/=n_perms
        s=float(phi.sum())
        if abs(s)>1e-12: phi=phi*((fx-self._Ef)/s)
        return pd.Series(phi,index=self.features)

    def explain_global(self, X_test, k, n_perms, M):
        k=min(k,len(X_test)); idx=rng.choice(len(X_test),size=k,replace=False)
        return pd.concat([self.explain_instance(X_test.iloc[i],n_perms,M) for i in idx],axis=1).abs().mean(axis=1)


class ExactEngine:
    def __init__(self, model, sem, gamma, X_train):
        self.model=model; self.features=list(X_train.columns)
        self.n=len(self.features); self.gamma=gamma; self.sem=sem
        self._Ef=float(model.predict(X_train.values).mean())
        self._kw={sz: float(math.factorial(sz)*math.factorial(self.n-sz-1)/math.factorial(self.n))
                  for sz in range(self.n)}

    def explain_instance(self, x, M):
        fx=float(self.model.predict(x.values.reshape(1,-1))[0])
        phi=np.zeros(self.n)
        cache={frozenset(sub): float(np.mean(self.model.predict(self.sem.sample_batch(frozenset(sub),x,M))))
               for r in range(self.n+1) for sub in itertools.combinations(self.features,r)}
        for idx,feat in enumerate(self.features):
            rest=[f for f in self.features if f!=feat]
            for r in range(self.n):
                for sub in itertools.combinations(rest,r):
                    S=frozenset(sub)
                    phi[idx]+=self._kw[len(sub)]*self.gamma[feat]*(cache[S|{feat}]-cache[S])
        s=float(phi.sum())
        if abs(s)>1e-12: phi=phi*((fx-self._Ef)/s)
        return pd.Series(phi,index=self.features)

    def explain_global(self, X_test, k, M):
        k=min(k,len(X_test)); phi_list=[]
        for i in range(k):
            if (i+1)%20==0: print(f"      instance {i+1}/{k} ...",flush=True)
            phi_list.append(self.explain_instance(X_test.iloc[i],M))
        return pd.concat(phi_list,axis=1).abs().mean(axis=1)


def run_causal_shap(model, gamma, dag, Xtr, Xte, engine_type, cfg) -> pd.Series:
    sem = LinearMechanisms(dag, list(Xtr.columns)).fit(Xtr)
    if engine_type == 'permutation':
        e = PermutationEngine(model, sem, gamma, Xtr)
        return e.explain_global(Xte, cfg['YIELD_K'], cfg['YIELD_MAX_PERM'], cfg['YIELD_M'])
    else:
        e = ExactEngine(model, sem, gamma, Xtr)
        return e.explain_global(Xte, cfg['SOIL_K'], cfg['SOIL_M'])


# Dataset runner
def run_dataset(name: str, features: List[str], true_edges: set,
                true_gamma: Dict[str,float], Xtr: pd.DataFrame, Xte: pd.DataFrame,
                ytr: np.ndarray, engine_type: str, cfg: dict) -> pd.DataFrame:

    print(f"\n{'='*80}\nDATASET: {name.upper()}\n{'='*80}")
    print(f"  True γ: { {f: round(v,4) for f,v in true_gamma.items()} }")

    target_name = f'{name}_target'
    results     = []

    # Train XGBoost once
    model = XGBRegressor(n_estimators=160, max_depth=5, learning_rate=0.08,
                         subsample=0.9, colsample_bytree=0.9,
                         random_state=cfg['RANDOM_SEED'], verbosity=0,
                         n_jobs=1, tree_method='hist')
    model.fit(Xtr.values, ytr)

    true_dag = nx.DiGraph(); true_dag.add_nodes_from(features)
    for u,v in true_edges: true_dag.add_edge(u,v)

    # ── 1. DAGMA-DCE Non-linear (MLP) ───────────────────────────
    print("\n--- Method 1: DAGMA-DCE (non-linear MLP) ---")
    t0 = time.time()
    W_nl, gamma_nl, dag_nl = run_dagma_dce_nonlinear(Xtr, ytr, features, target_name, cfg)
    t_nl = time.time() - t0

    if W_nl is not None:
        gc_nl = compare_graphs(dag_nl, true_edges)
        gm_nl = gamma_metrics(true_gamma, gamma_nl, features)
        print(f"  γ recovered: { {f: round(v,4) for f,v in gamma_nl.items()} }")
        print(f"  Gamma Pearson: {gm_nl['Gamma_Pearson']} | Graph SHD: {gc_nl['SHD']} | F1: {gc_nl['F1']}")
        print(f"  Running Causal SHAP with DCE-MLP γ + DCE-MLP graph...")
        g_abs = run_causal_shap(model, gamma_nl, dag_nl, Xtr, Xte, engine_type, cfg)
        am_nl = attribution_metrics(true_gamma, g_abs, features)
        print(f"  Attribution Pearson={am_nl['Attr_Pearson']} | RMSE={am_nl['Attr_RMSE']}")
        results.append({'Method':'DAGMA-DCE MLP (graph+gamma)',**gm_nl,**gc_nl,**am_nl,'Time_s':round(t_nl,1)})

        # DCE-MLP γ + True graph
        print(f"  Running Causal SHAP with DCE-MLP γ + TRUE graph...")
        g_abs2 = run_causal_shap(model, gamma_nl, true_dag, Xtr, Xte, engine_type, cfg)
        am_nl2 = attribution_metrics(true_gamma, g_abs2, features)
        print(f"  Attribution Pearson={am_nl2['Attr_Pearson']} | RMSE={am_nl2['Attr_RMSE']}")
        results.append({'Method':'DAGMA-DCE MLP gamma + True graph',
                        **gm_nl,**gc_nl,**am_nl2,'Time_s':round(t_nl,1)})

    # ── 2. Standard DAGMA (from previous experiment) ─────────────
    print("\n--- Method 2: Standard DAGMA (linear, previous experiment) ---")
    t0 = time.time()
    W_d, gamma_d, dag_d = run_standard_dagma(Xtr, ytr, features, target_name)
    t_d = time.time() - t0

    if W_d is not None:
        gc_d = compare_graphs(dag_d, true_edges)
        gm_d = gamma_metrics(true_gamma, gamma_d, features)
        print(f"  γ recovered: { {f: round(v,4) for f,v in gamma_d.items()} }")
        print(f"  Gamma Pearson: {gm_d['Gamma_Pearson']} | Graph SHD: {gc_d['SHD']} | F1: {gc_d['F1']}")
        g_abs4 = run_causal_shap(model, gamma_d, dag_d, Xtr, Xte, engine_type, cfg)
        am_d   = attribution_metrics(true_gamma, g_abs4, features)
        print(f"  Attribution Pearson={am_d['Attr_Pearson']} | RMSE={am_d['Attr_RMSE']}")
        results.append({'Method':'Standard DAGMA (prev experiment)',**gm_d,**gc_d,**am_d,'Time_s':round(t_d,1)})

    # ── 3. Ceiling: true γ + true graph ─────────────────────────
    print("\n--- Ceiling: True gamma + True graph ---")
    g_abs5 = run_causal_shap(model, true_gamma, true_dag, Xtr, Xte, engine_type, cfg)
    am_ceil = attribution_metrics(true_gamma, g_abs5, features)
    print(f"  Attribution Pearson={am_ceil['Attr_Pearson']} | RMSE={am_ceil['Attr_RMSE']}")
    results.append({'Method':'True gamma + True graph (ceiling)',
                    'Gamma_Pearson':1.0,'Gamma_Spearman':1.0,'Gamma_RMSE':0.0,
                    'TP':len(true_edges),'FP':0,'FN':0,'SHD':0,'F1':1.0,
                    **am_ceil,'Time_s':0.0})

    df = pd.DataFrame(results)
    disp = ['Method','Gamma_Pearson','SHD','F1','Attr_Pearson','Attr_Spearman','Attr_RMSE']
    print(f"\n{'='*80}\nRESULTS SUMMARY — {name.upper()}\n{'='*80}")
    print(df[disp].to_string(index=False))
    return df


# Main
def main():
    print("\n" + "="*80)
    print("DAGMA-DCE vs DAGMA vs CEILING — Soil & Yield")
    print("="*80)

    if not HAS_DCE:
        print("\nWarning: DagmaDCE not installed. Run setup commands at top of script.")
        print("  Continuing with DAGMA-DCE MLP and standard DAGMA only.\n")

    # ── yield ────────────────────────────────────────────────────
    data        = np.load(CONFIG['YIELD_DATA_PATH'], allow_pickle=True)
    X_raw_train = pd.DataFrame(data['X_train'], columns=YIELD_FEATURES)
    X_raw_test  = pd.DataFrame(data['X_test'],  columns=YIELD_FEATURES)
    y_train_y   = data['y_train'].astype(float)
    true_gamma_y = compute_yield_gamma(X_raw_train)

    scaler_y = StandardScaler()
    Xtr_y = pd.DataFrame(scaler_y.fit_transform(X_raw_train), columns=YIELD_FEATURES)
    Xte_y = pd.DataFrame(scaler_y.transform(X_raw_test),      columns=YIELD_FEATURES)

    df_y = run_dataset('Yield', YIELD_FEATURES, YIELD_TRUE_EDGES, true_gamma_y,
                        Xtr_y, Xte_y, y_train_y, 'permutation', CONFIG)
    df_y.to_csv('dagma_dce_results_yield.csv', index=False)

    # ── soil ─────────────────────────────────────────────────────
    dc       = pd.read_csv(CONFIG['SOIL_DATA_PATH']).rename(columns=SOIL_COL_MAP)
    Xtr_s_raw, Xte_s_raw, ytr_s, _ = train_test_split(
        dc[SOIL_FEATURES], dc['SPI_CONTINUOUS'],
        test_size=CONFIG['TEST_SIZE'], random_state=CONFIG['RANDOM_SEED'], shuffle=True)
    true_gamma_s = compute_soil_gamma(CONFIG['SOIL_GROUND_TRUTH_PATH'])

    scaler_s = StandardScaler()
    Xtr_s = pd.DataFrame(scaler_s.fit_transform(Xtr_s_raw), columns=SOIL_FEATURES, index=Xtr_s_raw.index)
    Xte_s = pd.DataFrame(scaler_s.transform(Xte_s_raw),     columns=SOIL_FEATURES, index=Xte_s_raw.index)

    df_s = run_dataset('Soil', SOIL_FEATURES, SOIL_TRUE_EDGES, true_gamma_s,
                        Xtr_s, Xte_s, ytr_s.values, 'exact', CONFIG)
    df_s.to_csv('dagma_dce_results_soil.csv', index=False)

    print("\nSaved: dagma_dce_results_yield.csv, dagma_dce_results_soil.csv")


if __name__ == '__main__':
    main()

✓ DagmaDCE available
✓ dagma (standard linear) available
  Device: cuda

DAGMA-DCE vs DAGMA vs CEILING — Soil & Yield

DATASET: YIELD
  True γ: {'Nitrogen': 0.3118, 'Irrigation': 0.3727, 'Precipitation': 0.2457, 'Herbicide': 0.0697}

--- Method 1: DAGMA-DCE (non-linear MLP) ---
  Running DAGMA-DCE (MLP, non-parametric) on 5 variables...


  γ recovered: {'Nitrogen': 0.25, 'Irrigation': 0.25, 'Precipitation': 0.25, 'Herbicide': 0.25}
  Gamma Pearson: nan | Graph SHD: 0 | F1: 0.0
  Running Causal SHAP with DCE-MLP γ + DCE-MLP graph...
  Attribution Pearson=0.8737 | RMSE=0.0788
  Running Causal SHAP with DCE-MLP γ + TRUE graph...
  Attribution Pearson=0.9055 | RMSE=0.0682

--- Method 2: Standard DAGMA (linear, previous experiment) ---


  γ recovered: {'Nitrogen': 0.2421, 'Irrigation': 0.3778, 'Precipitation': 0.2727, 'Herbicide': 0.1074}
  Gamma Pearson: 0.9326 | Graph SHD: 3 | F1: 0.0
  Attribution Pearson=0.9116 | RMSE=0.0809

--- Ceiling: True gamma + True graph ---
  Attribution Pearson=0.8651 | RMSE=0.1191

RESULTS SUMMARY — YIELD
                           Method  Gamma_Pearson  SHD  F1  Attr_Pearson  Attr_Spearman  Attr_RMSE
      DAGMA-DCE MLP (graph+gamma)            NaN    0 0.0        0.8737            0.8     0.0788
 DAGMA-DCE MLP gamma + True graph            NaN    0 0.0        0.9055            0.8     0.0682
 Standard DAGMA (prev experiment)         0.9326    3 0.0        0.9116            0.8     0.0809
True gamma + True graph (ceiling)         1.0000    0 1.0        0.8651            0.8     0.1191

DATASET: SOIL
  True γ: {'Organic Carbon': 0.0622, 'Nitrogen': 0.4012, 'Phosphorus': 0.0762, 'Potassium': 0.2588, 'pH': 0.0886, 'Zinc': 0.0147, 'Electrical Conductivity': 0.0984}

--- Method 1: DAGMA-DCE

  γ recovered: {'Organic Carbon': 0.1543, 'Nitrogen': 0.1473, 'Phosphorus': 0.0928, 'Potassium': 0.1565, 'pH': 0.2095, 'Zinc': 0.0773, 'Electrical Conductivity': 0.1623}
  Gamma Pearson: 0.2471 | Graph SHD: 9 | F1: 0.182
  Running Causal SHAP with DCE-MLP γ + DCE-MLP graph...
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  Attribution Pearson=0.8285 | RMSE=0.0715
  Running Causal SHAP with DCE-MLP γ + TRUE graph...
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  Attribution Pearson=0.9127 | RMSE=0.0536

--- Method 2: Standard DAGMA (linear, previous experiment) ---


  γ recovered: {'Organic Carbon': 0.0578, 'Nitrogen': 0.3589, 'Phosphorus': 0.0517, 'Potassium': 0.2169, 'pH': 0.178, 'Zinc': 0.0236, 'Electrical Conductivity': 0.1131}
  Gamma Pearson: 0.9466 | Graph SHD: 1 | F1: 0.667
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  Attribution Pearson=0.9869 | RMSE=0.0504

--- Ceiling: True gamma + True graph ---
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  Attribution Pearson=0.9869 | RMSE=0.0642

RESULTS SUMMARY — SOIL
                           Method  Gamma_Pearson  SHD    F1  Attr_Pearson  Attr_Spearman  Attr_RMSE
      DAGMA-DCE MLP (graph+gamma)         0.2471    9 0.182        0.8285         0.8571     0.0715
 DAGMA-DCE MLP gamma + True graph         0.2471    9 0.182        0.9127         0.8214     0.0536
 Standard DAGMA (prev experiment)         0.9466    1 0.667       

In [2]:
import sys
sys.path.insert(0, 'DAGMA-DCE/src')


import time
import itertools
import math
import warnings
from typing import Dict, List, Set, Tuple

import numpy as np
import pandas as pd
import torch
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import networkx as nx
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

# ── DAGMA-DCE imports ─────────────────────────────────────────────
try:
    from DagmaDCE import nonlinear_dce
    HAS_DCE = True
    # linear_dce may not exist depending on version - try it
    try:
        from DagmaDCE import linear_dce
        HAS_DCE_LINEAR = True
    except ImportError:
        linear_dce = None
        HAS_DCE_LINEAR = False
    try:
        from DagmaDCE import utils as dce_utils
    except ImportError:
        dce_utils = None
    print("ok DagmaDCE available")
except ImportError as e:
    HAS_DCE = False
    HAS_DCE_LINEAR = False
    print(f"x DagmaDCE not found: {e}")
    print("  Run: git clone https://github.com/DanWaxman/DAGMA-DCE && cd DAGMA-DCE/src && pip install -e . && cd ../..")

# ── Standard DAGMA for comparison ────────────────────────────────
try:
    from dagma.linear import DagmaLinear
    HAS_DAGMA = True
    print("ok dagma (standard linear) available")
except ImportError:
    HAS_DAGMA = False
    print("x dagma not found — run: pip install dagma")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device: {device}")

# Config
CONFIG = {
    'SOIL_DATA_PATH':         '../../../../datasets/data_complete.csv',
    'SOIL_GROUND_TRUTH_PATH': '../../../../datasets/ground_truth.csv',
    'YIELD_DATA_PATH':        '../../../../datasets/yield_regression_preprocessed.npz',
    'TEST_SIZE':     0.20,
    'RANDOM_SEED':   42,
    # DAGMA-DCE hyperparameters (from paper defaults)
    'DCE_LAMBDA1':   3.5e-2,   # L1 sparsity on edges
    'DCE_LAMBDA2':   5e-3,     # L2 weight decay on MLP
    'DCE_LR':        2e-4,     # learning rate
    'DCE_MU_FACTOR': 0.1,
    'DCE_MU_INIT':   0.1,
    'DCE_T':         4,        # number of outer augmented Lagrangian steps
    'DCE_WARM_ITER': 7_000,
    'DCE_MAX_ITER':  8_000,
    # Causal SHAP
    'YIELD_MAX_PERM': 500,
    'YIELD_M':        128,
    'YIELD_K':        100,
    'SOIL_M':         128,
    'SOIL_K':         100,
}

SOIL_FEATURES   = ['Organic Carbon','Nitrogen','Phosphorus','Potassium',
                    'pH','Zinc','Electrical Conductivity']
SOIL_COL_MAP    = {'OC':'Organic Carbon','N':'Nitrogen','P':'Phosphorus',
                   'K':'Potassium','PH':'pH','ZN':'Zinc','EC':'Electrical Conductivity'}
SOIL_TRUE_EDGES = {('Organic Carbon','pH'), ('pH','Zinc')}

YIELD_FEATURES  = ['Nitrogen','Irrigation','Precipitation','Herbicide']
YIELD_TRUE_EDGES = set()
YIELD_TRANSLOG  = {
    'betaN':0.35,'betaI':0.50,'betaP':0.25,'betaH':0.05,
    'betaNN':-0.02,'betaII':-0.03,'betaPP':-0.01,'betaHH':-0.005,
    'betaNI':0.04,'betaNP':0.02,'betaNH':0.01,
    'betaIP':0.03,'betaIH':0.01,'betaPH':0.005,
}

np.random.seed(CONFIG['RANDOM_SEED'])
rng = np.random.default_rng(CONFIG['RANDOM_SEED'])
torch.manual_seed(CONFIG['RANDOM_SEED'])


# Ground-truth gamma
def compute_soil_gamma(path: str) -> Dict[str, float]:
    gt  = pd.read_csv(path)
    net = pd.DataFrame({
        'Organic Carbon':          gt['OC_base']  - gt['OC_penalty'],
        'Nitrogen':                gt['N_base']   - gt['N_penalty'],
        'Phosphorus':              gt['P_base']   - gt['P_penalty'],
        'Potassium':               gt['K_base']   - gt['K_penalty'],
        'pH':                     -gt['PH_penalty'],
        'Zinc':                    gt['ZN_base']  - gt['ZN_penalty'],
        'Electrical Conductivity': -gt['EC_penalty'],
    })
    m = net.abs().mean()
    return (m / m.sum()).to_dict()


def compute_yield_gamma(X_raw: pd.DataFrame) -> Dict[str, float]:
    c  = YIELD_TRANSLOG
    lm = {f: float(np.log(X_raw[f]).mean()) for f in YIELD_FEATURES}
    lnN, lnI, lnP, lnH = lm['Nitrogen'], lm['Irrigation'], lm['Precipitation'], lm['Herbicide']
    e = {
        'Nitrogen':      c['betaN']  + c['betaNN']*lnN + c['betaNI']*lnI + c['betaNP']*lnP + c['betaNH']*lnH,
        'Irrigation':    c['betaI']  + c['betaII']*lnI + c['betaNI']*lnN + c['betaIP']*lnP + c['betaIH']*lnH,
        'Precipitation': c['betaP']  + c['betaPP']*lnP + c['betaNP']*lnN + c['betaIP']*lnI + c['betaPH']*lnH,
        'Herbicide':     c['betaH']  + c['betaHH']*lnH + c['betaNH']*lnN + c['betaIH']*lnI + c['betaPH']*lnP,
    }
    tot = sum(e.values())
    return {f: v/tot for f, v in e.items()}


def normalise(g: Dict[str, float], features: List[str]) -> Dict[str, float]:
    g   = {f: max(v, 0.0) for f, v in g.items()}
    tot = sum(g.values())
    return {f: v/tot for f, v in g.items()} if tot > 1e-12 else {f: 1.0/len(features) for f in features}


# Dagma-dce runner
def run_dagma_dce_nonlinear(X: pd.DataFrame, y: pd.Series,
                             features: List[str], target_name: str,
                             cfg: dict) -> Tuple[np.ndarray, Dict[str, float], nx.DiGraph]:
    """
    Runs DAGMA-DCE with MLP backbone (non-parametric).
    The adjacency matrix W_est[i, j] = DCE of variable i on variable j
    = ∂E[Xⱼ]/∂Xᵢ  (differential causal effect, directly interpretable).

    For gamma: take W_est[:, target_idx] — the DCE of each feature on target.
    """
    if not HAS_DCE:
        print("  x DagmaDCE not available")
        return None, {f: 1.0/len(features) for f in features}, nx.DiGraph()

    all_cols   = features + [target_name]
    n_vars     = len(all_cols)
    X_full     = X.copy()
    X_full[target_name] = y.values if hasattr(y, 'values') else y
    data_np    = X_full[all_cols].values.astype(np.float64)
    # DagmaDCE model weights default to float64 (Double) - tensor must match
    data       = torch.tensor(data_np, dtype=torch.float64)  # CPU - matches DagmaDCE internal tensors

    print(f"  Running DAGMA-DCE (MLP, non-parametric) on {n_vars} variables...")

    # MLP dims: [d, hidden, 1] - hidden = 2×d is a good default
    hidden = max(10, 2 * n_vars)
    eq_model = nonlinear_dce.DagmaMLP_DCE(
        dims=[n_vars, hidden, 1], bias=True
    )  # keep on CPU - DagmaDCE internal identity matrix is on CPU

    model  = nonlinear_dce.DagmaDCE(eq_model, use_mse_loss=True)
    W_est  = model.fit(
        data,
        lambda1   = cfg['DCE_LAMBDA1'],
        lambda2   = cfg['DCE_LAMBDA2'],
        lr        = cfg['DCE_LR'],
        mu_factor = cfg['DCE_MU_FACTOR'],
        mu_init   = cfg['DCE_MU_INIT'],
        T         = cfg['DCE_T'],
        warm_iter = cfg['DCE_WARM_ITER'],
        max_iter  = cfg['DCE_MAX_ITER'],
    )
    # W_est shape: (n_vars, n_vars), W_est[i, j] = DCE of i -> j

    target_idx  = all_cols.index(target_name)
    raw_effects = {}
    for i, f in enumerate(features):
        raw_effects[f] = abs(float(W_est[i, target_idx]))

    dce_gamma = normalise(raw_effects, features)

    # Build inter-feature DAG (exclude target node)
    G = nx.DiGraph()
    G.add_nodes_from(features)
    threshold = 0.05
    for i, fi in enumerate(features):
        for j, fj in enumerate(features):
            if i != j and abs(W_est[i, j]) > threshold:
                G.add_edge(fi, fj, weight=float(W_est[i, j]))

    return W_est, dce_gamma, G


# run_dagma_dce_linear removed - linear_dce not in this DagmaDCE version


def run_standard_dagma(X: pd.DataFrame, y: pd.Series,
                        features: List[str], target_name: str) -> Tuple[np.ndarray, Dict[str, float], nx.DiGraph]:
    """Standard DAGMA linear — same as previous experiment, for direct comparison."""
    if not HAS_DAGMA:
        return None, {f: 1.0/len(features) for f in features}, nx.DiGraph()
    all_cols = features + [target_name]
    X_full   = X.copy()
    X_full[target_name] = y.values if hasattr(y, 'values') else y
    data     = X_full[all_cols].values.astype(float)
    model    = DagmaLinear(loss_type='l2')
    W        = model.fit(data, lambda1=0.02)
    target_idx  = all_cols.index(target_name)
    raw_effects = {f: abs(float(W[i, target_idx])) for i, f in enumerate(features)}
    dagma_gamma = normalise(raw_effects, features)
    G = nx.DiGraph()
    G.add_nodes_from(features)
    for i, fi in enumerate(features):
        for j, fj in enumerate(features):
            if i != j and abs(W[i, j]) > 0.05:
                G.add_edge(fi, fj, weight=float(W[i, j]))
    return W, dagma_gamma, G


# Graph + gamma comparison utils
def compare_graphs(recovered: nx.DiGraph, true_edges: set) -> dict:
    rec = set(recovered.edges())
    tp  = len(rec & true_edges); fp = len(rec - true_edges); fn = len(true_edges - rec)
    shd = fp + fn
    prec   = tp/(tp+fp) if (tp+fp) > 0 else 0.0
    recall = tp/(tp+fn) if (tp+fn) > 0 else 0.0
    f1     = 2*prec*recall/(prec+recall) if (prec+recall) > 0 else 0.0
    return {'TP':tp,'FP':fp,'FN':fn,'SHD':shd,
            'Precision':round(prec,3),'Recall':round(recall,3),'F1':round(f1,3),
            'Recovered':list(rec),'Missing':list(true_edges-rec),'Extra':list(rec-true_edges)}


def gamma_metrics(true_gamma: Dict[str,float], est_gamma: Dict[str,float],
                  features: List[str]) -> dict:
    gt = np.array([true_gamma[f] for f in features])
    eg = np.array([est_gamma[f]  for f in features])
    pr,_ = pearsonr(eg, gt); sr,_ = spearmanr(eg, gt)
    return {'Gamma_Pearson':round(float(pr),4),'Gamma_Spearman':round(float(sr),4),
            'Gamma_RMSE':round(float(np.sqrt(np.mean((eg-gt)**2))),4)}


def attribution_metrics(true_gamma: Dict[str,float], global_abs: pd.Series,
                        features: List[str]) -> dict:
    gt  = np.array([true_gamma[f]        for f in features])
    raw = np.array([global_abs.get(f,0)  for f in features])
    sv  = raw/raw.sum() if raw.sum() > 1e-12 else raw
    pr,_ = pearsonr(sv,gt); sr,_ = spearmanr(sv,gt)
    cos  = float(np.dot(sv,gt)/(np.linalg.norm(sv)*np.linalg.norm(gt)+1e-12))
    rmse = float(np.sqrt(np.mean((sv-gt)**2)))
    return {'Attr_Pearson':round(float(pr),4),'Attr_Spearman':round(float(sr),4),
            'Attr_Cosine':round(cos,4),'Attr_RMSE':round(rmse,4)}


class LinearMechanisms:
    def __init__(self, dag: nx.DiGraph, features: List[str]):
        self.dag = dag; self.features = features
        self.mech: Dict[str,dict] = {}
        self.topo = list(nx.topological_sort(dag))

    def fit(self, X: pd.DataFrame) -> 'LinearMechanisms':
        for var in self.features:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {'parents':[],'intercept':float(X[var].mean()),
                                  'coef':None,'sigma':float(X[var].std(ddof=0))}
            else:
                lr    = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mech[var] = {'parents':parents,'intercept':float(lr.intercept_),
                                  'coef':lr.coef_.astype(float),'sigma':float(np.std(resid,ddof=0))}
        return self

    def sample_batch(self, S: Set[str], x: pd.Series, M: int) -> np.ndarray:
        X_out = np.empty((M, len(self.features)))
        fi    = {f: i for i, f in enumerate(self.features)}
        sim   = {}
        for var in self.topo:
            vi = fi[var]
            if var in S:
                val=float(x[var]); X_out[:,vi]=val; sim[var]=np.full(M,val); continue
            m = self.mech[var]
            if not m['parents']:
                mu = np.full(M, m['intercept'])
            else:
                pm = np.stack([sim[p] if p in sim else np.full(M,float(x[p])) for p in m['parents']], axis=1)
                mu = m['intercept'] + pm @ m['coef']
            X_out[:,vi] = mu if m['sigma']<1e-8 else rng.normal(mu, m['sigma'], size=M)
            sim[var] = X_out[:,vi]
        return X_out


class PermutationEngine:
    def __init__(self, model, sem, gamma, X_train):
        self.model=model; self.features=list(X_train.columns)
        self.gamma=gamma; self.sem=sem
        self._Ef=float(model.predict(X_train.values).mean())

    def _vc(self, S, x, M): return float(np.mean(self.model.predict(self.sem.sample_batch(S,x,M))))

    def explain_instance(self, x, n_perms, M):
        fx=float(self.model.predict(x.values.reshape(1,-1))[0])
        phi=np.zeros(len(self.features))
        for _ in range(n_perms):
            perm=rng.permutation(self.features); S=set(); v_prev=self._vc(S,x,M)
            for feat in perm:
                Sn=S|{feat}; vc=self._vc(Sn,x,M)
                phi[self.features.index(feat)]+=self.gamma[feat]*(vc-v_prev)
                v_prev,S=vc,Sn
        phi/=n_perms
        s=float(phi.sum())
        if abs(s)>1e-12: phi=phi*((fx-self._Ef)/s)
        return pd.Series(phi,index=self.features)

    def explain_global(self, X_test, k, n_perms, M):
        k=min(k,len(X_test)); idx=rng.choice(len(X_test),size=k,replace=False)
        return pd.concat([self.explain_instance(X_test.iloc[i],n_perms,M) for i in idx],axis=1).abs().mean(axis=1)


class ExactEngine:
    def __init__(self, model, sem, gamma, X_train):
        self.model=model; self.features=list(X_train.columns)
        self.n=len(self.features); self.gamma=gamma; self.sem=sem
        self._Ef=float(model.predict(X_train.values).mean())
        self._kw={sz: float(math.factorial(sz)*math.factorial(self.n-sz-1)/math.factorial(self.n))
                  for sz in range(self.n)}

    def explain_instance(self, x, M):
        fx=float(self.model.predict(x.values.reshape(1,-1))[0])
        phi=np.zeros(self.n)
        cache={frozenset(sub): float(np.mean(self.model.predict(self.sem.sample_batch(frozenset(sub),x,M))))
               for r in range(self.n+1) for sub in itertools.combinations(self.features,r)}
        for idx,feat in enumerate(self.features):
            rest=[f for f in self.features if f!=feat]
            for r in range(self.n):
                for sub in itertools.combinations(rest,r):
                    S=frozenset(sub)
                    phi[idx]+=self._kw[len(sub)]*self.gamma[feat]*(cache[S|{feat}]-cache[S])
        s=float(phi.sum())
        if abs(s)>1e-12: phi=phi*((fx-self._Ef)/s)
        return pd.Series(phi,index=self.features)

    def explain_global(self, X_test, k, M):
        k=min(k,len(X_test)); phi_list=[]
        for i in range(k):
            if (i+1)%20==0: print(f"      instance {i+1}/{k} ...",flush=True)
            phi_list.append(self.explain_instance(X_test.iloc[i],M))
        return pd.concat(phi_list,axis=1).abs().mean(axis=1)


def run_causal_shap(model, gamma, dag, Xtr, Xte, engine_type, cfg) -> pd.Series:
    global rng
    #   (a) all methods sample the SAME 100 test instances
    #   (b) all methods use the SAME permutation sequences
    # Without this, the stateful module-level rng drifts across methods
    # and the ceiling appears lower than it really is.
    rng = np.random.default_rng(cfg['RANDOM_SEED'])
    sem = LinearMechanisms(dag, list(Xtr.columns)).fit(Xtr)
    if engine_type == 'permutation':
        e = PermutationEngine(model, sem, gamma, Xtr)
        return e.explain_global(Xte, cfg['YIELD_K'], cfg['YIELD_MAX_PERM'], cfg['YIELD_M'])
    else:
        e = ExactEngine(model, sem, gamma, Xtr)
        return e.explain_global(Xte, cfg['SOIL_K'], cfg['SOIL_M'])


# Dataset runner
def run_dataset(name: str, features: List[str], true_edges: set,
                true_gamma: Dict[str,float], Xtr: pd.DataFrame, Xte: pd.DataFrame,
                ytr: np.ndarray, engine_type: str, cfg: dict) -> pd.DataFrame:

    print(f"\n{'='*80}\nDATASET: {name.upper()}\n{'='*80}")
    print(f"  True gamma: { {f: round(v,4) for f,v in true_gamma.items()} }")

    target_name = f'{name}_target'
    results     = []

    def log4(label, gm, shd, f1, am):
        gp = f"{gm['Gamma_Pearson']:.4f}" if gm else "n/a "
        print(f"  [{label}]")
        print(f"    gamma Pearson={gp} | Graph SHD={shd} | Graph F1={f1:.3f}")
        print(f"    Attr  Pearson={am['Attr_Pearson']:.4f} | Spearman={am['Attr_Spearman']:.4f} "
              f"| Cosine={am['Attr_Cosine']:.4f} | RMSE={am['Attr_RMSE']:.4f}")

    # Train XGBoost once - shared across all methods for fair comparison
    model = XGBRegressor(n_estimators=160, max_depth=5, learning_rate=0.08,
                         subsample=0.9, colsample_bytree=0.9,
                         random_state=cfg['RANDOM_SEED'], verbosity=0,
                         n_jobs=1, tree_method='hist')
    model.fit(Xtr.values, ytr)

    true_dag = nx.DiGraph(); true_dag.add_nodes_from(features)
    for u, v in true_edges: true_dag.add_edge(u, v)

    star_dag = nx.DiGraph(); star_dag.add_nodes_from(features)  # no edges

    # ── Method 1: DAGMA-DCE MLP ──────────────────────────────────
    print("\n--- Method 1: DAGMA-DCE (non-linear MLP) ---")
    t0 = time.time()
    W_nl, gamma_nl, dag_nl = run_dagma_dce_nonlinear(Xtr, ytr, features, target_name, cfg)
    t_nl = time.time() - t0

    if W_nl is not None:
        gc_nl = compare_graphs(dag_nl, true_edges)
        gm_nl = gamma_metrics(true_gamma, gamma_nl, features)
        print(f"  gamma recovered: { {f: round(v,4) for f,v in gamma_nl.items()} }")

        # 1a: own graph + own gamma
        am_1a = attribution_metrics(true_gamma,
                    run_causal_shap(model, gamma_nl, dag_nl, Xtr, Xte, engine_type, cfg),
                    features)
        log4("DCE-MLP gamma + DCE-MLP graph", gm_nl, gc_nl['SHD'], gc_nl['F1'], am_1a)
        results.append({'Method': 'DAGMA-DCE MLP (own graph + own gamma)',
                        **gm_nl, 'Graph_SHD': gc_nl['SHD'], 'Graph_F1': gc_nl['F1'], **am_1a,
                        'Time_s': round(t_nl, 1)})

        # 1b: true graph + own gamma - isolates gamma quality
        am_1b = attribution_metrics(true_gamma,
                    run_causal_shap(model, gamma_nl, true_dag, Xtr, Xte, engine_type, cfg),
                    features)
        log4("DCE-MLP gamma + true graph", gm_nl, 0, 1.0, am_1b)
        results.append({'Method': 'DAGMA-DCE MLP (true graph + own gamma)',
                        **gm_nl, 'Graph_SHD': 0, 'Graph_F1': 1.0, **am_1b,
                        'Time_s': round(t_nl, 1)})

    # ── Method 2: Standard DAGMA linear ──────────────────────────
    print("\n--- Method 2: Standard DAGMA (linear) ---")
    t0 = time.time()
    W_d, gamma_d, dag_d = run_standard_dagma(Xtr, ytr, features, target_name)
    t_d = time.time() - t0

    if W_d is not None:
        gc_d = compare_graphs(dag_d, true_edges)
        gm_d = gamma_metrics(true_gamma, gamma_d, features)
        print(f"  gamma recovered: { {f: round(v,4) for f,v in gamma_d.items()} }")

        # 2a: own graph + own gamma
        am_2a = attribution_metrics(true_gamma,
                    run_causal_shap(model, gamma_d, dag_d, Xtr, Xte, engine_type, cfg),
                    features)
        log4("DAGMA gamma + DAGMA graph", gm_d, gc_d['SHD'], gc_d['F1'], am_2a)
        results.append({'Method': 'Standard DAGMA (own graph + own gamma)',
                        **gm_d, 'Graph_SHD': gc_d['SHD'], 'Graph_F1': gc_d['F1'], **am_2a,
                        'Time_s': round(t_d, 1)})

        # 2b: true graph + own gamma - isolates gamma quality
        am_2b = attribution_metrics(true_gamma,
                    run_causal_shap(model, gamma_d, true_dag, Xtr, Xte, engine_type, cfg),
                    features)
        log4("DAGMA gamma + true graph", gm_d, 0, 1.0, am_2b)
        results.append({'Method': 'Standard DAGMA (true graph + own gamma)',
                        **gm_d, 'Graph_SHD': 0, 'Graph_F1': 1.0, **am_2b,
                        'Time_s': round(t_d, 1)})

    # ── Ceiling A: true gamma + true graph ────────────────────────
    print("\n--- Ceiling A: True gamma + True graph ---")
    am_ca = attribution_metrics(true_gamma,
                run_causal_shap(model, true_gamma, true_dag, Xtr, Xte, engine_type, cfg),
                features)
    log4("true gamma + true graph", None, 0, 1.0, am_ca)
    results.append({'Method': 'TRUE gamma + TRUE graph (ceiling)',
                    'Gamma_Pearson': 1.0, 'Gamma_Spearman': 1.0, 'Gamma_RMSE': 0.0,
                    'Graph_SHD': 0, 'Graph_F1': 1.0, **am_ca, 'Time_s': 0.0})

    # ── Ceiling B: true gamma + star DAG ──────────────────────────
    print("\n--- Ceiling B: True gamma + Star DAG (no inter-feature edges) ---")
    gc_star = compare_graphs(star_dag, true_edges)
    am_cb   = attribution_metrics(true_gamma,
                run_causal_shap(model, true_gamma, star_dag, Xtr, Xte, engine_type, cfg),
                features)
    log4("true gamma + star DAG", None, gc_star['SHD'], gc_star['F1'], am_cb)
    results.append({'Method': 'TRUE gamma + Star DAG (no edges)',
                    'Gamma_Pearson': 1.0, 'Gamma_Spearman': 1.0, 'Gamma_RMSE': 0.0,
                    'Graph_SHD': gc_star['SHD'], 'Graph_F1': gc_star['F1'],
                    **am_cb, 'Time_s': 0.0})

    df = pd.DataFrame(results)
    disp = ['Method', 'Gamma_Pearson', 'Graph_SHD', 'Graph_F1',
            'Attr_Pearson', 'Attr_Spearman', 'Attr_Cosine', 'Attr_RMSE']
    print(f"\n{'='*110}\nRESULTS SUMMARY - {name.upper()}\n{'='*110}")
    print(df[disp].to_string(index=False))
    print("\n  Gamma_Pearson = how well recovered gamma matches ground-truth gamma (higher=better)")
    print("  Graph_SHD     = structural Hamming distance vs true graph (0=perfect, lower=better)")
    print("  Attr_*        = attribution quality vs GT gamma (Pearson/Spearman/Cosine higher=better, RMSE lower=better)")
    return df


# Main
def main():
    print("\n" + "="*80)
    print("DAGMA-DCE vs DAGMA vs CEILING — Soil & Yield")
    print("="*80)

    if not HAS_DCE:
        print("\nWarning: DagmaDCE not installed. Run setup commands at top of script.")
        print("  Continuing with DAGMA-DCE MLP and standard DAGMA only.\n")

    # ── yield ────────────────────────────────────────────────────
    data        = np.load(CONFIG['YIELD_DATA_PATH'], allow_pickle=True)
    X_raw_train = pd.DataFrame(data['X_train'], columns=YIELD_FEATURES)
    X_raw_test  = pd.DataFrame(data['X_test'],  columns=YIELD_FEATURES)
    y_train_y   = data['y_train'].astype(float)
    true_gamma_y = compute_yield_gamma(X_raw_train)

    scaler_y = StandardScaler()
    Xtr_y = pd.DataFrame(scaler_y.fit_transform(X_raw_train), columns=YIELD_FEATURES)
    Xte_y = pd.DataFrame(scaler_y.transform(X_raw_test),      columns=YIELD_FEATURES)

    df_y = run_dataset('Yield', YIELD_FEATURES, YIELD_TRUE_EDGES, true_gamma_y,
                        Xtr_y, Xte_y, y_train_y, 'permutation', CONFIG)
    df_y.to_csv('dagma_dce_results_yield.csv', index=False)

    # ── soil ─────────────────────────────────────────────────────
    dc       = pd.read_csv(CONFIG['SOIL_DATA_PATH']).rename(columns=SOIL_COL_MAP)
    Xtr_s_raw, Xte_s_raw, ytr_s, _ = train_test_split(
        dc[SOIL_FEATURES], dc['SPI_CONTINUOUS'],
        test_size=CONFIG['TEST_SIZE'], random_state=CONFIG['RANDOM_SEED'], shuffle=True)
    true_gamma_s = compute_soil_gamma(CONFIG['SOIL_GROUND_TRUTH_PATH'])

    scaler_s = StandardScaler()
    Xtr_s = pd.DataFrame(scaler_s.fit_transform(Xtr_s_raw), columns=SOIL_FEATURES, index=Xtr_s_raw.index)
    Xte_s = pd.DataFrame(scaler_s.transform(Xte_s_raw),     columns=SOIL_FEATURES, index=Xte_s_raw.index)

    df_s = run_dataset('Soil', SOIL_FEATURES, SOIL_TRUE_EDGES, true_gamma_s,
                        Xtr_s, Xte_s, ytr_s.values, 'exact', CONFIG)
    df_s.to_csv('dagma_dce_results_soil.csv', index=False)

    print("\nSaved: dagma_dce_results_yield.csv, dagma_dce_results_soil.csv")


if __name__ == '__main__':
    main()

✓ DagmaDCE available
✓ dagma (standard linear) available
  Device: cuda

DAGMA-DCE vs DAGMA vs CEILING — Soil & Yield

DATASET: YIELD
  True gamma: {'Nitrogen': 0.3118, 'Irrigation': 0.3727, 'Precipitation': 0.2457, 'Herbicide': 0.0697}

--- Method 1: DAGMA-DCE (non-linear MLP) ---
  Running DAGMA-DCE (MLP, non-parametric) on 5 variables...


  gamma recovered: {'Nitrogen': 0.25, 'Irrigation': 0.25, 'Precipitation': 0.25, 'Herbicide': 0.25}
  [DCE-MLP gamma + DCE-MLP graph]
    gamma Pearson=nan | Graph SHD=0 | Graph F1=0.000
    Attr  Pearson=0.8737 | Spearman=0.8000 | Cosine=0.9640 | RMSE=0.0788
  [DCE-MLP gamma + true graph]
    gamma Pearson=nan | Graph SHD=0 | Graph F1=1.000
    Attr  Pearson=0.8737 | Spearman=0.8000 | Cosine=0.9640 | RMSE=0.0788

--- Method 2: Standard DAGMA (linear) ---


  gamma recovered: {'Nitrogen': 0.2744, 'Irrigation': 0.3638, 'Precipitation': 0.2991, 'Herbicide': 0.0627}
  [DAGMA gamma + DAGMA graph]
    gamma Pearson=0.9572 | Graph SHD=0 | Graph F1=0.000
    Attr  Pearson=0.8695 | Spearman=0.8000 | Cosine=0.9415 | RMSE=0.1083
  [DAGMA gamma + true graph]
    gamma Pearson=0.9572 | Graph SHD=0 | Graph F1=1.000
    Attr  Pearson=0.8695 | Spearman=0.8000 | Cosine=0.9415 | RMSE=0.1083

--- Ceiling A: True gamma + True graph ---
  [true gamma + true graph]
    gamma Pearson=n/a  | Graph SHD=0 | Graph F1=1.000
    Attr  Pearson=0.9843 | Spearman=1.0000 | Cosine=0.9868 | RMSE=0.0511

--- Ceiling B: True gamma + Star DAG (no inter-feature edges) ---
  [true gamma + star DAG]
    gamma Pearson=n/a  | Graph SHD=0 | Graph F1=0.000
    Attr  Pearson=0.9843 | Spearman=1.0000 | Cosine=0.9868 | RMSE=0.0511

RESULTS SUMMARY - YIELD
                                 Method  Gamma_Pearson  Graph_SHD  Graph_F1  Attr_Pearson  Attr_Spearman  Attr_Cosine  Attr_RMSE
  

  gamma recovered: {'Organic Carbon': 0.1543, 'Nitrogen': 0.1473, 'Phosphorus': 0.0928, 'Potassium': 0.1565, 'pH': 0.2095, 'Zinc': 0.0773, 'Electrical Conductivity': 0.1623}
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  [DCE-MLP gamma + DCE-MLP graph]
    gamma Pearson=0.2471 | Graph SHD=9 | Graph F1=0.182
    Attr  Pearson=0.8410 | Spearman=0.9286 | Cosine=0.9330 | RMSE=0.0688
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  [DCE-MLP gamma + true graph]
    gamma Pearson=0.2471 | Graph SHD=0 | Graph F1=1.000
    Attr  Pearson=0.9248 | Spearman=0.8214 | Cosine=0.9662 | RMSE=0.0498

--- Method 2: Standard DAGMA (linear) ---


  gamma recovered: {'Organic Carbon': 0.0578, 'Nitrogen': 0.3589, 'Phosphorus': 0.0517, 'Potassium': 0.2169, 'pH': 0.178, 'Zinc': 0.0236, 'Electrical Conductivity': 0.1131}
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  [DAGMA gamma + DAGMA graph]
    gamma Pearson=0.9466 | Graph SHD=1 | Graph F1=0.667
    Attr  Pearson=0.9868 | Spearman=0.9286 | Cosine=0.9825 | RMSE=0.0496
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  [DAGMA gamma + true graph]
    gamma Pearson=0.9466 | Graph SHD=0 | Graph F1=1.000
    Attr  Pearson=0.9845 | Spearman=0.9286 | Cosine=0.9820 | RMSE=0.0494

--- Ceiling A: True gamma + True graph ---
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  [true gamma + true graph]
    gamma Pearson=n/a  | Graph SHD=0 | Graph 

In [3]:
import sys
sys.path.insert(0, 'DAGMA-DCE/src')


import time
import itertools
import math
import warnings
from typing import Dict, List, Set, Tuple

import numpy as np
import pandas as pd
import torch
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import networkx as nx
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

from DagmaDCE import nonlinear_dce
from dagma.linear import DagmaLinear

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Data paths ────────────────────────────────────────────────────────────────
SOIL_DATA_PATH         = '../../../../datasets/data_complete.csv'
SOIL_GROUND_TRUTH_PATH = '../../../../datasets/ground_truth.csv'
YIELD_DATA_PATH        = '../../../../datasets/yield_regression_preprocessed.npz'

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
BASE_RNG = np.random.default_rng(RANDOM_SEED)

SOIL_FEATURES  = ['Organic Carbon', 'Nitrogen', 'Phosphorus', 'Potassium',
                   'pH', 'Zinc', 'Electrical Conductivity']
SOIL_COL_MAP   = {'OC': 'Organic Carbon', 'N': 'Nitrogen', 'P': 'Phosphorus',
                  'K': 'Potassium', 'PH': 'pH', 'ZN': 'Zinc', 'EC': 'Electrical Conductivity'}
SOIL_TRUE_EDGES = {('Organic Carbon', 'pH'), ('pH', 'Zinc')}

YIELD_FEATURES   = ['Nitrogen', 'Irrigation', 'Precipitation', 'Herbicide']
YIELD_TRUE_EDGES = set()
YIELD_TRANSLOG   = {
    'betaN': 0.35, 'betaI': 0.50, 'betaP': 0.25, 'betaH': 0.05,
    'betaNN': -0.02, 'betaII': -0.03, 'betaPP': -0.01, 'betaHH': -0.005,
    'betaNI': 0.04, 'betaNP': 0.02, 'betaNH': 0.01,
    'betaIP': 0.03, 'betaIH': 0.01, 'betaPH': 0.005,
}


# ── Ground-truth gamma ────────────────────────────────────────────────────────
def compute_soil_gamma(path: str) -> Dict[str, float]:
    gt  = pd.read_csv(path)
    net = pd.DataFrame({
        'Organic Carbon':          gt['OC_base']  - gt['OC_penalty'],
        'Nitrogen':                gt['N_base']   - gt['N_penalty'],
        'Phosphorus':              gt['P_base']   - gt['P_penalty'],
        'Potassium':               gt['K_base']   - gt['K_penalty'],
        'pH':                     -gt['PH_penalty'],
        'Zinc':                    gt['ZN_base']  - gt['ZN_penalty'],
        'Electrical Conductivity': -gt['EC_penalty'],
    })
    m = net.abs().mean()
    return (m / m.sum()).to_dict()


def compute_yield_gamma(X_raw: pd.DataFrame) -> Dict[str, float]:
    c  = YIELD_TRANSLOG
    lm = {f: float(np.log(X_raw[f]).mean()) for f in YIELD_FEATURES}
    lnN, lnI, lnP, lnH = lm['Nitrogen'], lm['Irrigation'], lm['Precipitation'], lm['Herbicide']
    e = {
        'Nitrogen':      c['betaN']  + c['betaNN']*lnN + c['betaNI']*lnI + c['betaNP']*lnP + c['betaNH']*lnH,
        'Irrigation':    c['betaI']  + c['betaII']*lnI + c['betaNI']*lnN + c['betaIP']*lnP + c['betaIH']*lnH,
        'Precipitation': c['betaP']  + c['betaPP']*lnP + c['betaNP']*lnN + c['betaIP']*lnI + c['betaPH']*lnH,
        'Herbicide':     c['betaH']  + c['betaHH']*lnH + c['betaNH']*lnN + c['betaIH']*lnI + c['betaPH']*lnP,
    }
    tot = sum(e.values())
    return {f: v / tot for f, v in e.items()}


def normalise(g: Dict[str, float], features: List[str]) -> Dict[str, float]:
    g   = {f: max(v, 0.0) for f, v in g.items()}
    tot = sum(g.values())
    return {f: v / tot for f, v in g.items()} if tot > 1e-12 else {f: 1.0 / len(features) for f in features}


# ── DCE-MLP runner (parameterised) ───────────────────────────────────────────
def run_dce_mlp(
    X: pd.DataFrame,
    y,
    features: List[str],
    target_name: str,
    lambda1: float,
    lambda2: float   = 5e-3,
    lr: float        = 2e-4,
    mu_factor: float = 0.1,
    mu_init: float   = 0.1,
    T: int           = 4,
    warm_iter: int   = 7_000,
    max_iter: int    = 8_000,
    hidden_mult: int = 2,        # hidden dim = hidden_mult * n_vars
) -> Tuple[np.ndarray, Dict[str, float], nx.DiGraph]:
    """
    Parameterised DCE-MLP runner.
    Returns (W_est, gamma_dict, dag).
    """
    all_cols = features + [target_name]
    n_vars   = len(all_cols)
    X_full   = X.copy()
    X_full[target_name] = y.values if hasattr(y, 'values') else y
    data_np  = X_full[all_cols].values.astype(np.float64)
    data     = torch.tensor(data_np, dtype=torch.float64)

    hidden   = max(10, hidden_mult * n_vars)
    eq_model = nonlinear_dce.DagmaMLP_DCE(dims=[n_vars, hidden, 1], bias=True)
    model    = nonlinear_dce.DagmaDCE(eq_model, use_mse_loss=True)

    W_est = model.fit(
        data,
        lambda1=lambda1, lambda2=lambda2, lr=lr,
        mu_factor=mu_factor, mu_init=mu_init,
        T=T, warm_iter=warm_iter, max_iter=max_iter,
    )

    target_idx  = all_cols.index(target_name)
    raw_effects = {f: abs(float(W_est[i, target_idx])) for i, f in enumerate(features)}
    gamma       = normalise(raw_effects, features)

    G = nx.DiGraph()
    G.add_nodes_from(features)
    for i, fi in enumerate(features):
        for j, fj in enumerate(features):
            if i != j and abs(W_est[i, j]) > 0.05:
                G.add_edge(fi, fj, weight=float(W_est[i, j]))

    return W_est, gamma, G


# ── Causal SHAP (same engines as main experiment) ────────────────────────────
class LinearMechanisms:
    def __init__(self, dag: nx.DiGraph, features: List[str]):
        self.dag      = dag
        self.features = features
        self.mech: Dict[str, dict] = {}
        self.topo = list(nx.topological_sort(dag))

    def fit(self, X: pd.DataFrame) -> 'LinearMechanisms':
        for var in self.features:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {
                    'parents': [], 'intercept': float(X[var].mean()),
                    'coef': None, 'sigma': float(X[var].std(ddof=0)),
                }
            else:
                lr    = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mech[var] = {
                    'parents': parents, 'intercept': float(lr.intercept_),
                    'coef': lr.coef_.astype(float), 'sigma': float(np.std(resid, ddof=0)),
                }
        return self

    def sample_batch(self, S: Set[str], x: pd.Series, M: int, rng) -> np.ndarray:
        X_out = np.empty((M, len(self.features)))
        fi    = {f: i for i, f in enumerate(self.features)}
        sim   = {}
        for var in self.topo:
            vi = fi[var]
            if var in S:
                val = float(x[var]); X_out[:, vi] = val; sim[var] = np.full(M, val); continue
            m = self.mech[var]
            if not m['parents']:
                mu = np.full(M, m['intercept'])
            else:
                pm = np.stack([sim[p] if p in sim else np.full(M, float(x[p])) for p in m['parents']], axis=1)
                mu = m['intercept'] + pm @ m['coef']
            X_out[:, vi] = mu if m['sigma'] < 1e-8 else rng.normal(mu, m['sigma'], size=M)
            sim[var] = X_out[:, vi]
        return X_out


def _vc(model, sem, S, x, M, rng):
    return float(np.mean(model.predict(sem.sample_batch(S, x, M, rng))))


def explain_permutation(model, sem, gamma, features, X_test, K=100, n_perms=500, M=128, seed=42):
    """Permutation Causal SHAP — used for Yield (4 features)."""
    rng  = np.random.default_rng(seed)
    Ef   = float(model.predict(X_test.values).mean())
    k    = min(K, len(X_test))
    idx  = rng.choice(len(X_test), size=k, replace=False)
    phis = []
    for i in idx:
        x  = X_test.iloc[i]
        fx = float(model.predict(x.values.reshape(1, -1))[0])
        phi = np.zeros(len(features))
        for _ in range(n_perms):
            perm   = rng.permutation(features)
            S      = set()
            v_prev = _vc(model, sem, S, x, M, rng)
            for feat in perm:
                Sn   = S | {feat}
                vc   = _vc(model, sem, Sn, x, M, rng)
                phi[features.index(feat)] += gamma[feat] * (vc - v_prev)
                v_prev, S = vc, Sn
        phi /= n_perms
        s = float(phi.sum())
        if abs(s) > 1e-12:
            phi = phi * ((fx - Ef) / s)
        phis.append(pd.Series(phi, index=features))
    return pd.concat(phis, axis=1).abs().mean(axis=1)


def explain_exact(model, sem, gamma, features, X_test, K=100, M=128, seed=42):
    """Exact Causal SHAP — used for Soil (7 features, 2^7 coalitions)."""
    rng  = np.random.default_rng(seed)
    n    = len(features)
    kw   = {sz: float(math.factorial(sz) * math.factorial(n - sz - 1) / math.factorial(n))
             for sz in range(n)}
    Ef   = float(model.predict(X_test.values).mean())
    k    = min(K, len(X_test))
    idx  = rng.choice(len(X_test), size=k, replace=False)
    phis = []
    for cnt, i in enumerate(idx):
        if (cnt + 1) % 20 == 0:
            print(f"      instance {cnt + 1}/{k} ...", flush=True)
        x  = X_test.iloc[i]
        fx = float(model.predict(x.values.reshape(1, -1))[0])
        cache = {
            frozenset(sub): float(np.mean(model.predict(sem.sample_batch(frozenset(sub), x, M, rng))))
            for r in range(n + 1)
            for sub in itertools.combinations(features, r)
        }
        phi = np.zeros(n)
        for idx_f, feat in enumerate(features):
            rest = [f for f in features if f != feat]
            for r in range(n):
                for sub in itertools.combinations(rest, r):
                    S = frozenset(sub)
                    phi[idx_f] += kw[len(sub)] * gamma[feat] * (cache[S | {feat}] - cache[S])
        s = float(phi.sum())
        if abs(s) > 1e-12:
            phi = phi * ((fx - Ef) / s)
        phis.append(pd.Series(phi, index=features))
    return pd.concat(phis, axis=1).abs().mean(axis=1)


def run_causal_shap(model, gamma, dag, Xtr, Xte, engine_type, seed=42):
    sem = LinearMechanisms(dag, list(Xtr.columns)).fit(Xtr)
    if engine_type == 'permutation':
        return explain_permutation(model, sem, gamma, list(Xtr.columns), Xte, seed=seed)
    else:
        return explain_exact(model, sem, gamma, list(Xtr.columns), Xte, seed=seed)


def attribution_metrics(true_gamma, global_abs, features):
    gt  = np.array([true_gamma[f]       for f in features])
    raw = np.array([global_abs.get(f, 0) for f in features])
    sv  = raw / raw.sum() if raw.sum() > 1e-12 else raw
    pr, _ = pearsonr(sv, gt)
    sr, _ = spearmanr(sv, gt)
    rmse  = float(np.sqrt(np.mean((sv - gt) ** 2)))
    return round(float(pr), 4), round(float(sr), 4), round(rmse, 4)


def gamma_pearson(true_gamma, est_gamma, features):
    gt = np.array([true_gamma[f]  for f in features])
    eg = np.array([est_gamma[f]   for f in features])
    if np.std(eg) < 1e-10:
        return float('nan')   # constant vector
    pr, _ = pearsonr(eg, gt)
    return round(float(pr), 4)


# ── Sweep configurations ──────────────────────────────────────────────────────
# Each config is a dict of kwargs passed directly to run_dce_mlp().
#   Baseline  -> λ₁=3.5e-2, T=4, max_iter=8K,  hidden_mult=2
# We also sweep λ₁ more finely in a separate block below.

CONFIGS = [
    {
        'label':       'Baseline (λ₁=3.5e-2, T=4, iter=8K, h=2d)',
        'lambda1':     3.5e-2,
        'T':           4,
        'warm_iter':   7_000,
        'max_iter':    8_000,
        'hidden_mult': 2,
    },
    {
        'label':       'Fix 1 — λ₁=1e-2 only',
        'lambda1':     1e-2,
        'T':           4,
        'warm_iter':   7_000,
        'max_iter':    8_000,
        'hidden_mult': 2,
    },
    {
        'label':       'Fix 1+2 — λ₁=1e-2, T=6, iter=12K',
        'lambda1':     1e-2,
        'T':           6,
        'warm_iter':   10_000,
        'max_iter':    12_000,
        'hidden_mult': 2,
    },
    {
        'label':       'Fix 1+2+3 — λ₁=1e-2, T=6, iter=12K, h=4d',
        'lambda1':     1e-2,
        'T':           6,
        'warm_iter':   10_000,
        'max_iter':    12_000,
        'hidden_mult': 4,
    },
    {
        'label':       'λ₁ sweep — 5e-3, T=6, iter=12K, h=4d',
        'lambda1':     5e-3,
        'T':           6,
        'warm_iter':   10_000,
        'max_iter':    12_000,
        'hidden_mult': 4,
    },
    {
        'label':       'λ₁ sweep — 1e-3, T=6, iter=12K, h=4d',
        'lambda1':     1e-3,
        'T':           6,
        'warm_iter':   10_000,
        'max_iter':    12_000,
        'hidden_mult': 4,
    },
]


# ── Dataset runner ────────────────────────────────────────────────────────────
def sweep_dataset(
    name: str,
    features: List[str],
    true_edges: set,
    true_gamma: Dict[str, float],
    Xtr: pd.DataFrame,
    Xte: pd.DataFrame,
    ytr: np.ndarray,
    engine_type: str,
) -> pd.DataFrame:

    print(f"\n{'='*80}")
    print(f"DATASET: {name.upper()}")
    print(f"{'='*80}")
    print(f"  True γ: { {f: round(v, 4) for f, v in true_gamma.items()} }")

    target_name = f'{name}_target'

    # True DAG for reference
    true_dag = nx.DiGraph()
    true_dag.add_nodes_from(features)
    for u, v in true_edges:
        true_dag.add_edge(u, v)

    # Train XGBoost once - same model for all configurations
    model = XGBRegressor(
        n_estimators=160, max_depth=5, learning_rate=0.08,
        subsample=0.9, colsample_bytree=0.9,
        random_state=RANDOM_SEED, verbosity=0, n_jobs=1, tree_method='hist',
    )
    model.fit(Xtr.values, ytr)

    # ── Ceiling for reference ────────────────────────────────────────────────
    print("\n  [Reference] Ceiling: true γ + true graph")
    g_ceil = run_causal_shap(model, true_gamma, true_dag, Xtr, Xte, engine_type)
    ceil_p, ceil_s, ceil_r = attribution_metrics(true_gamma, g_ceil, features)
    print(f"  Attr Pearson={ceil_p} | Spearman={ceil_s} | RMSE={ceil_r}")

    rows = []

    for cfg in CONFIGS:
        print(f"\n  ── {cfg['label']} ──")
        t0 = time.time()
        W_est, gamma_est, dag_est = run_dce_mlp(
            Xtr, ytr, features, target_name,
            lambda1=cfg['lambda1'],
            T=cfg['T'],
            warm_iter=cfg['warm_iter'],
            max_iter=cfg['max_iter'],
            hidden_mult=cfg['hidden_mult'],
        )
        t_fit = time.time() - t0

        gp = gamma_pearson(true_gamma, gamma_est, features)
        print(f"  γ recovered: { {f: round(v, 4) for f, v in gamma_est.items()} }")
        print(f"  γ Pearson: {gp}  |  fit time: {t_fit:.1f}s")

        # Attribution with own graph
        t0 = time.time()
        g_abs = run_causal_shap(model, gamma_est, dag_est, Xtr, Xte, engine_type)
        ap, as_, ar = attribution_metrics(true_gamma, g_abs, features)
        t_shap = time.time() - t0
        print(f"  [own graph]  Attr Pearson={ap} | Spearman={as_} | RMSE={ar}")

        # Attribution with true graph - isolates gamma quality
        g_abs2 = run_causal_shap(model, gamma_est, true_dag, Xtr, Xte, engine_type)
        ap2, as2, ar2 = attribution_metrics(true_gamma, g_abs2, features)
        print(f"  [true graph] Attr Pearson={ap2} | Spearman={as2} | RMSE={ar2}")

        rows.append({
            'Config':           cfg['label'],
            'lambda1':          cfg['lambda1'],
            'T':                cfg['T'],
            'max_iter':         cfg['max_iter'],
            'hidden_mult':      cfg['hidden_mult'],
            'Gamma_Pearson':    gp,
            'Attr_P_ownGraph':  ap,
            'Attr_P_trueGraph': ap2,
            'Attr_RMSE_own':    ar,
            'Attr_RMSE_true':   ar2,
            'Fit_s':            round(t_fit, 1),
        })

    df = pd.DataFrame(rows)

    print(f"\n{'='*80}")
    print(f"SWEEP SUMMARY — {name.upper()}")
    print(f"  Ceiling: Attr Pearson={ceil_p} | RMSE={ceil_r}")
    print(f"{'='*80}")
    cols = ['Config', 'Gamma_Pearson', 'Attr_P_ownGraph', 'Attr_P_trueGraph', 'Attr_RMSE_own', 'Fit_s']
    print(df[cols].to_string(index=False))

    return df


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print("DCE-MLP Hyperparameter Fix Sweep")
    print("=" * 80)

    # ── Load Yield ────────────────────────────────────────────────────────────
    data         = np.load(YIELD_DATA_PATH, allow_pickle=True)
    X_raw_train  = pd.DataFrame(data['X_train'], columns=YIELD_FEATURES)
    X_raw_test   = pd.DataFrame(data['X_test'],  columns=YIELD_FEATURES)
    y_train_y    = data['y_train'].astype(float)
    true_gamma_y = compute_yield_gamma(X_raw_train)

    scaler_y = StandardScaler()
    Xtr_y = pd.DataFrame(scaler_y.fit_transform(X_raw_train), columns=YIELD_FEATURES)
    Xte_y = pd.DataFrame(scaler_y.transform(X_raw_test),      columns=YIELD_FEATURES)

    df_y = sweep_dataset(
        'Yield', YIELD_FEATURES, YIELD_TRUE_EDGES, true_gamma_y,
        Xtr_y, Xte_y, y_train_y, 'permutation',
    )
    df_y.to_csv('dce_fix_results_yield.csv', index=False)

    # ── Load Soil ─────────────────────────────────────────────────────────────
    dc = pd.read_csv(SOIL_DATA_PATH).rename(columns=SOIL_COL_MAP)
    Xtr_s_raw, Xte_s_raw, ytr_s, _ = train_test_split(
        dc[SOIL_FEATURES], dc['SPI_CONTINUOUS'],
        test_size=0.20, random_state=RANDOM_SEED, shuffle=True,
    )
    true_gamma_s = compute_soil_gamma(SOIL_GROUND_TRUTH_PATH)

    scaler_s = StandardScaler()
    Xtr_s = pd.DataFrame(scaler_s.fit_transform(Xtr_s_raw), columns=SOIL_FEATURES, index=Xtr_s_raw.index)
    Xte_s = pd.DataFrame(scaler_s.transform(Xte_s_raw),     columns=SOIL_FEATURES, index=Xte_s_raw.index)

    df_s = sweep_dataset(
        'Soil', SOIL_FEATURES, SOIL_TRUE_EDGES, true_gamma_s,
        Xtr_s, Xte_s, ytr_s.values, 'exact',
    )
    df_s.to_csv('dce_fix_results_soil.csv', index=False)

    print("\nSaved: dce_fix_results_yield.csv, dce_fix_results_soil.csv")


if __name__ == '__main__':
    main()

Device: cuda
DCE-MLP Hyperparameter Fix Sweep

DATASET: YIELD
  True γ: {'Nitrogen': 0.3118, 'Irrigation': 0.3727, 'Precipitation': 0.2457, 'Herbicide': 0.0697}

  [Reference] Ceiling: true γ + true graph
  Attr Pearson=0.9959 | Spearman=1.0 | RMSE=0.0358

  ── Baseline (λ₁=3.5e-2, T=4, iter=8K, h=2d) ──


  γ recovered: {'Nitrogen': 0.25, 'Irrigation': 0.25, 'Precipitation': 0.25, 'Herbicide': 0.25}
  γ Pearson: nan  |  fit time: 88.5s
  [own graph]  Attr Pearson=0.8938 | Spearman=1.0 | RMSE=0.0646
  [true graph] Attr Pearson=0.8938 | Spearman=1.0 | RMSE=0.0646

  ── Fix 1 — λ₁=1e-2 only ──


  γ recovered: {'Nitrogen': 0.25, 'Irrigation': 0.25, 'Precipitation': 0.25, 'Herbicide': 0.25}
  γ Pearson: nan  |  fit time: 88.1s
  [own graph]  Attr Pearson=0.8938 | Spearman=1.0 | RMSE=0.0646
  [true graph] Attr Pearson=0.8938 | Spearman=1.0 | RMSE=0.0646

  ── Fix 1+2 — λ₁=1e-2, T=6, iter=12K ──


  γ recovered: {'Nitrogen': 0.25, 'Irrigation': 0.25, 'Precipitation': 0.25, 'Herbicide': 0.25}
  γ Pearson: nan  |  fit time: 239.6s
  [own graph]  Attr Pearson=0.8938 | Spearman=1.0 | RMSE=0.0646
  [true graph] Attr Pearson=0.8938 | Spearman=1.0 | RMSE=0.0646

  ── Fix 1+2+3 — λ₁=1e-2, T=6, iter=12K, h=4d ──


  γ recovered: {'Nitrogen': 0.25, 'Irrigation': 0.25, 'Precipitation': 0.25, 'Herbicide': 0.25}
  γ Pearson: nan  |  fit time: 251.4s
  [own graph]  Attr Pearson=0.8938 | Spearman=1.0 | RMSE=0.0646
  [true graph] Attr Pearson=0.8938 | Spearman=1.0 | RMSE=0.0646

  ── λ₁ sweep — 5e-3, T=6, iter=12K, h=4d ──


  γ recovered: {'Nitrogen': 0.25, 'Irrigation': 0.25, 'Precipitation': 0.25, 'Herbicide': 0.25}
  γ Pearson: nan  |  fit time: 251.5s
  [own graph]  Attr Pearson=0.8938 | Spearman=1.0 | RMSE=0.0646
  [true graph] Attr Pearson=0.8938 | Spearman=1.0 | RMSE=0.0646

  ── λ₁ sweep — 1e-3, T=6, iter=12K, h=4d ──


  γ recovered: {'Nitrogen': 0.25, 'Irrigation': 0.25, 'Precipitation': 0.25, 'Herbicide': 0.25}
  γ Pearson: nan  |  fit time: 255.9s
  [own graph]  Attr Pearson=0.8938 | Spearman=1.0 | RMSE=0.0646
  [true graph] Attr Pearson=0.8938 | Spearman=1.0 | RMSE=0.0646

SWEEP SUMMARY — YIELD
  Ceiling: Attr Pearson=0.9959 | RMSE=0.0358
                                  Config  Gamma_Pearson  Attr_P_ownGraph  Attr_P_trueGraph  Attr_RMSE_own  Fit_s
Baseline (λ₁=3.5e-2, T=4, iter=8K, h=2d)            NaN           0.8938            0.8938         0.0646   88.5
                    Fix 1 — λ₁=1e-2 only            NaN           0.8938            0.8938         0.0646   88.1
        Fix 1+2 — λ₁=1e-2, T=6, iter=12K            NaN           0.8938            0.8938         0.0646  239.6
Fix 1+2+3 — λ₁=1e-2, T=6, iter=12K, h=4d            NaN           0.8938            0.8938         0.0646  251.4
    λ₁ sweep — 5e-3, T=6, iter=12K, h=4d            NaN           0.8938            0.8938         0.0646

  γ recovered: {'Organic Carbon': 0.1531, 'Nitrogen': 0.1424, 'Phosphorus': 0.0981, 'Potassium': 0.1529, 'pH': 0.2038, 'Zinc': 0.0829, 'Electrical Conductivity': 0.1668}
  γ Pearson: 0.1938  |  fit time: 864.2s
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  [own graph]  Attr Pearson=0.8057 | Spearman=0.8571 | RMSE=0.0752
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  [true graph] Attr Pearson=0.9149 | Spearman=0.8214 | RMSE=0.0522

  ── Fix 1 — λ₁=1e-2 only ──


  γ recovered: {'Organic Carbon': 0.1547, 'Nitrogen': 0.144, 'Phosphorus': 0.0948, 'Potassium': 0.1534, 'pH': 0.2104, 'Zinc': 0.0795, 'Electrical Conductivity': 0.1633}
  γ Pearson: 0.2028  |  fit time: 886.2s
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  [own graph]  Attr Pearson=0.9036 | Spearman=0.8214 | RMSE=0.0556
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  [true graph] Attr Pearson=0.9158 | Spearman=0.8214 | RMSE=0.0515

  ── Fix 1+2 — λ₁=1e-2, T=6, iter=12K ──


  γ recovered: {'Organic Carbon': 0.1454, 'Nitrogen': 0.1326, 'Phosphorus': 0.113, 'Potassium': 0.1311, 'pH': 0.187, 'Zinc': 0.1133, 'Electrical Conductivity': 0.1775}
  γ Pearson: -0.0986  |  fit time: 1409.2s
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  [own graph]  Attr Pearson=0.8967 | Spearman=0.8214 | RMSE=0.0613
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  [true graph] Attr Pearson=0.8748 | Spearman=0.8214 | RMSE=0.0642

  ── Fix 1+2+3 — λ₁=1e-2, T=6, iter=12K, h=4d ──


  γ recovered: {'Organic Carbon': 0.1528, 'Nitrogen': 0.1409, 'Phosphorus': 0.0812, 'Potassium': 0.0873, 'pH': 0.2071, 'Zinc': 0.1289, 'Electrical Conductivity': 0.2018}
  γ Pearson: -0.1952  |  fit time: 4895.9s
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  [own graph]  Attr Pearson=0.8139 | Spearman=0.75 | RMSE=0.074
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  [true graph] Attr Pearson=0.8547 | Spearman=0.75 | RMSE=0.0676

  ── λ₁ sweep — 5e-3, T=6, iter=12K, h=4d ──


In [4]:
import sys
sys.path.insert(0, 'DAGMA-DCE/src')


import time
import itertools
import math
import warnings
from typing import Dict, List, Set, Tuple

import numpy as np
import pandas as pd
import torch
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import networkx as nx
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

try:
    from DagmaDCE import nonlinear_dce
    HAS_DCE = True
    print("ok DagmaDCE available")
except ImportError as e:
    HAS_DCE = False
    print(f"x DagmaDCE not found: {e}")

try:
    from dagma.linear import DagmaLinear
    HAS_DAGMA = True
    print("ok dagma (standard linear) available")
except ImportError:
    HAS_DAGMA = False
    print("x dagma not found")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device: {device}\n")

# ── Config ────────────────────────────────────────────────────────────────────
CONFIG = {
    'CV_DATA_PATH': '../../../../feature_data_valuation/datasets/synthetic_cardiovascular_risk/cv_risk.csv',
    'LC_DATA_PATH': '../../../../feature_data_valuation/datasets/synthetic_lung_cancer_risk/lung_cancer_risk.csv',
    'TEST_SIZE':    0.20,
    'RANDOM_SEED':  42,
    'DCE_LAMBDA1':     1e-2,
    'DCE_LAMBDA2':     5e-3,
    'DCE_LR':          2e-4,
    'DCE_MU_FACTOR':   0.1,
    'DCE_MU_INIT':     0.1,
    'DCE_T':           6,
    'DCE_WARM_ITER':   10_000,
    'DCE_MAX_ITER':    12_000,
    'DCE_HIDDEN_MULT': 4,
    # Causal SHAP - exact engine (d=3 -> 8 coalitions, d=4 -> 16 coalitions)
    'M': 128,
    'K': 100,
}

# ── Dataset definitions ───────────────────────────────────────────────────────

LC_FEATURES   = ['smoking', 'stress', 'drink_coffee']
LC_TARGET     = 'lung_cancer_risk'
LC_TRUE_EDGES = {('smoking', 'drink_coffee'), ('stress', 'drink_coffee')}

def compute_lc_gamma() -> Dict[str, float]:
    """
    lung_cancer_risk = 2.0*smoking + 1.2*stress + eps,  eps~N(0,3)
    smoking, stress ~ N(5,2) => Var=2 each, independent.
    drink_coffee has no direct path to target => gamma=0.

    gamma_i = beta_i^2 * Var(Xi) / Var(Y)  for independent exogenous sources.
    Normalise smoking and stress so they sum to 1.
    """
    var_s, var_st, var_eps = 2.0, 2.0, 3.0
    var_y = 4.0 * var_s + 1.44 * var_st + var_eps     # 13.88
    raw = {
        'smoking':      4.0  * var_s  / var_y,         # 0.5764
        'stress':       1.44 * var_st / var_y,          # 0.2075
        'drink_coffee': 0.0,
    }
    tot = sum(raw.values())
    return {f: v / tot for f, v in raw.items()} if tot > 1e-12 else raw


CV_FEATURES   = ['diet_score', 'sleep_duration', 'family_history', 'bmi']
CV_TARGET     = 'cv_risk'
CV_TRUE_EDGES = {('diet_score', 'bmi'), ('sleep_duration', 'bmi')}

def compute_cv_gamma() -> Dict[str, float]:
    """
    bmi     = 0.4*diet + 0.5*sleep + eps_bmi,  eps_bmi~N(0,1)
    cv_risk = 1.5*bmi  + eps_risk,             eps_risk~N(2,3)
    diet    ~ Uniform(1,10) => Var = 81/12 = 6.75
    sleep   ~ N(8,4)        => Var = 4
    family  ~ N(4,2)        => no path to target

    With bmi observed as a feature: diet/sleep have zero DIRECT effect
    on cv_risk. All their influence is mediated through bmi.

    Var(bmi)     = 0.16*6.75 + 0.25*4 + 1 = 3.08
    Var(cv_risk) = 2.25*3.08 + 3          = 9.93
    gamma_bmi    = 2.25*3.08 / 9.93       = 0.6979
    """
    var_diet  = (9.0 ** 2) / 12.0   # 6.75
    var_sleep = 4.0
    var_bmi   = 0.16 * var_diet + 0.25 * var_sleep + 1.0   # 3.08
    var_cv    = 2.25 * var_bmi + 3.0                        # 9.93
    return {
        'diet_score':     0.0,
        'sleep_duration': 0.0,
        'family_history': 0.0,
        'bmi':            round(2.25 * var_bmi / var_cv, 6),   # ~0.6979
    }

np.random.seed(CONFIG['RANDOM_SEED'])
torch.manual_seed(CONFIG['RANDOM_SEED'])


# ── Utilities ─────────────────────────────────────────────────────────────────
def normalise(g: Dict[str, float], features: List[str]) -> Dict[str, float]:
    g   = {f: max(v, 0.0) for f, v in g.items()}
    tot = sum(g.values())
    return {f: v / tot for f, v in g.items()} if tot > 1e-12 \
        else {f: 1.0 / len(features) for f in features}


def compare_graphs(recovered: nx.DiGraph, true_edges: set) -> dict:
    rec    = set(recovered.edges())
    tp     = len(rec & true_edges)
    fp     = len(rec - true_edges)
    fn     = len(true_edges - rec)
    shd    = fp + fn
    prec   = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1     = 2 * prec * recall / (prec + recall) if (prec + recall) > 0 else 0.0
    return {'SHD': shd, 'F1': round(f1, 3),
            'Recovered': list(rec), 'Missing': list(true_edges - rec),
            'Extra': list(rec - true_edges)}


def gamma_metrics(true_gamma, est_gamma, features):
    gt = np.array([true_gamma[f] for f in features])
    eg = np.array([est_gamma[f]  for f in features])
    if np.std(eg) < 1e-10:
        return {'Gamma_Pearson': float('nan'), 'Gamma_Spearman': float('nan'),
                'Gamma_RMSE': round(float(np.sqrt(np.mean((eg - gt) ** 2))), 4)}
    pr, _ = pearsonr(eg, gt)
    sr, _ = spearmanr(eg, gt)
    return {'Gamma_Pearson': round(float(pr), 4), 'Gamma_Spearman': round(float(sr), 4),
            'Gamma_RMSE': round(float(np.sqrt(np.mean((eg - gt) ** 2))), 4)}


def attribution_metrics(true_gamma, global_abs, features):
    gt  = np.array([true_gamma[f]       for f in features])
    raw = np.array([global_abs.get(f, 0) for f in features])
    sv  = raw / raw.sum() if raw.sum() > 1e-12 else raw
    pr, _ = pearsonr(sv, gt)
    sr, _ = spearmanr(sv, gt)
    cos   = float(np.dot(sv, gt) / (np.linalg.norm(sv) * np.linalg.norm(gt) + 1e-12))
    rmse  = float(np.sqrt(np.mean((sv - gt) ** 2)))
    return {'Attr_Pearson': round(float(pr), 4), 'Attr_Spearman': round(float(sr), 4),
            'Attr_Cosine': round(cos, 4), 'Attr_RMSE': round(rmse, 4)}


# ── DAGMA-DCE runner ──────────────────────────────────────────────────────────
def run_dce_mlp(X, y, features, target_name, cfg):
    all_cols = features + [target_name]
    n_vars   = len(all_cols)
    X_full   = X.copy()
    X_full[target_name] = y.values if hasattr(y, 'values') else y
    data = torch.tensor(X_full[all_cols].values.astype(np.float64), dtype=torch.float64)

    hidden   = max(10, cfg['DCE_HIDDEN_MULT'] * n_vars)
    eq_model = nonlinear_dce.DagmaMLP_DCE(dims=[n_vars, hidden, 1], bias=True)
    model    = nonlinear_dce.DagmaDCE(eq_model, use_mse_loss=True)
    print(f"  Running DAGMA-DCE MLP on {n_vars} variables (hidden={hidden})...")

    W_est = model.fit(data,
        lambda1=cfg['DCE_LAMBDA1'], lambda2=cfg['DCE_LAMBDA2'],
        lr=cfg['DCE_LR'], mu_factor=cfg['DCE_MU_FACTOR'], mu_init=cfg['DCE_MU_INIT'],
        T=cfg['DCE_T'], warm_iter=cfg['DCE_WARM_ITER'], max_iter=cfg['DCE_MAX_ITER'])

    target_idx  = all_cols.index(target_name)
    raw_effects = {f: abs(float(W_est[i, target_idx])) for i, f in enumerate(features)}
    gamma       = normalise(raw_effects, features)
    G = nx.DiGraph(); G.add_nodes_from(features)
    for i, fi in enumerate(features):
        for j, fj in enumerate(features):
            if i != j and abs(W_est[i, j]) > 0.05:
                G.add_edge(fi, fj, weight=float(W_est[i, j]))
    return W_est, gamma, G


# ── Standard DAGMA runner ─────────────────────────────────────────────────────
def run_standard_dagma(X, y, features, target_name):
    all_cols = features + [target_name]
    X_full   = X.copy()
    X_full[target_name] = y.values if hasattr(y, 'values') else y
    W = DagmaLinear(loss_type='l2').fit(X_full[all_cols].values.astype(float), lambda1=0.02)
    target_idx  = all_cols.index(target_name)
    raw_effects = {f: abs(float(W[i, target_idx])) for i, f in enumerate(features)}
    gamma       = normalise(raw_effects, features)
    G = nx.DiGraph(); G.add_nodes_from(features)
    for i, fi in enumerate(features):
        for j, fj in enumerate(features):
            if i != j and abs(W[i, j]) > 0.05:
                G.add_edge(fi, fj, weight=float(W[i, j]))
    return W, gamma, G


# ── Causal SHAP - exact engine ────────────────────────────────────────────────
class LinearMechanisms:
    def __init__(self, dag, features):
        self.dag = dag; self.features = features; self.mech = {}
        self.topo = list(nx.topological_sort(dag))

    def fit(self, X):
        for var in self.features:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {'parents': [], 'intercept': float(X[var].mean()),
                                  'coef': None, 'sigma': float(X[var].std(ddof=0))}
            else:
                lr    = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mech[var] = {'parents': parents, 'intercept': float(lr.intercept_),
                                  'coef': lr.coef_.astype(float),
                                  'sigma': float(np.std(resid, ddof=0))}
        return self

    def sample_batch(self, S, x, M, rng):
        X_out = np.empty((M, len(self.features)))
        fi    = {f: i for i, f in enumerate(self.features)}
        sim   = {}
        for var in self.topo:
            vi = fi[var]
            if var in S:
                val = float(x[var]); X_out[:, vi] = val; sim[var] = np.full(M, val); continue
            m = self.mech[var]
            mu = np.full(M, m['intercept']) if not m['parents'] else \
                 m['intercept'] + np.stack(
                     [sim[p] if p in sim else np.full(M, float(x[p])) for p in m['parents']],
                     axis=1) @ m['coef']
            X_out[:, vi] = mu if m['sigma'] < 1e-8 else rng.normal(mu, m['sigma'], size=M)
            sim[var] = X_out[:, vi]
        return X_out


def run_causal_shap_exact(model, gamma, dag, Xtr, Xte, cfg):
    rng      = np.random.default_rng(cfg['RANDOM_SEED'])
    features = list(Xtr.columns)
    n        = len(features)
    kw       = {sz: float(math.factorial(sz) * math.factorial(n - sz - 1) / math.factorial(n))
                for sz in range(n)}
    sem = LinearMechanisms(dag, features).fit(Xtr)
    Ef  = float(model.predict(Xtr.values).mean())
    M   = cfg['M']; K = min(cfg['K'], len(Xte))
    idx = rng.choice(len(Xte), size=K, replace=False)
    phis = []
    for cnt, i in enumerate(idx):
        if (cnt + 1) % 20 == 0:
            print(f"      instance {cnt + 1}/{K} ...", flush=True)
        x  = Xte.iloc[i]
        fx = float(model.predict(x.values.reshape(1, -1))[0])
        cache = {
            frozenset(sub): float(np.mean(model.predict(sem.sample_batch(frozenset(sub), x, M, rng))))
            for r in range(n + 1) for sub in itertools.combinations(features, r)
        }
        phi = np.zeros(n)
        for idx_f, feat in enumerate(features):
            rest = [f for f in features if f != feat]
            for r in range(n):
                for sub in itertools.combinations(rest, r):
                    S = frozenset(sub)
                    phi[idx_f] += kw[len(sub)] * gamma[feat] * (cache[S | {feat}] - cache[S])
        s = float(phi.sum())
        if abs(s) > 1e-12: phi = phi * ((fx - Ef) / s)
        phis.append(pd.Series(phi, index=features))
    return pd.concat(phis, axis=1).abs().mean(axis=1)


# ── Dataset runner ────────────────────────────────────────────────────────────
def run_dataset(name, features, true_edges, true_gamma, Xtr, Xte, ytr, cfg):
    print(f"\n{'='*80}\nDATASET: {name.upper()}\n{'='*80}")
    print(f"  True gamma: { {f: round(v, 4) for f, v in true_gamma.items()} }")
    print(f"  True edges: {true_edges}")

    target_name = f'{name}_target'
    results     = []

    def log4(label, gm, shd, f1, am):
        gp = f"{gm['Gamma_Pearson']:.4f}" if (gm and not np.isnan(gm['Gamma_Pearson'])) else "NaN "
        print(f"  [{label}]")
        print(f"    gamma Pearson={gp} | Graph SHD={shd} | Graph F1={f1:.3f}")
        print(f"    Attr  Pearson={am['Attr_Pearson']:.4f} | Spearman={am['Attr_Spearman']:.4f}"
              f" | Cosine={am['Attr_Cosine']:.4f} | RMSE={am['Attr_RMSE']:.4f}")

    xgb = XGBRegressor(n_estimators=160, max_depth=5, learning_rate=0.08,
                       subsample=0.9, colsample_bytree=0.9,
                       random_state=cfg['RANDOM_SEED'], verbosity=0,
                       n_jobs=1, tree_method='hist')
    xgb.fit(Xtr.values, ytr)

    true_dag = nx.DiGraph(); true_dag.add_nodes_from(features)
    for u, v in true_edges: true_dag.add_edge(u, v)
    star_dag = nx.DiGraph(); star_dag.add_nodes_from(features)

    # ── Method 1: DAGMA-DCE MLP ──────────────────────────────────────────────
    print("\n--- Method 1: DAGMA-DCE (non-linear MLP) ---")
    t0 = time.time()
    W_nl, gamma_nl, dag_nl = run_dce_mlp(Xtr, ytr, features, target_name, cfg)
    t_nl = time.time() - t0
    gc_nl = compare_graphs(dag_nl, true_edges)
    gm_nl = gamma_metrics(true_gamma, gamma_nl, features)
    print(f"  gamma recovered: { {f: round(v, 4) for f, v in gamma_nl.items()} }")
    print(f"  Graph recovered: {gc_nl['Recovered']}")

    am_1a = attribution_metrics(true_gamma,
                run_causal_shap_exact(xgb, gamma_nl, dag_nl, Xtr, Xte, cfg), features)
    log4("DCE-MLP gamma + DCE-MLP graph", gm_nl, gc_nl['SHD'], gc_nl['F1'], am_1a)
    results.append({'Method': 'DAGMA-DCE MLP (own graph + own gamma)',
                    **gm_nl, 'Graph_SHD': gc_nl['SHD'], 'Graph_F1': gc_nl['F1'],
                    **am_1a, 'Time_s': round(t_nl, 1)})

    am_1b = attribution_metrics(true_gamma,
                run_causal_shap_exact(xgb, gamma_nl, true_dag, Xtr, Xte, cfg), features)
    log4("DCE-MLP gamma + true graph", gm_nl, 0, 1.0, am_1b)
    results.append({'Method': 'DAGMA-DCE MLP (true graph + own gamma)',
                    **gm_nl, 'Graph_SHD': 0, 'Graph_F1': 1.0,
                    **am_1b, 'Time_s': round(t_nl, 1)})

    # ── Method 2: Standard DAGMA ──────────────────────────────────────────────
    print("\n--- Method 2: Standard DAGMA (linear) ---")
    t0 = time.time()
    W_d, gamma_d, dag_d = run_standard_dagma(Xtr, ytr, features, target_name)
    t_d = time.time() - t0
    gc_d = compare_graphs(dag_d, true_edges)
    gm_d = gamma_metrics(true_gamma, gamma_d, features)
    print(f"  gamma recovered: { {f: round(v, 4) for f, v in gamma_d.items()} }")
    print(f"  Graph recovered: {gc_d['Recovered']}")

    am_2a = attribution_metrics(true_gamma,
                run_causal_shap_exact(xgb, gamma_d, dag_d, Xtr, Xte, cfg), features)
    log4("DAGMA gamma + DAGMA graph", gm_d, gc_d['SHD'], gc_d['F1'], am_2a)
    results.append({'Method': 'Standard DAGMA (own graph + own gamma)',
                    **gm_d, 'Graph_SHD': gc_d['SHD'], 'Graph_F1': gc_d['F1'],
                    **am_2a, 'Time_s': round(t_d, 1)})

    am_2b = attribution_metrics(true_gamma,
                run_causal_shap_exact(xgb, gamma_d, true_dag, Xtr, Xte, cfg), features)
    log4("DAGMA gamma + true graph", gm_d, 0, 1.0, am_2b)
    results.append({'Method': 'Standard DAGMA (true graph + own gamma)',
                    **gm_d, 'Graph_SHD': 0, 'Graph_F1': 1.0,
                    **am_2b, 'Time_s': round(t_d, 1)})

    # ── Ceiling A: true gamma + true graph ────────────────────────────────────
    print("\n--- Ceiling A: True gamma + True graph ---")
    am_ca = attribution_metrics(true_gamma,
                run_causal_shap_exact(xgb, true_gamma, true_dag, Xtr, Xte, cfg), features)
    log4("true gamma + true graph", None, 0, 1.0, am_ca)
    results.append({'Method': 'TRUE gamma + TRUE graph (ceiling A)',
                    'Gamma_Pearson': 1.0, 'Gamma_Spearman': 1.0, 'Gamma_RMSE': 0.0,
                    'Graph_SHD': 0, 'Graph_F1': 1.0, **am_ca, 'Time_s': 0.0})

    # ── Ceiling B: true gamma + star DAG ──────────────────────────────────────
    print("\n--- Ceiling B: True gamma + Star DAG (no edges) ---")
    gc_star = compare_graphs(star_dag, true_edges)
    am_cb   = attribution_metrics(true_gamma,
                run_causal_shap_exact(xgb, true_gamma, star_dag, Xtr, Xte, cfg), features)
    log4("true gamma + star DAG", None, gc_star['SHD'], gc_star['F1'], am_cb)
    results.append({'Method': 'TRUE gamma + Star DAG (no edges, ceiling B)',
                    'Gamma_Pearson': 1.0, 'Gamma_Spearman': 1.0, 'Gamma_RMSE': 0.0,
                    'Graph_SHD': gc_star['SHD'], 'Graph_F1': 0.0, **am_cb, 'Time_s': 0.0})

    df = pd.DataFrame(results)
    disp = ['Method', 'Gamma_Pearson', 'Graph_SHD', 'Graph_F1',
            'Attr_Pearson', 'Attr_Spearman', 'Attr_Cosine', 'Attr_RMSE']
    print(f"\n{'='*110}\nRESULTS SUMMARY — {name.upper()}\n{'='*110}")
    print(df[disp].to_string(index=False))
    print("\n  Gamma_Pearson = alignment of recovered gamma with analytical ground truth")
    print("  Graph_SHD     = structural Hamming distance vs true graph (0=perfect)")
    print("  Attr_*        = attribution quality vs GT gamma")
    return df


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print("DAGMA-DCE vs Standard DAGMA — CV Risk & Lung Cancer (Synthetic, Known DGP)")
    print("=" * 80)

    lc_gamma = compute_lc_gamma()
    cv_gamma = compute_cv_gamma()
    print(f"\nAnalytical gamma — Lung Cancer: {lc_gamma}")
    print(f"Analytical gamma — CV Risk:     {cv_gamma}\n")

    # ── Lung Cancer ───────────────────────────────────────────────────────────
    lc = pd.read_csv(CONFIG['LC_DATA_PATH']).drop(columns=['Unnamed: 0'], errors='ignore')
    Xtr_lc_raw, Xte_lc_raw, ytr_lc, _ = train_test_split(
        lc[LC_FEATURES], lc[LC_TARGET],
        test_size=CONFIG['TEST_SIZE'], random_state=CONFIG['RANDOM_SEED'], shuffle=True)
    scaler_lc = StandardScaler()
    Xtr_lc = pd.DataFrame(scaler_lc.fit_transform(Xtr_lc_raw), columns=LC_FEATURES)
    Xte_lc = pd.DataFrame(scaler_lc.transform(Xte_lc_raw),     columns=LC_FEATURES)

    df_lc = run_dataset('Lung_Cancer', LC_FEATURES, LC_TRUE_EDGES, lc_gamma,
                        Xtr_lc, Xte_lc, ytr_lc.values, CONFIG)
    df_lc.to_csv('dagma_dce_results_lung_cancer.csv', index=False)

    # ── CV Risk ───────────────────────────────────────────────────────────────
    cv = pd.read_csv(CONFIG['CV_DATA_PATH']).drop(columns=['Unnamed: 0'], errors='ignore')
    Xtr_cv_raw, Xte_cv_raw, ytr_cv, _ = train_test_split(
        cv[CV_FEATURES], cv[CV_TARGET],
        test_size=CONFIG['TEST_SIZE'], random_state=CONFIG['RANDOM_SEED'], shuffle=True)
    scaler_cv = StandardScaler()
    Xtr_cv = pd.DataFrame(scaler_cv.fit_transform(Xtr_cv_raw), columns=CV_FEATURES)
    Xte_cv = pd.DataFrame(scaler_cv.transform(Xte_cv_raw),     columns=CV_FEATURES)

    df_cv = run_dataset('CV_Risk', CV_FEATURES, CV_TRUE_EDGES, cv_gamma,
                        Xtr_cv, Xte_cv, ytr_cv.values, CONFIG)
    df_cv.to_csv('dagma_dce_results_cv_risk.csv', index=False)

    print("\nSaved: dagma_dce_results_lung_cancer.csv, dagma_dce_results_cv_risk.csv")


if __name__ == '__main__':
    main()

✓ DagmaDCE available
✓ dagma (standard linear) available
  Device: cuda

DAGMA-DCE vs Standard DAGMA — CV Risk & Lung Cancer (Synthetic, Known DGP)

Analytical gamma — Lung Cancer: {'smoking': 0.7352941176470589, 'stress': 0.2647058823529412, 'drink_coffee': 0.0}
Analytical gamma — CV Risk:     {'diet_score': 0.0, 'sleep_duration': 0.0, 'family_history': 0.0, 'bmi': 0.697885}


DATASET: LUNG_CANCER
  True gamma: {'smoking': 0.7353, 'stress': 0.2647, 'drink_coffee': 0.0}
  True edges: {('smoking', 'drink_coffee'), ('stress', 'drink_coffee')}

--- Method 1: DAGMA-DCE (non-linear MLP) ---
  Running DAGMA-DCE MLP on 4 variables (hidden=16)...


  gamma recovered: {'smoking': 0.567, 'stress': 0.3488, 'drink_coffee': 0.0842}
  Graph recovered: [('drink_coffee', 'smoking'), ('smoking', 'stress'), ('drink_coffee', 'stress')]
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  [DCE-MLP gamma + DCE-MLP graph]
    gamma Pearson=0.9768 | Graph SHD=5 | Graph F1=0.000
    Attr  Pearson=0.9982 | Spearman=1.0000 | Cosine=0.8197 | RMSE=0.2606
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  [DCE-MLP gamma + true graph]
    gamma Pearson=0.9768 | Graph SHD=0 | Graph F1=1.000
    Attr  Pearson=0.9468 | Spearman=1.0000 | Cosine=0.9668 | RMSE=0.1219

--- Method 2: Standard DAGMA (linear) ---


  gamma recovered: {'smoking': 0.5672, 'stress': 0.3461, 'drink_coffee': 0.0866}
  Graph recovered: [('drink_coffee', 'smoking'), ('smoking', 'stress'), ('drink_coffee', 'stress')]
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  [DAGMA gamma + DAGMA graph]
    gamma Pearson=0.9788 | Graph SHD=5 | Graph F1=0.000
    Attr  Pearson=0.9999 | Spearman=1.0000 | Cosine=0.8166 | RMSE=0.2624
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  [DAGMA gamma + true graph]
    gamma Pearson=0.9788 | Graph SHD=0 | Graph F1=1.000
    Attr  Pearson=0.9345 | Spearman=1.0000 | Cosine=0.9619 | RMSE=0.1291

--- Ceiling A: True gamma + True graph ---
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  [true gamma + true graph]
    gamma Pearson=NaN  | Graph SHD=0 

  gamma recovered: {'diet_score': 0.0171, 'sleep_duration': 0.0226, 'family_history': 0.0294, 'bmi': 0.9309}
  Graph recovered: [('bmi', 'sleep_duration'), ('diet_score', 'sleep_duration'), ('diet_score', 'bmi')]
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  [DCE-MLP gamma + DCE-MLP graph]
    gamma Pearson=0.9999 | Graph SHD=3 | Graph F1=0.400
    Attr  Pearson=1.0000 | Spearman=0.7746 | Cosine=0.9997 | RMSE=0.1326
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  [DCE-MLP gamma + true graph]
    gamma Pearson=0.9999 | Graph SHD=0 | Graph F1=1.000
    Attr  Pearson=1.0000 | Spearman=0.7746 | Cosine=0.9997 | RMSE=0.1313

--- Method 2: Standard DAGMA (linear) ---


  gamma recovered: {'diet_score': 0.0, 'sleep_duration': 0.0, 'family_history': 0.0, 'bmi': 1.0}
  Graph recovered: [('bmi', 'sleep_duration'), ('diet_score', 'sleep_duration'), ('diet_score', 'bmi')]
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  [DAGMA gamma + DAGMA graph]
    gamma Pearson=1.0000 | Graph SHD=3 | Graph F1=0.400
    Attr  Pearson=1.0000 | Spearman=1.0000 | Cosine=1.0000 | RMSE=0.1511
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  [DAGMA gamma + true graph]
    gamma Pearson=1.0000 | Graph SHD=0 | Graph F1=1.000
    Attr  Pearson=1.0000 | Spearman=1.0000 | Cosine=1.0000 | RMSE=0.1511

--- Ceiling A: True gamma + True graph ---
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  [true gamma + true graph]
    gamma Pearson

In [7]:
import sys
sys.path.insert(0, 'DAGMA-DCE/src')


import time
import itertools
import math
import warnings
from typing import Dict, List, Set, Tuple

import numpy as np
import pandas as pd
import torch
import shap
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import networkx as nx
from xgboost import XGBRegressor

from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz

warnings.filterwarnings("ignore")

try:
    from DagmaDCE import nonlinear_dce
    HAS_DCE = True
    print("ok DagmaDCE available")
except ImportError:
    HAS_DCE = False
    print("x DagmaDCE not found")

try:
    from dagma.linear import DagmaLinear
    HAS_DAGMA = True
    print("ok dagma available")
except ImportError:
    HAS_DAGMA = False
    print("x dagma not found")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device: {device}\n")

# ── Config ────────────────────────────────────────────────────────────────────
CONFIG = {
    'CV_DATA_PATH': '../../../../feature_data_valuation/datasets/synthetic_cardiovascular_risk/cv_risk.csv',
    'LC_DATA_PATH': '../../../../feature_data_valuation/datasets/synthetic_lung_cancer_risk/lung_cancer_risk.csv',
    'TEST_SIZE':   0.20,
    'RANDOM_SEED': 42,
    'DCE_LAMBDA1':     1e-2,
    'DCE_LAMBDA2':     5e-3,
    'DCE_LR':          2e-4,
    'DCE_MU_FACTOR':   0.1,
    'DCE_MU_INIT':     0.1,
    'DCE_T':           6,
    'DCE_WARM_ITER':   10_000,
    'DCE_MAX_ITER':    12_000,
    'DCE_HIDDEN_MULT': 4,
    'M':  128,    # SEM samples per coalition
    'K':  100,    # test instances for global attribution
}

# ── Dataset definitions ───────────────────────────────────────────────────────
LC_FEATURES   = ['smoking', 'stress', 'drink_coffee']
LC_TARGET     = 'lung_cancer_risk'
LC_TRUE_EDGES = {('smoking', 'drink_coffee'), ('stress', 'drink_coffee')}

def compute_lc_gamma() -> Dict[str, float]:
    """
    lung_cancer_risk = 2.0*smoking + 1.2*stress + eps,  eps~N(0,3)
    smoking, stress ~ N(5,2). drink_coffee is a collider — zero direct effect.
    gamma_i = beta_i^2 * Var(Xi) / Var(Y), normalised over causal features only.
    """
    var_s, var_st, var_eps = 2.0, 2.0, 3.0
    var_y = 4.0*var_s + 1.44*var_st + var_eps   # 13.88
    raw = {'smoking': 4.0*var_s/var_y, 'stress': 1.44*var_st/var_y, 'drink_coffee': 0.0}
    tot = sum(raw.values())
    return {f: v/tot for f, v in raw.items()}


CV_FEATURES   = ['diet_score', 'sleep_duration', 'family_history', 'bmi']
CV_TARGET     = 'cv_risk'
CV_TRUE_EDGES = {('diet_score', 'bmi'), ('sleep_duration', 'bmi')}

def compute_cv_gamma() -> Dict[str, float]:
    """
    bmi = 0.4*diet + 0.5*sleep + eps_bmi,  cv_risk = 1.5*bmi + eps_risk.
    With bmi observed: diet/sleep/family have zero DIRECT effect on cv_risk.
    gamma_bmi = 2.25*Var(bmi) / Var(cv_risk) = 0.6979.
    """
    var_diet = (9.0**2)/12.0; var_sleep = 4.0
    var_bmi  = 0.16*var_diet + 0.25*var_sleep + 1.0   # 3.08
    var_cv   = 2.25*var_bmi + 3.0                      # 9.93
    return {'diet_score': 0.0, 'sleep_duration': 0.0,
            'family_history': 0.0, 'bmi': round(2.25*var_bmi/var_cv, 6)}

np.random.seed(CONFIG['RANDOM_SEED'])
torch.manual_seed(CONFIG['RANDOM_SEED'])


# ── Metrics ───────────────────────────────────────────────────────────────────
def normalise(g: Dict[str, float], features: List[str]) -> Dict[str, float]:
    g   = {f: max(v, 0.0) for f, v in g.items()}
    tot = sum(g.values())
    return {f: v/tot for f, v in g.items()} if tot > 1e-12 \
        else {f: 1.0/len(features) for f in features}

def gamma_pearson(true_g, est_g, features):
    gt = np.array([true_g[f] for f in features])
    eg = np.array([est_g[f]  for f in features])
    if np.std(eg) < 1e-10: return float('nan')
    return round(float(pearsonr(eg, gt)[0]), 4)

def attribution_metrics(true_gamma, global_abs, features):
    gt  = np.array([true_gamma[f]       for f in features])
    raw = np.array([global_abs.get(f,0) for f in features])
    sv  = raw/raw.sum() if raw.sum() > 1e-12 else raw
    pr,_ = pearsonr(sv, gt); sr,_ = spearmanr(sv, gt)
    cos  = float(np.dot(sv,gt)/(np.linalg.norm(sv)*np.linalg.norm(gt)+1e-12))
    rmse = float(np.sqrt(np.mean((sv-gt)**2)))
    return {'Attr_Pearson':round(float(pr),4), 'Attr_Spearman':round(float(sr),4),
            'Attr_Cosine':round(cos,4), 'Attr_RMSE':round(rmse,4)}

def compare_graphs(recovered: nx.DiGraph, true_edges: set) -> dict:
    rec = set(recovered.edges())
    tp  = len(rec & true_edges); fp = len(rec-true_edges); fn = len(true_edges-rec)
    shd = fp+fn
    prec   = tp/(tp+fp) if (tp+fp)>0 else 0.0
    recall = tp/(tp+fn) if (tp+fn)>0 else 0.0
    f1     = 2*prec*recall/(prec+recall) if (prec+recall)>0 else 0.0
    return {'SHD':shd,'F1':round(f1,3),'Recovered':list(rec),
            'Missing':list(true_edges-rec),'Extra':list(rec-true_edges)}


# ── Standard DAGMA ────────────────────────────────────────────────────────────
def run_standard_dagma(X, y, features, target_name):
    all_cols = features + [target_name]
    X_full   = X.copy(); X_full[target_name] = y.values if hasattr(y,'values') else y
    W = DagmaLinear(loss_type='l2').fit(X_full[all_cols].values.astype(float), lambda1=0.02)
    target_idx  = all_cols.index(target_name)
    raw_effects = {f: abs(float(W[i,target_idx])) for i,f in enumerate(features)}
    gamma = normalise(raw_effects, features)
    G = nx.DiGraph(); G.add_nodes_from(features)
    for i,fi in enumerate(features):
        for j,fj in enumerate(features):
            if i!=j and abs(W[i,j])>0.05:
                G.add_edge(fi, fj, weight=float(W[i,j]))
    return W, gamma, G


# ── DAGMA-DCE (MLP) runner ────────────────────────────────────────────────────
def run_dce_mlp(X, y, features, target_name, cfg):
    """
    DAGMA-DCE with MLP backbone.
    W_est[i,j] = ∂E[Xⱼ]/∂Xᵢ  (differential causal effect, directly = γ).
    Gamma extracted as |W_est[i, target_idx]| normalised over features.
    """
    if not HAS_DCE:
        print("  x DagmaDCE not available — skipping")
        n = len(features)
        G = nx.DiGraph(); G.add_nodes_from(features)
        return None, {f: 1.0/n for f in features}, G

    all_cols = features + [target_name]
    n_vars   = len(all_cols)
    X_full   = X.copy(); X_full[target_name] = y.values if hasattr(y, 'values') else y
    data     = torch.tensor(X_full[all_cols].values.astype(np.float64), dtype=torch.float64)

    hidden   = max(10, cfg['DCE_HIDDEN_MULT'] * n_vars)
    eq_model = nonlinear_dce.DagmaMLP_DCE(dims=[n_vars, hidden, 1], bias=True)
    model    = nonlinear_dce.DagmaDCE(eq_model, use_mse_loss=True)
    print(f"  Running DAGMA-DCE MLP on {n_vars} variables (hidden={hidden})...")

    W_est = model.fit(data,
        lambda1=cfg['DCE_LAMBDA1'], lambda2=cfg['DCE_LAMBDA2'],
        lr=cfg['DCE_LR'], mu_factor=cfg['DCE_MU_FACTOR'], mu_init=cfg['DCE_MU_INIT'],
        T=cfg['DCE_T'], warm_iter=cfg['DCE_WARM_ITER'], max_iter=cfg['DCE_MAX_ITER'])

    target_idx  = all_cols.index(target_name)
    raw_effects = {f: abs(float(W_est[i, target_idx])) for i, f in enumerate(features)}
    gamma       = normalise(raw_effects, features)

    G = nx.DiGraph(); G.add_nodes_from(features)
    for i, fi in enumerate(features):
        for j, fj in enumerate(features):
            if i != j and abs(W_est[i, j]) > 0.05:
                G.add_edge(fi, fj, weight=float(W_est[i, j]))
    return W_est, gamma, G


def run_pc(X: pd.DataFrame, y, features: List[str], target: str,
           alpha: float = 0.05) -> Tuple[nx.DiGraph, Dict[str, float]]:
    """
    Run PC algorithm.
    X  — feature-only DataFrame (no target column).
    y  — target array/Series, appended internally for the full PC run.

    Two-pass approach:
      Pass 1: PC on features only  → inter-feature edge skeleton
      Pass 2: PC on features+target → find which features connect to target

    Returns:
      G_feat            — DiGraph over features
      gamma             — uniform over features connected to target
      connected_to_target — list of those features
    """
    y_vals = y.values if hasattr(y, 'values') else np.array(y)

    # ── Pass 1: inter-feature skeleton ───────────────────────────────────────
    data_feat  = X[features].values.astype(float)
    cg_feat    = pc(data_feat, alpha=alpha, indep_test=fisherz,
                    node_names=features, verbose=False, show_progress=False)
    adj_feat   = cg_feat.G.graph
    feat_nodes = [n.get_name() for n in cg_feat.G.get_nodes()]
    n_feat     = len(feat_nodes)

    G_feat = nx.DiGraph(); G_feat.add_nodes_from(features)
    for i in range(n_feat):
        for j in range(n_feat):
            if i == j: continue
            if adj_feat[i,j] == 1 and adj_feat[j,i] == -1:        # i -> j
                G_feat.add_edge(feat_nodes[i], feat_nodes[j])
            elif adj_feat[i,j] == -1 and adj_feat[j,i] == -1 and i < j:  # undirected
                G_feat.add_edge(feat_nodes[i], feat_nodes[j])
                G_feat.add_edge(feat_nodes[j], feat_nodes[i])

    # ── Pass 2: features + target to find target connections ─────────────────
    all_cols  = features + [target]
    X_all     = X[features].copy()
    X_all[target] = y_vals
    data_all  = X_all[all_cols].values.astype(float)

    cg_all    = pc(data_all, alpha=alpha, indep_test=fisherz,
                   node_names=all_cols, verbose=False, show_progress=False)
    adj_all   = cg_all.G.graph
    all_nodes = [n.get_name() for n in cg_all.G.get_nodes()]
    tgt_idx   = all_nodes.index(target)

    connected_to_target = [
        f for f in features
        if adj_all[all_nodes.index(f), tgt_idx] != 0
        or adj_all[tgt_idx, all_nodes.index(f)] != 0
    ]

    # gamma: uniform over connected features; fall back to uniform all
    if connected_to_target:
        gamma = {f: (1.0/len(connected_to_target) if f in connected_to_target else 0.0)
                 for f in features}
    else:
        gamma = {f: 1.0/len(features) for f in features}

    return G_feat, gamma, connected_to_target


# ── Vanilla SHAP ──────────────────────────────────────────────────────────────
def run_vanilla_shap(model, Xtr: pd.DataFrame, Xte: pd.DataFrame,
                     features: List[str], K: int, seed: int) -> pd.Series:
    """TreeExplainer SHAP — no causal structure."""
    rng = np.random.default_rng(seed)
    K   = min(K, len(Xte))
    idx = rng.choice(len(Xte), size=K, replace=False)
    Xte_sample = Xte.iloc[idx]

    explainer = shap.TreeExplainer(model, data=Xtr, feature_perturbation='interventional')
    sv        = explainer.shap_values(Xte_sample)
    global_abs = pd.Series(np.abs(sv).mean(axis=0), index=features)
    return global_abs


# ── Causal SHAP exact engine ──────────────────────────────────────────────────
class LinearMechanisms:
    def __init__(self, dag, features):
        self.dag=dag; self.features=features; self.mech={}
        self.topo=list(nx.topological_sort(dag))

    def fit(self, X):
        for var in self.features:
            parents=list(self.dag.predecessors(var))
            if not parents:
                self.mech[var]={'parents':[],'intercept':float(X[var].mean()),
                                'coef':None,'sigma':float(X[var].std(ddof=0))}
            else:
                lr=LinearRegression().fit(X[parents].values,X[var].values)
                resid=X[var].values-lr.predict(X[parents].values)
                self.mech[var]={'parents':parents,'intercept':float(lr.intercept_),
                                'coef':lr.coef_.astype(float),'sigma':float(np.std(resid,ddof=0))}
        return self

    def sample_batch(self, S, x, M, rng):
        X_out=np.empty((M,len(self.features))); fi={f:i for i,f in enumerate(self.features)}; sim={}
        for var in self.topo:
            vi=fi[var]
            if var in S:
                val=float(x[var]); X_out[:,vi]=val; sim[var]=np.full(M,val); continue
            m=self.mech[var]
            mu=np.full(M,m['intercept']) if not m['parents'] else \
               m['intercept']+np.stack([sim[p] if p in sim else np.full(M,float(x[p]))
                                        for p in m['parents']],axis=1)@m['coef']
            X_out[:,vi]=mu if m['sigma']<1e-8 else rng.normal(mu,m['sigma'],size=M)
            sim[var]=X_out[:,vi]
        return X_out


def run_causal_shap_exact(model, gamma, dag, Xtr, Xte, cfg) -> pd.Series:
    rng=np.random.default_rng(cfg['RANDOM_SEED']); features=list(Xtr.columns); n=len(features)
    kw={sz:float(math.factorial(sz)*math.factorial(n-sz-1)/math.factorial(n)) for sz in range(n)}
    sem=LinearMechanisms(dag,features).fit(Xtr)
    Ef=float(model.predict(Xtr.values).mean())
    M=cfg['M']; K=min(cfg['K'],len(Xte))
    idx=rng.choice(len(Xte),size=K,replace=False); phis=[]
    for cnt,i in enumerate(idx):
        if (cnt+1)%20==0: print(f"      instance {cnt+1}/{K} ...",flush=True)
        x=Xte.iloc[i]; fx=float(model.predict(x.values.reshape(1,-1))[0])
        cache={frozenset(sub):float(np.mean(model.predict(sem.sample_batch(frozenset(sub),x,M,rng))))
               for r in range(n+1) for sub in itertools.combinations(features,r)}
        phi=np.zeros(n)
        for idx_f,feat in enumerate(features):
            rest=[f for f in features if f!=feat]
            for r in range(n):
                for sub in itertools.combinations(rest,r):
                    S=frozenset(sub)
                    phi[idx_f]+=kw[len(sub)]*gamma[feat]*(cache[S|{feat}]-cache[S])
        s=float(phi.sum())
        if abs(s)>1e-12: phi=phi*((fx-Ef)/s)
        phis.append(pd.Series(phi,index=features))
    return pd.concat(phis,axis=1).abs().mean(axis=1)


# ── Pretty print attributions ─────────────────────────────────────────────────
def print_attributions(label: str, global_abs: pd.Series, true_gamma: Dict[str,float],
                       features: List[str], width: int = 64):
    """Print a formatted attribution table with bar chart and metrics."""
    raw = np.array([global_abs.get(f,0) for f in features])
    sv  = raw/raw.sum() if raw.sum()>1e-12 else raw
    gt  = np.array([true_gamma[f] for f in features])

    # Metrics
    pr, _ = pearsonr(sv, gt)
    sr, _ = spearmanr(sv, gt)
    cos   = float(np.dot(sv, gt) / (np.linalg.norm(sv)*np.linalg.norm(gt) + 1e-12))
    rmse  = float(np.sqrt(np.mean((sv - gt)**2)))

    print(f"\n  {'─'*width}")
    print(f"  {label}")
    print(f"  {'─'*width}")
    print(f"  {'Feature':<22} {'Attr (norm)':>11}  {'True γ':>8}  {'Δ':>8}  Bar")
    print(f"  {'─'*width}")
    bar_max = sv.max() if sv.max() > 0 else 1
    for f, s, g in zip(features, sv, gt):
        bar  = '█' * int(round(s/bar_max*20))
        diff = s - g
        sign = '+' if diff >= 0 else ''
        print(f"  {f:<22} {s:>11.4f}  {g:>8.4f}  {sign}{diff:>7.4f}  {bar}")
    print(f"  {'─'*width}")
    print(f"  Pearson r = {pr:+.4f}  |  Spearman r = {sr:+.4f}  |  "
          f"Cosine sim = {cos:.4f}  |  RMSE = {rmse:.4f}")
    print(f"  {'─'*width}")


# ── Master comparison runner ──────────────────────────────────────────────────
def run_comparison(name: str, features: List[str], true_edges: set,
                   true_gamma: Dict[str,float], all_cols: List[str], target: str,
                   Xtr: pd.DataFrame, Xte: pd.DataFrame,
                   ytr: np.ndarray, cfg: dict) -> pd.DataFrame:

    print(f"\n{'='*80}")
    print(f"DATASET: {name.upper()}")
    print(f"{'='*80}")
    print(f"  True gamma:   { {f:round(v,4) for f,v in true_gamma.items()} }")
    print(f"  True edges:   {true_edges}")

    results = []

    # XGBoost trained once - shared across all methods
    xgb = XGBRegressor(n_estimators=160, max_depth=5, learning_rate=0.08,
                       subsample=0.9, colsample_bytree=0.9,
                       random_state=cfg['RANDOM_SEED'], verbosity=0,
                       n_jobs=1, tree_method='hist')
    xgb.fit(Xtr.values, ytr)

    true_dag = nx.DiGraph(); true_dag.add_nodes_from(features)
    for u,v in true_edges: true_dag.add_edge(u,v)
    star_dag = nx.DiGraph(); star_dag.add_nodes_from(features)

    # METHOD 1: Standard DAGMA Causal SHAP
    print("\n" + "─"*60)
    print("METHOD 1: Standard DAGMA → Causal SHAP")
    print("─"*60)
    t0 = time.time()
    W_d, gamma_d, dag_d = run_standard_dagma(Xtr, ytr, features, f'{name}_target')
    t_dagma = time.time()-t0

    gc_d = compare_graphs(dag_d, true_edges)
    gp_d = gamma_pearson(true_gamma, gamma_d, features)
    print(f"  Gamma recovered:    { {f:round(v,4) for f,v in gamma_d.items()} }")
    print(f"  Gamma Pearson:      {gp_d}")
    print(f"  Graph SHD={gc_d['SHD']} F1={gc_d['F1']}")
    print(f"  Edges recovered:    {gc_d['Recovered']}")
    print(f"  Missing edges:      {gc_d['Missing']}")
    print(f"  Spurious edges:     {gc_d['Extra']}")

    g_dagma = run_causal_shap_exact(xgb, gamma_d, dag_d, Xtr, Xte, cfg)
    am_d    = attribution_metrics(true_gamma, g_dagma, features)
    print_attributions("DAGMA Causal SHAP — per-feature attributions", g_dagma, true_gamma, features)
    results.append({'Method':'DAGMA Causal SHAP','Gamma_Pearson':gp_d,
                    'Graph_SHD':gc_d['SHD'],'Graph_F1':gc_d['F1'],**am_d,
                    'Fit_s':round(t_dagma,1),
                    **{f'attr_{f}':round(float(g_dagma.get(f,0)/g_dagma.sum()),4) for f in features}})

    # METHOD 2: DAGMA-DCE Causal SHAP
    print("\n" + "─"*60)
    print("METHOD 2: DAGMA-DCE (MLP) → Causal SHAP")
    print("─"*60)
    t0 = time.time()
    W_nl, gamma_nl, dag_nl = run_dce_mlp(Xtr, ytr, features, f'{name}_target', cfg)
    t_dce = time.time()-t0

    gc_nl = compare_graphs(dag_nl, true_edges)
    gp_nl = gamma_pearson(true_gamma, gamma_nl, features)
    print(f"  Gamma recovered:    { {f:round(v,4) for f,v in gamma_nl.items()} }")
    print(f"  Gamma Pearson:      {gp_nl}")
    print(f"  Graph SHD={gc_nl['SHD']} F1={gc_nl['F1']}")
    print(f"  Edges recovered:    {gc_nl['Recovered']}")
    print(f"  Missing edges:      {gc_nl['Missing']}")
    print(f"  Spurious edges:     {gc_nl['Extra']}")

    g_dce  = run_causal_shap_exact(xgb, gamma_nl, dag_nl, Xtr, Xte, cfg)
    am_dce = attribution_metrics(true_gamma, g_dce, features)
    print_attributions("DAGMA-DCE Causal SHAP — per-feature attributions", g_dce, true_gamma, features)
    results.append({'Method':'DAGMA-DCE Causal SHAP','Gamma_Pearson':gp_nl,
                    'Graph_SHD':gc_nl['SHD'],'Graph_F1':gc_nl['F1'],**am_dce,
                    'Fit_s':round(t_dce,1),
                    **{f'attr_{f}':round(float(g_dce.get(f,0)/g_dce.sum()),4) for f in features}})

    # METHOD 3: PC Algorithm -> Causal SHAP
    print("\n" + "─"*60)
    print("METHOD 3: PC Algorithm → Causal SHAP")
    print("─"*60)
    t0 = time.time()
    dag_pc, gamma_pc, connected = run_pc(Xtr, ytr, features, target, alpha=0.05)
    t_pc = time.time()-t0

    gc_pc = compare_graphs(dag_pc, true_edges)
    gp_pc = gamma_pearson(true_gamma, gamma_pc, features)
    print(f"  Gamma (uniform over connected): { {f:round(v,4) for f,v in gamma_pc.items()} }")
    print(f"  Gamma Pearson:      {gp_pc}")
    print(f"  PC edges found:     {gc_pc['Recovered']}")
    print(f"  Graph SHD={gc_pc['SHD']} F1={gc_pc['F1']}")
    print(f"  Features connected to target: {connected}")

    g_pc = run_causal_shap_exact(xgb, gamma_pc, dag_pc, Xtr, Xte, cfg)
    am_pc = attribution_metrics(true_gamma, g_pc, features)
    print_attributions("PC Causal SHAP — per-feature attributions", g_pc, true_gamma, features)
    results.append({'Method':'PC Causal SHAP','Gamma_Pearson':gp_pc,
                    'Graph_SHD':gc_pc['SHD'],'Graph_F1':gc_pc['F1'],**am_pc,
                    'Fit_s':round(t_pc,1),
                    **{f'attr_{f}':round(float(g_pc.get(f,0)/g_pc.sum()),4) for f in features}})

    # METHOD 4: Vanilla SHAP (no causal structure)
    print("\n" + "─"*60)
    print("METHOD 4: Vanilla SHAP (TreeExplainer, no causal structure)")
    print("─"*60)
    t0 = time.time()
    g_shap = run_vanilla_shap(xgb, Xtr, Xte, features, cfg['K'], cfg['RANDOM_SEED'])
    t_shap = time.time()-t0

    am_shap = attribution_metrics(true_gamma, g_shap, features)
    print(f"  (No graph or gamma — global |SHAP| values)")
    print_attributions("Vanilla SHAP — per-feature attributions", g_shap, true_gamma, features)
    results.append({'Method':'Vanilla SHAP','Gamma_Pearson':float('nan'),
                    'Graph_SHD':float('nan'),'Graph_F1':float('nan'),**am_shap,
                    'Fit_s':round(t_shap,1),
                    **{f'attr_{f}':round(float(g_shap.get(f,0)/g_shap.sum()),4) for f in features}})

    # CEILING: True gamma + True graph
    print("\n" + "─"*60)
    print("CEILING: True gamma + True graph → Causal SHAP")
    print("─"*60)
    g_ceil = run_causal_shap_exact(xgb, true_gamma, true_dag, Xtr, Xte, cfg)
    am_ceil = attribution_metrics(true_gamma, g_ceil, features)
    print_attributions("Ceiling — per-feature attributions", g_ceil, true_gamma, features)
    results.append({'Method':'Ceiling (true γ + true graph)','Gamma_Pearson':1.0,
                    'Graph_SHD':0,'Graph_F1':1.0,**am_ceil,'Fit_s':0.0,
                    **{f'attr_{f}':round(float(g_ceil.get(f,0)/g_ceil.sum()),4) for f in features}})

    # Summary table
    df = pd.DataFrame(results)
    disp = ['Method','Gamma_Pearson','Graph_SHD','Graph_F1',
            'Attr_Pearson','Attr_Spearman','Attr_Cosine','Attr_RMSE']

    print(f"\n{'='*110}")
    print(f"RESULTS SUMMARY — {name.upper()}")
    print(f"{'='*110}")
    print(df[disp].to_string(index=False))

    # Per-feature attribution summary
    attr_cols = ['Method'] + [f'attr_{f}' for f in features]
    print(f"\nNORMALISED ATTRIBUTIONS (should match true gamma above)")
    print(f"  True gamma: { {f:round(v,4) for f,v in true_gamma.items()} }")
    print(df[attr_cols].to_string(index=False))

    print("\n  Gamma_Pearson = how well recovered gamma aligns with analytical ground truth")
    print("  Attr_Pearson  = how well final attribution aligns with true gamma (end-to-end)")
    print("  Attr_RMSE     = per-feature attribution error (lower = better)")
    return df


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print("DAGMA Causal SHAP vs DAGMA-DCE Causal SHAP vs PC Causal SHAP vs Vanilla SHAP")
    print("CV Risk & Lung Cancer — Synthetic Known DGP")
    print("=" * 80)

    lc_gamma = compute_lc_gamma()
    cv_gamma = compute_cv_gamma()
    print(f"\nAnalytical gamma — Lung Cancer: {lc_gamma}")
    print(f"Analytical gamma — CV Risk:     {cv_gamma}")

    # ── Lung Cancer ───────────────────────────────────────────────────────────
    lc = pd.read_csv(CONFIG['LC_DATA_PATH']).drop(columns=['Unnamed: 0'], errors='ignore')
    Xtr_lc_r, Xte_lc_r, ytr_lc, _ = train_test_split(
        lc[LC_FEATURES], lc[LC_TARGET],
        test_size=CONFIG['TEST_SIZE'], random_state=CONFIG['RANDOM_SEED'], shuffle=True)
    sc_lc = StandardScaler()
    Xtr_lc = pd.DataFrame(sc_lc.fit_transform(Xtr_lc_r), columns=LC_FEATURES)
    Xte_lc = pd.DataFrame(sc_lc.transform(Xte_lc_r),     columns=LC_FEATURES)

    df_lc = run_comparison(
        'Lung_Cancer', LC_FEATURES, LC_TRUE_EDGES, lc_gamma,
        LC_FEATURES+[LC_TARGET], LC_TARGET,
        Xtr_lc, Xte_lc, ytr_lc.values, CONFIG)
    df_lc.to_csv('comparison_results_lung_cancer.csv', index=False)

    # ── CV Risk ───────────────────────────────────────────────────────────────
    cv = pd.read_csv(CONFIG['CV_DATA_PATH']).drop(columns=['Unnamed: 0'], errors='ignore')
    Xtr_cv_r, Xte_cv_r, ytr_cv, _ = train_test_split(
        cv[CV_FEATURES], cv[CV_TARGET],
        test_size=CONFIG['TEST_SIZE'], random_state=CONFIG['RANDOM_SEED'], shuffle=True)
    sc_cv = StandardScaler()
    Xtr_cv = pd.DataFrame(sc_cv.fit_transform(Xtr_cv_r), columns=CV_FEATURES)
    Xte_cv = pd.DataFrame(sc_cv.transform(Xte_cv_r),     columns=CV_FEATURES)

    df_cv = run_comparison(
        'CV_Risk', CV_FEATURES, CV_TRUE_EDGES, cv_gamma,
        CV_FEATURES+[CV_TARGET], CV_TARGET,
        Xtr_cv, Xte_cv, ytr_cv.values, CONFIG)
    df_cv.to_csv('comparison_results_cv_risk.csv', index=False)

    print("\nSaved: comparison_results_lung_cancer.csv, comparison_results_cv_risk.csv")


if __name__ == '__main__':
    main()

✓ DagmaDCE available
✓ dagma available
  Device: cuda

DAGMA Causal SHAP vs DAGMA-DCE Causal SHAP vs PC Causal SHAP vs Vanilla SHAP
CV Risk & Lung Cancer — Synthetic Known DGP

Analytical gamma — Lung Cancer: {'smoking': 0.7352941176470589, 'stress': 0.2647058823529412, 'drink_coffee': 0.0}
Analytical gamma — CV Risk:     {'diet_score': 0.0, 'sleep_duration': 0.0, 'family_history': 0.0, 'bmi': 0.697885}

DATASET: LUNG_CANCER
  True gamma:   {'smoking': 0.7353, 'stress': 0.2647, 'drink_coffee': 0.0}
  True edges:   {('smoking', 'drink_coffee'), ('stress', 'drink_coffee')}

────────────────────────────────────────────────────────────
METHOD 1: Standard DAGMA → Causal SHAP
────────────────────────────────────────────────────────────


  Gamma recovered:    {'smoking': 0.5672, 'stress': 0.3461, 'drink_coffee': 0.0866}
  Gamma Pearson:      0.9788
  Graph SHD=5 F1=0.0
  Edges recovered:    [('drink_coffee', 'smoking'), ('smoking', 'stress'), ('drink_coffee', 'stress')]
  Missing edges:      [('smoking', 'drink_coffee'), ('stress', 'drink_coffee')]
  Spurious edges:     [('drink_coffee', 'smoking'), ('smoking', 'stress'), ('drink_coffee', 'stress')]
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...

  ────────────────────────────────────────────────────────────────
  DAGMA Causal SHAP — per-feature attributions
  ────────────────────────────────────────────────────────────────
  Feature                Attr (norm)    True γ         Δ  Bar
  ────────────────────────────────────────────────────────────────
  smoking                     0.3882    0.7353  -0.3471  ████████████████████
  stress                      0.3245    0.2647  + 0.0598  ██

  Gamma recovered:    {'smoking': 0.567, 'stress': 0.3488, 'drink_coffee': 0.0842}
  Gamma Pearson:      0.9768
  Graph SHD=5 F1=0.0
  Edges recovered:    [('drink_coffee', 'smoking'), ('smoking', 'stress'), ('drink_coffee', 'stress')]
  Missing edges:      [('smoking', 'drink_coffee'), ('stress', 'drink_coffee')]
  Spurious edges:     [('drink_coffee', 'smoking'), ('smoking', 'stress'), ('drink_coffee', 'stress')]
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...

  ────────────────────────────────────────────────────────────────
  DAGMA-DCE Causal SHAP — per-feature attributions
  ────────────────────────────────────────────────────────────────
  Feature                Attr (norm)    True γ         Δ  Bar
  ────────────────────────────────────────────────────────────────
  smoking                     0.3895    0.7353  -0.3458  ████████████████████
  stress                      0.3272    0.2647  + 0.0624 

  Gamma recovered:    {'diet_score': 0.0, 'sleep_duration': 0.0, 'family_history': 0.0, 'bmi': 1.0}
  Gamma Pearson:      1.0
  Graph SHD=3 F1=0.4
  Edges recovered:    [('bmi', 'sleep_duration'), ('diet_score', 'sleep_duration'), ('diet_score', 'bmi')]
  Missing edges:      [('sleep_duration', 'bmi')]
  Spurious edges:     [('bmi', 'sleep_duration'), ('diet_score', 'sleep_duration')]
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...

  ────────────────────────────────────────────────────────────────
  DAGMA Causal SHAP — per-feature attributions
  ────────────────────────────────────────────────────────────────
  Feature                Attr (norm)    True γ         Δ  Bar
  ────────────────────────────────────────────────────────────────
  diet_score                  0.0000    0.0000  + 0.0000  
  sleep_duration              0.0000    0.0000  + 0.0000  
  family_history              0.0000    0.0000  + 0.

  Gamma recovered:    {'diet_score': 0.0171, 'sleep_duration': 0.0226, 'family_history': 0.0294, 'bmi': 0.9309}
  Gamma Pearson:      0.9999
  Graph SHD=3 F1=0.4
  Edges recovered:    [('bmi', 'sleep_duration'), ('diet_score', 'sleep_duration'), ('diet_score', 'bmi')]
  Missing edges:      [('sleep_duration', 'bmi')]
  Spurious edges:     [('bmi', 'sleep_duration'), ('diet_score', 'sleep_duration')]
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...

  ────────────────────────────────────────────────────────────────
  DAGMA-DCE Causal SHAP — per-feature attributions
  ────────────────────────────────────────────────────────────────
  Feature                Attr (norm)    True γ         Δ  Bar
  ────────────────────────────────────────────────────────────────
  diet_score                  0.0192    0.0000  + 0.0192  
  sleep_duration              0.0104    0.0000  + 0.0104  
  family_history              0.0

In [2]:
import sys
sys.path.insert(0, 'DAGMA-DCE/src')


import time
import warnings
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from scipy.stats import pearsonr, spearmanr
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
import networkx as nx
from xgboost import XGBClassifier

from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz

warnings.filterwarnings("ignore")

try:
    from DagmaDCE import nonlinear_dce
    HAS_DCE = True
    print("ok DagmaDCE available")
except ImportError:
    HAS_DCE = False; print("x DagmaDCE not found")

try:
    from dagma.linear import DagmaLinear
    HAS_DAGMA = True; print("ok dagma available")
except ImportError:
    HAS_DAGMA = False; print("x dagma not found")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device: {device}\n")

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH      = '../../../../datasets/wine_preprocessed.npz'
RANDOM_SEED    = 42
N_CLASSES      = 3
MC_ITERATIONS  = 500
MC_K_INSTANCES = 50
PERTURB_SIGMA  = 0.05
PERTURB_RUNS   = 5
ROB_INSTANCES  = 20
CAUSAL_M       = 64
CAUSAL_K       = 30
CAUSAL_PERMS   = 200
DCE_CFG = dict(lambda1=1e-2, lambda2=5e-3, lr=2e-4,
               mu_factor=0.1, mu_init=0.1, T=6,
               warm_iter=10_000, max_iter=12_000, hidden_mult=4)

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# ── Load data ─────────────────────────────────────────────────────────────────
raw     = np.load(DATA_PATH, allow_pickle=True)
X_train = pd.DataFrame(raw['X_train'], columns=raw['feature_names'])
X_test  = pd.DataFrame(raw['X_test'],  columns=raw['feature_names'])
y_train = raw['y_train'] - 1
y_test  = raw['y_test']  - 1
FEATURES = list(X_train.columns)
N_FEAT   = len(FEATURES)
print(f"Features ({N_FEAT}): {FEATURES}")
print(f"Train: {X_train.shape}  Test: {X_test.shape}  Classes: {np.unique(y_train)}\n")

# ── XGBoost classifier ────────────────────────────────────────────────────────
xgb = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                    subsample=0.8, colsample_bytree=0.8,
                    random_state=RANDOM_SEED, verbosity=0, n_jobs=1,
                    tree_method='hist', objective='multi:softprob',
                    num_class=N_CLASSES, eval_metric='mlogloss')
xgb.fit(X_train.values, y_train)
print(f"XGBoost  train acc={np.mean(xgb.predict(X_train.values)==y_train):.4f}"
      f"  test acc={np.mean(xgb.predict(X_test.values)==y_test):.4f}\n")

# ── Permutation importance oracle ─────────────────────────────────────────────
print("Computing permutation importance...")
perm_res = permutation_importance(xgb, X_test.values, y_test,
                                   n_repeats=20, random_state=RANDOM_SEED,
                                   scoring='accuracy')
PERM_IMP = pd.Series(np.abs(perm_res.importances_mean), index=FEATURES)
PERM_IMP = PERM_IMP / PERM_IMP.sum()
print(f"  Top 5: {dict(PERM_IMP.nlargest(5).round(4))}\n")

# ── Utilities ─────────────────────────────────────────────────────────────────
def normalise(g: Dict[str,float], features: List[str]) -> Dict[str,float]:
    g   = {f: max(v,0.0) for f,v in g.items()}
    tot = sum(g.values())
    return {f: v/tot for f,v in g.items()} if tot>1e-12 \
        else {f: 1.0/len(features) for f in features}

def run_standard_dagma(X: pd.DataFrame) -> Tuple[np.ndarray, Dict[str,float], nx.DiGraph]:
    """
    Include mean predicted class probability (per class) as soft targets
    in the DAGMA joint matrix so gamma = direct effect on model output,
    not just outgoing causal centrality.
    """
    probs        = xgb.predict_proba(X[FEATURES].values)
    all_cols     = FEATURES + [f'prob_cls{c}' for c in range(N_CLASSES)]
    Xfull        = X[FEATURES].copy()
    for c in range(N_CLASSES):
        Xfull[f'prob_cls{c}'] = probs[:, c]

    W = DagmaLinear(loss_type='l2').fit(Xfull[all_cols].values.astype(float), lambda1=0.02)

    tgt_indices = [all_cols.index(f'prob_cls{c}') for c in range(N_CLASSES)]
    effects = np.mean([np.abs(W[:N_FEAT, t]) for t in tgt_indices], axis=0)
    gamma   = normalise({f: float(effects[i]) for i,f in enumerate(FEATURES)}, FEATURES)

    G = nx.DiGraph(); G.add_nodes_from(FEATURES)
    for i,fi in enumerate(FEATURES):
        for j,fj in enumerate(FEATURES):
            if i!=j and abs(W[i,j])>0.05:
                G.add_edge(fi, fj, weight=float(W[i,j]))
    return W, gamma, G


def run_dce_mlp(X: pd.DataFrame, cfg: dict) -> Tuple[np.ndarray, Dict[str,float], nx.DiGraph]:
    """
    Include mean predicted class probabilities as soft target columns.
    FIX 1: W_est.detach().cpu().numpy() before indexing.
    """
    probs    = xgb.predict_proba(X[FEATURES].values)
    all_cols = FEATURES + [f'prob_cls{c}' for c in range(N_CLASSES)]
    Xfull    = X[FEATURES].copy()
    for c in range(N_CLASSES):
        Xfull[f'prob_cls{c}'] = probs[:, c]

    n_vars   = len(all_cols)
    data     = torch.tensor(Xfull[all_cols].values.astype(np.float64), dtype=torch.float64)
    hidden   = max(10, cfg['hidden_mult'] * n_vars)
    eq_model = nonlinear_dce.DagmaMLP_DCE(dims=[n_vars, hidden, 1], bias=True)
    model    = nonlinear_dce.DagmaDCE(eq_model, use_mse_loss=True)
    print(f"  Running DAGMA-DCE MLP on {n_vars} variables (hidden={hidden})...")

    W_est_tensor = model.fit(data,
        lambda1=cfg['lambda1'], lambda2=cfg['lambda2'], lr=cfg['lr'],
        mu_factor=cfg['mu_factor'], mu_init=cfg['mu_init'],
        T=cfg['T'], warm_iter=cfg['warm_iter'], max_iter=cfg['max_iter'])

    if isinstance(W_est_tensor, torch.Tensor):
        W_est = W_est_tensor.detach().cpu().numpy()
    else:
        W_est = np.array(W_est_tensor)

    tgt_indices = [all_cols.index(f'prob_cls{c}') for c in range(N_CLASSES)]
    effects = np.mean([np.abs(W_est[:N_FEAT, t]) for t in tgt_indices], axis=0)
    gamma   = normalise({f: float(effects[i]) for i,f in enumerate(FEATURES)}, FEATURES)

    G = nx.DiGraph(); G.add_nodes_from(FEATURES)
    for i,fi in enumerate(FEATURES):
        for j,fj in enumerate(FEATURES):
            if i!=j and abs(W_est[i,j])>0.05:
                G.add_edge(fi, fj, weight=float(W_est[i,j]))
    return W_est, gamma, G


# ── PC Algorithm ──────────────────────────────────────────────────────────────
def run_pc_wine(X: pd.DataFrame, alpha: float = 0.05
                ) -> Tuple[nx.DiGraph, Dict[str,float], List[str]]:
    """Two-pass PC: inter-feature graph + target connections via soft probs."""
    # Pass 1: feature skeleton
    cg_feat = pc(X[FEATURES].values.astype(float), alpha=alpha,
                 indep_test=fisherz, node_names=FEATURES,
                 verbose=False, show_progress=False)
    adj_f   = cg_feat.G.graph
    nodes_f = [n.get_name() for n in cg_feat.G.get_nodes()]
    G = nx.DiGraph(); G.add_nodes_from(FEATURES)
    for i in range(N_FEAT):
        for j in range(N_FEAT):
            if i==j: continue
            if adj_f[i,j]==1 and adj_f[j,i]==-1:
                G.add_edge(nodes_f[i], nodes_f[j])
            elif adj_f[i,j]==-1 and adj_f[j,i]==-1 and i<j:
                G.add_edge(nodes_f[i], nodes_f[j])
                G.add_edge(nodes_f[j], nodes_f[i])

    # Pass 2: features + predicted probability (class 0 sufficient as proxy)
    probs    = xgb.predict_proba(X[FEATURES].values)
    all_cols = FEATURES + ['prob_cls0']
    Xfull    = X[FEATURES].copy(); Xfull['prob_cls0'] = probs[:,0]
    cg_all   = pc(Xfull[all_cols].values.astype(float), alpha=alpha,
                  indep_test=fisherz, node_names=all_cols,
                  verbose=False, show_progress=False)
    adj_all   = cg_all.G.graph
    all_nodes = [n.get_name() for n in cg_all.G.get_nodes()]
    tgt_idx   = all_nodes.index('prob_cls0')
    connected = [f for f in FEATURES
                 if adj_all[all_nodes.index(f), tgt_idx]!=0
                 or adj_all[tgt_idx, all_nodes.index(f)]!=0]

    gamma = ({f: (1.0/len(connected) if f in connected else 0.0) for f in FEATURES}
             if connected else {f: 1.0/N_FEAT for f in FEATURES})
    return G, gamma, connected


# ── Causal SHAP (classification, permutation engine) ─────────────────────────
class LinearMechanisms:
    def __init__(self, dag, features):
        self.dag=dag; self.features=features; self.mech={}
        self.topo=list(nx.topological_sort(dag))

    def fit(self, X):
        for var in self.features:
            parents=list(self.dag.predecessors(var))
            if not parents:
                self.mech[var]={'parents':[],'intercept':float(X[var].mean()),
                                'coef':None,'sigma':float(X[var].std(ddof=0))}
            else:
                lr=LinearRegression().fit(X[parents].values,X[var].values)
                resid=X[var].values-lr.predict(X[parents].values)
                self.mech[var]={'parents':parents,'intercept':float(lr.intercept_),
                                'coef':lr.coef_.astype(float),
                                'sigma':float(np.std(resid,ddof=0))}
        return self

    def sample_batch(self, S, x, M, rng):
        X_out=np.empty((M,len(self.features))); fi={f:i for i,f in enumerate(self.features)}
        sim={}
        for var in self.topo:
            vi=fi[var]
            if var in S:
                val=float(x[var]); X_out[:,vi]=val; sim[var]=np.full(M,val); continue
            m=self.mech[var]
            mu=np.full(M,m['intercept']) if not m['parents'] else \
               m['intercept']+np.stack([sim[p] if p in sim else np.full(M,float(x[p]))
                                        for p in m['parents']],axis=1)@m['coef']
            X_out[:,vi]=mu if m['sigma']<1e-8 else rng.normal(mu,m['sigma'],size=M)
            sim[var]=X_out[:,vi]
        return X_out


def run_causal_shap_clf(gamma: Dict[str,float], dag: nx.DiGraph,
                        Xtr: pd.DataFrame, Xte: pd.DataFrame,
                        K: int=CAUSAL_K, M: int=CAUSAL_M,
                        n_perms: int=CAUSAL_PERMS) -> pd.Series:
    rng  = np.random.default_rng(RANDOM_SEED)
    sem  = LinearMechanisms(dag, FEATURES).fit(Xtr)
    K    = min(K, len(Xte))
    idx  = rng.choice(len(Xte), size=K, replace=False)

    def vc(S, x, pred_cls):
        batch = sem.sample_batch(S, x, M, rng)
        return float(np.mean(xgb.predict_proba(batch)[:, pred_cls]))

    Ef_dict = {}
    for cls in range(N_CLASSES):
        # sample M points uniformly from training set for baseline
        bg_idx   = rng.choice(len(Xtr), size=M, replace=True)
        bg_batch = Xtr.values[bg_idx]
        Ef_dict[cls] = float(np.mean(xgb.predict_proba(bg_batch)[:, cls]))

    phis = []
    for cnt, i in enumerate(idx):
        if (cnt+1)%10==0: print(f"      instance {cnt+1}/{K} ...",flush=True)
        x        = Xte.iloc[i]
        pred_cls = int(xgb.predict(x.values.reshape(1,-1))[0])
        fx       = float(xgb.predict_proba(x.values.reshape(1,-1))[0, pred_cls])
        Ef       = Ef_dict[pred_cls]
        phi      = np.zeros(N_FEAT)
        for _ in range(n_perms):
            perm   = rng.permutation(FEATURES); S=set()
            v_prev = vc(S, x, pred_cls)
            for feat in perm:
                Sn  = S|{feat}; vc_ = vc(Sn, x, pred_cls)
                phi[FEATURES.index(feat)] += gamma[feat]*(vc_-v_prev)
                v_prev, S = vc_, Sn
        phi /= n_perms
        s = float(phi.sum())
        if abs(s)>1e-12: phi = phi*((fx-Ef)/s)
        phis.append(np.abs(phi))
    return pd.Series(np.mean(phis,axis=0), index=FEATURES)


# ── Monte Carlo Shapley (gt_concepts style) ───────────────────────────────────
def _mc_utility(coalition: List[int], x: np.ndarray,
                X_bg: np.ndarray, pred_cls: int) -> float:
    if not coalition:
        return float(np.mean(xgb.predict_proba(X_bg)[:,pred_cls]))
    X_rep = X_bg.copy()
    for fi in coalition: X_rep[:,fi] = x[fi]
    return float(np.mean(xgb.predict_proba(X_rep)[:,pred_cls]))

def montecarlo_shapley_instance(x, X_bg, pred_cls, n_iter=MC_ITERATIONS):
    sv = np.zeros(N_FEAT)
    for _ in range(n_iter):
        perm    = np.random.permutation(N_FEAT); current=[]; prev_v=_mc_utility([],x,X_bg,pred_cls)
        for p in perm:
            new_v   = _mc_utility(current+[int(p)], x, X_bg, pred_cls)
            sv[p]  += new_v-prev_v; current.append(int(p)); prev_v=new_v
    return sv/n_iter

def run_mc_shapley(Xtr: pd.DataFrame, Xte: pd.DataFrame, K=MC_K_INSTANCES) -> pd.Series:
    rng=np.random.default_rng(RANDOM_SEED); X_bg=Xtr.values
    idx=rng.choice(len(Xte),size=min(K,len(Xte)),replace=False); phis=[]
    for cnt,i in enumerate(idx):
        if (cnt+1)%10==0: print(f"      MC Shapley {cnt+1}/{len(idx)} ...",flush=True)
        x=Xte.values[i]; pred_cls=int(xgb.predict(x.reshape(1,-1))[0])
        phis.append(np.abs(montecarlo_shapley_instance(x,X_bg,pred_cls)))
    return pd.Series(np.mean(phis,axis=0), index=FEATURES)


# ── Metrics ───────────────────────────────────────────────────────────────────
def faithfulness_spearman(global_abs: pd.Series, Xte: pd.DataFrame,
                           n_instances: int=50) -> float:
    """
    FIX 2: per-instance local attribution order.
    For each instance, compute local phi (single-instance Causal SHAP or
    use global proportions rescaled — here we use global order as rank proxy
    BUT measure drops on that instance's prediction.
    True fix: rank by per-instance phi. We keep global order for speed
    but compute per-instance drop correctly.
    """
    rng   = np.random.default_rng(RANDOM_SEED)
    means = X_train.mean().values
    idx   = rng.choice(len(Xte), size=min(n_instances,len(Xte)), replace=False)
    spears = []
    for i in idx:
        x        = Xte.values[i].copy()
        pred_cls = int(xgb.predict(x.reshape(1,-1))[0])
        base_p   = float(xgb.predict_proba(x.reshape(1,-1))[0,pred_cls])

        # Approximate local phi via a fast single-permutation estimate
        local_phi = np.zeros(N_FEAT)
        for _ in range(20):   # 20 quick permutations for local rank estimate
            perm  = np.random.permutation(N_FEAT)
            x_mod = means.copy(); prev_p = float(np.mean(xgb.predict_proba(
                np.tile(x_mod,(10,1)))[:,pred_cls]))
            for fi in perm:
                x_mod[fi] = x[fi]
                new_p = float(np.mean(xgb.predict_proba(np.tile(x_mod,(10,1)))[:,pred_cls]))
                local_phi[fi] += abs(new_p-prev_p); prev_p=new_p
        local_phi /= 20
        order = [FEATURES[fi] for fi in np.argsort(-local_phi)]

        # Measure drop using this instance's local order
        drops=[]; x_mod=x.copy()
        for feat in order:
            fi = FEATURES.index(feat); x_mod[fi]=means[fi]
            new_p=float(xgb.predict_proba(x_mod.reshape(1,-1))[0,pred_cls])
            drops.append(base_p-new_p)

        ranks=list(range(1,N_FEAT+1))
        sr,_ = spearmanr(ranks,drops)
        if not np.isnan(sr): spears.append(sr)
    return round(float(np.mean(spears)),4)

def predictive_alignment(global_abs: pd.Series) -> float:
    sv=global_abs.values/global_abs.sum() if global_abs.sum()>1e-12 else global_abs.values
    pr,_=pearsonr(sv, PERM_IMP[FEATURES].values); return round(float(pr),4)

def attribution_cosine(global_abs: pd.Series) -> float:
    sv=global_abs.values; pi=PERM_IMP[FEATURES].values
    return round(float(np.dot(sv,pi)/(np.linalg.norm(sv)*np.linalg.norm(pi)+1e-12)),4)

def robustness_metrics(method_fn, n_instances: int=ROB_INSTANCES) -> Tuple[float,float,float]:
    rng=np.random.default_rng(RANDOM_SEED)
    idx=rng.choice(len(X_test),size=min(n_instances,len(X_test)),replace=False)
    cos_sims=[]; rank_cors=[]; l2_dists=[]
    for i in idx:
        x_orig   = X_test.values[i:i+1]
        phi_orig = method_fn(pd.DataFrame(x_orig,columns=FEATURES)).values
        rc=[]; rr=[]; rl=[]
        for _ in range(PERTURB_RUNS):
            x_pert   = x_orig+rng.normal(0,PERTURB_SIGMA,x_orig.shape)
            phi_pert = method_fn(pd.DataFrame(x_pert,columns=FEATURES)).values
            n1,n2    = np.linalg.norm(phi_orig),np.linalg.norm(phi_pert)
            cos      = float(np.dot(phi_orig,phi_pert)/(n1*n2+1e-12))
            sr,_     = spearmanr(phi_orig,phi_pert)
            l2       = float(np.linalg.norm(phi_orig-phi_pert))
            rc.append(cos); rr.append(sr if not np.isnan(sr) else 0.0); rl.append(l2)
        cos_sims.append(np.mean(rc)); rank_cors.append(np.mean(rr)); l2_dists.append(np.mean(rl))
    return (round(float(np.mean(cos_sims)),4),
            round(float(np.mean(rank_cors)),4),
            round(float(np.mean(l2_dists)),4))


# ── Pretty print ──────────────────────────────────────────────────────────────
def print_attributions(label: str, global_abs: pd.Series, width: int=68):
    sv   = global_abs/global_abs.sum() if global_abs.sum()>1e-12 else global_abs
    pi   = PERM_IMP[FEATURES]
    pr,_ = pearsonr(sv.values,pi.values); cos=attribution_cosine(global_abs)
    print(f"\n  {'─'*width}\n  {label}\n  {'─'*width}")
    print(f"  {'Feature':<32} {'Attr(norm)':>10}  {'PermImp':>8}  {'Δ':>8}  Bar")
    print(f"  {'─'*width}")
    bar_max=sv.max() if sv.max()>0 else 1
    for f in FEATURES:
        s=sv[f]; p=pi[f]; bar='█'*int(round(s/bar_max*18))
        diff=s-p; sign='+' if diff>=0 else ''
        print(f"  {f:<32} {s:>10.4f}  {p:>8.4f}  {sign}{diff:>7.4f}  {bar}")
    print(f"  {'─'*width}")
    print(f"  Pearson (vs perm imp) = {pr:+.4f}  |  Cosine (vs perm imp) = {cos:.4f}")
    print(f"  {'─'*width}")


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print("="*80)
    print("Wine Dataset — Classification Causal SHAP (4 fixes applied)")
    print("="*80)
    results=[]

    # Method 1: dagma
    print("\n"+"─"*60+"\nMETHOD 1: Standard DAGMA → Causal SHAP\n"+"─"*60)
    t0=time.time()
    W_d,gamma_d,dag_d=run_standard_dagma(X_train)
    print(f"  Fit: {time.time()-t0:.1f}s | Edges: {len(dag_d.edges())} | "
          f"Gamma top3: {dict(sorted(gamma_d.items(),key=lambda x:-x[1])[:3])}")
    g_dagma=run_causal_shap_clf(gamma_d,dag_d,X_train,X_test)
    print_attributions("DAGMA Causal SHAP",g_dagma)
    faith_d=faithfulness_spearman(g_dagma,X_test)
    align_d=predictive_alignment(g_dagma); cos_d=attribution_cosine(g_dagma)
    def dagma_fn(Xdf): return run_causal_shap_clf(gamma_d,dag_d,X_train,Xdf,K=1,n_perms=50)
    rc_d,rr_d,rl_d=robustness_metrics(dagma_fn)
    print(f"  Faith={faith_d:.4f} | Align={align_d:.4f} | Cos={cos_d:.4f} | "
          f"RobCos={rc_d:.4f} | RankRob={rr_d:.4f} | L2={rl_d:.4f}")
    results.append({'Method':'DAGMA Causal SHAP','Faithfulness':faith_d,'Pred_Align':align_d,
                    'Attr_Cosine':cos_d,'Rob_Cosine':rc_d,'Rank_Rob':rr_d,'L2':rl_d,
                    **{f'attr_{f}':round(float(g_dagma[f]/g_dagma.sum()),4) for f in FEATURES}})

    # Method 2: dagma-dce
    print("\n"+"─"*60+"\nMETHOD 2: DAGMA-DCE (MLP) → Causal SHAP\n"+"─"*60)
    t0=time.time()
    W_nl,gamma_nl,dag_nl=run_dce_mlp(X_train,DCE_CFG)
    print(f"  Fit: {time.time()-t0:.1f}s | Edges: {len(dag_nl.edges())} | "
          f"Gamma top3: {dict(sorted(gamma_nl.items(),key=lambda x:-x[1])[:3])}")
    g_dce=run_causal_shap_clf(gamma_nl,dag_nl,X_train,X_test)
    print_attributions("DAGMA-DCE Causal SHAP",g_dce)
    faith_nl=faithfulness_spearman(g_dce,X_test)
    align_nl=predictive_alignment(g_dce); cos_nl=attribution_cosine(g_dce)
    def dce_fn(Xdf): return run_causal_shap_clf(gamma_nl,dag_nl,X_train,Xdf,K=1,n_perms=50)
    rc_nl,rr_nl,rl_nl=robustness_metrics(dce_fn)
    print(f"  Faith={faith_nl:.4f} | Align={align_nl:.4f} | Cos={cos_nl:.4f} | "
          f"RobCos={rc_nl:.4f} | RankRob={rr_nl:.4f} | L2={rl_nl:.4f}")
    results.append({'Method':'DAGMA-DCE Causal SHAP','Faithfulness':faith_nl,'Pred_Align':align_nl,
                    'Attr_Cosine':cos_nl,'Rob_Cosine':rc_nl,'Rank_Rob':rr_nl,'L2':rl_nl,
                    **{f'attr_{f}':round(float(g_dce[f]/g_dce.sum()),4) for f in FEATURES}})

    # Method 3: pc
    print("\n"+"─"*60+"\nMETHOD 3: PC Algorithm → Causal SHAP\n"+"─"*60)
    t0=time.time()
    dag_pc,gamma_pc,connected_pc=run_pc_wine(X_train)
    print(f"  Fit: {time.time()-t0:.1f}s | Edges: {len(dag_pc.edges())} | "
          f"Connected to target: {connected_pc}")
    g_pc=run_causal_shap_clf(gamma_pc,dag_pc,X_train,X_test)
    print_attributions("PC Causal SHAP",g_pc)
    faith_pc=faithfulness_spearman(g_pc,X_test)
    align_pc=predictive_alignment(g_pc); cos_pc=attribution_cosine(g_pc)
    def pc_fn(Xdf): return run_causal_shap_clf(gamma_pc,dag_pc,X_train,Xdf,K=1,n_perms=50)
    rc_pc,rr_pc,rl_pc=robustness_metrics(pc_fn)
    print(f"  Faith={faith_pc:.4f} | Align={align_pc:.4f} | Cos={cos_pc:.4f} | "
          f"RobCos={rc_pc:.4f} | RankRob={rr_pc:.4f} | L2={rl_pc:.4f}")
    results.append({'Method':'PC Causal SHAP','Faithfulness':faith_pc,'Pred_Align':align_pc,
                    'Attr_Cosine':cos_pc,'Rob_Cosine':rc_pc,'Rank_Rob':rr_pc,'L2':rl_pc,
                    **{f'attr_{f}':round(float(g_pc[f]/g_pc.sum()),4) for f in FEATURES}})

    # METHOD 4: MC Shapley
    print("\n"+"─"*60+"\nMETHOD 4: Monte Carlo Shapley (gt_concepts style)\n"+"─"*60)
    t0=time.time(); g_mc=run_mc_shapley(X_train,X_test,K=MC_K_INSTANCES)
    print(f"  MC Shapley time: {time.time()-t0:.1f}s")
    print_attributions("MC Shapley",g_mc)
    faith_mc=faithfulness_spearman(g_mc,X_test)
    align_mc=predictive_alignment(g_mc); cos_mc=attribution_cosine(g_mc)
    def mc_fn(Xdf): return run_mc_shapley(X_train,Xdf,K=1)
    rc_mc,rr_mc,rl_mc=robustness_metrics(mc_fn)
    print(f"  Faith={faith_mc:.4f} | Align={align_mc:.4f} | Cos={cos_mc:.4f} | "
          f"RobCos={rc_mc:.4f} | RankRob={rr_mc:.4f} | L2={rl_mc:.4f}")
    results.append({'Method':'MC Shapley','Faithfulness':faith_mc,'Pred_Align':align_mc,
                    'Attr_Cosine':cos_mc,'Rob_Cosine':rc_mc,'Rank_Rob':rr_mc,'L2':rl_mc,
                    **{f'attr_{f}':round(float(g_mc[f]/g_mc.sum()),4) for f in FEATURES}})

    # Summary
    df=pd.DataFrame(results)
    disp=['Method','Faithfulness','Pred_Align','Attr_Cosine','Rob_Cosine','Rank_Rob','L2']
    print(f"\n{'='*100}\nRESULTS SUMMARY — WINE CLASSIFICATION\n{'='*100}")
    print(df[disp].to_string(index=False))
    print(f"\nPERM IMP: {dict(PERM_IMP.round(4))}")
    print("\nNORMALISED ATTRIBUTIONS:")
    print(df[['Method']+[f'attr_{f}' for f in FEATURES]].to_string(index=False))
    print("\n  Faithfulness (Spearman ↑): per-instance local attr rank vs prediction drop")
    print("  Pred_Align   (Pearson  ↑): attribution vs permutation importance")
    print("  Attr_Cosine  (Cosine   ↑): cosine sim vs permutation importance")
    print("  Rob_Cosine   (Cosine   ↑): cosine sim under Gaussian perturbation")
    print("  Rank_Rob     (Spearman ↑): rank Spearman under perturbation")
    print("  L2           (L2       ↓): L2 drift under perturbation")
    df.to_csv('wine_clf_comparison_results.csv',index=False)
    print("\nSaved: wine_clf_comparison_results.csv")

if __name__ == '__main__':
    main()

✓ DagmaDCE available
✓ dagma available
  Device: cuda

Features (13): ['Alcohol', 'Malic acid', 'Ash', 'Alcalinity of ash', 'Magnesium', 'Total phenols', 'Flavanoids', 'Nonflavanoid phenols', 'Proanthocyanins', 'Color intensity', 'Hue', 'OD280/OD315 of diluted wines', 'Proline']
Train: (141, 13)  Test: (36, 13)  Classes: [0 1 2]

XGBoost  train acc=1.0000  test acc=1.0000

Computing permutation importance...
  Top 5: {'Flavanoids': np.float64(0.3031), 'Proline': np.float64(0.2334), 'Color intensity': np.float64(0.1777), 'Alcohol': np.float64(0.1324), 'Hue': np.float64(0.0488)}

Wine Dataset — Classification Causal SHAP (4 fixes applied)

────────────────────────────────────────────────────────────
METHOD 1: Standard DAGMA → Causal SHAP
────────────────────────────────────────────────────────────


  Fit: 2.0s | Edges: 11 | Gamma top3: {'Alcohol': 0.07692307692307693, 'Malic acid': 0.07692307692307693, 'Ash': 0.07692307692307693}
      instance 10/30 ...
      instance 20/30 ...
      instance 30/30 ...

  ────────────────────────────────────────────────────────────────────
  DAGMA Causal SHAP
  ────────────────────────────────────────────────────────────────────
  Feature                          Attr(norm)   PermImp         Δ  Bar
  ────────────────────────────────────────────────────────────────────
  Alcohol                              0.1949    0.1324  + 0.0625  ████████████████
  Malic acid                           0.0480    0.0383  + 0.0096  ████
  Ash                                  0.0127    0.0174  -0.0047  █
  Alcalinity of ash                    0.0040    0.0000  + 0.0040  
  Magnesium                            0.0217    0.0139  + 0.0078  ██
  Total phenols                        0.0051    0.0000  + 0.0051  
  Flavanoids                           0.2198    0.3031 

  Fit: 1628.1s | Edges: 66 | Gamma top3: {'OD280/OD315 of diluted wines': 0.37323124275507397, 'Flavanoids': 0.13618474354320761, 'Color intensity': 0.1319360379364927}
      instance 10/30 ...
      instance 20/30 ...
      instance 30/30 ...

  ────────────────────────────────────────────────────────────────────
  DAGMA-DCE Causal SHAP
  ────────────────────────────────────────────────────────────────────
  Feature                          Attr(norm)   PermImp         Δ  Bar
  ────────────────────────────────────────────────────────────────────
  Alcohol                              0.0844    0.1324  -0.0480  █████
  Malic acid                           0.0246    0.0383  -0.0138  █
  Ash                                  0.0035    0.0174  -0.0139  
  Alcalinity of ash                    0.0008    0.0000  + 0.0008  
  Magnesium                            0.0013    0.0139  -0.0126  
  Total phenols                        0.0021    0.0000  + 0.0021  
  Flavanoids                         

In [3]:
import sys
sys.path.insert(0, 'DAGMA-DCE/src')


import time
import warnings
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from scipy.stats import pearsonr, spearmanr
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
import networkx as nx
from xgboost import XGBClassifier

from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz

warnings.filterwarnings("ignore")

try:
    from DagmaDCE import nonlinear_dce
    HAS_DCE = True
    print("ok DagmaDCE available")
except ImportError:
    HAS_DCE = False; print("x DagmaDCE not found")

try:
    from dagma.linear import DagmaLinear
    HAS_DAGMA = True; print("ok dagma available")
except ImportError:
    HAS_DAGMA = False; print("x dagma not found")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device: {device}\n")

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH      = '../../../../datasets/wine_preprocessed.npz'
RANDOM_SEED    = 42
N_CLASSES      = 3
MC_ITERATIONS  = 500
MC_K_INSTANCES = 50
PERTURB_SIGMA  = 0.05
PERTURB_RUNS   = 5
ROB_INSTANCES  = 20
CAUSAL_M       = 64
CAUSAL_K       = 30
CAUSAL_PERMS   = 200
DCE_CFG = dict(lambda1=1e-2, lambda2=5e-3, lr=2e-4,
               mu_factor=0.1, mu_init=0.1, T=6,
               warm_iter=10_000, max_iter=12_000, hidden_mult=4)

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# ── Load data ─────────────────────────────────────────────────────────────────
raw     = np.load(DATA_PATH, allow_pickle=True)
X_train = pd.DataFrame(raw['X_train'], columns=raw['feature_names'])
X_test  = pd.DataFrame(raw['X_test'],  columns=raw['feature_names'])
y_train = raw['y_train'] - 1
y_test  = raw['y_test']  - 1
FEATURES = list(X_train.columns)
N_FEAT   = len(FEATURES)
print(f"Features ({N_FEAT}): {FEATURES}")
print(f"Train: {X_train.shape}  Test: {X_test.shape}  Classes: {np.unique(y_train)}\n")

# ── XGBoost classifier ────────────────────────────────────────────────────────
xgb = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                    subsample=0.8, colsample_bytree=0.8,
                    random_state=RANDOM_SEED, verbosity=0, n_jobs=1,
                    tree_method='hist', objective='multi:softprob',
                    num_class=N_CLASSES, eval_metric='mlogloss')
xgb.fit(X_train.values, y_train)
print(f"XGBoost  train acc={np.mean(xgb.predict(X_train.values)==y_train):.4f}"
      f"  test acc={np.mean(xgb.predict(X_test.values)==y_test):.4f}\n")

# ── Permutation importance oracle ─────────────────────────────────────────────
print("Computing permutation importance...")
perm_res = permutation_importance(xgb, X_test.values, y_test,
                                   n_repeats=20, random_state=RANDOM_SEED,
                                   scoring='accuracy')
PERM_IMP = pd.Series(np.abs(perm_res.importances_mean), index=FEATURES)
PERM_IMP = PERM_IMP / PERM_IMP.sum()
print(f"  Top 5: {dict(PERM_IMP.nlargest(5).round(4))}\n")

# ── Utilities ─────────────────────────────────────────────────────────────────
def normalise(g: Dict[str,float], features: List[str]) -> Dict[str,float]:
    g   = {f: max(v,0.0) for f,v in g.items()}
    tot = sum(g.values())
    return {f: v/tot for f,v in g.items()} if tot>1e-12 \
        else {f: 1.0/len(features) for f in features}

def make_dag(G):
    """Convert any DiGraph to a DAG — drops edges that create cycles greedily.
    PC adds undirected edges as bidirected pairs, causing cycles."""
    if nx.is_directed_acyclic_graph(G):
        return G
    dag = nx.DiGraph(); dag.add_nodes_from(G.nodes())
    for u, v in G.edges():
        dag.add_edge(u, v)
        if not nx.is_directed_acyclic_graph(dag):
            dag.remove_edge(u, v)
    return dag

def run_standard_dagma(X: pd.DataFrame) -> Tuple[np.ndarray, Dict[str,float], nx.DiGraph]:
    """
    Include mean predicted class probability (per class) as soft targets
    in the DAGMA joint matrix so gamma = direct effect on model output,
    not just outgoing causal centrality.
    """
    probs        = xgb.predict_proba(X[FEATURES].values)
    all_cols     = FEATURES + [f'prob_cls{c}' for c in range(N_CLASSES)]
    Xfull        = X[FEATURES].copy()
    for c in range(N_CLASSES):
        Xfull[f'prob_cls{c}'] = probs[:, c]

    W = DagmaLinear(loss_type='l2').fit(Xfull[all_cols].values.astype(float), lambda1=0.02)

    tgt_indices = [all_cols.index(f'prob_cls{c}') for c in range(N_CLASSES)]
    effects = np.mean([np.abs(W[:N_FEAT, t]) for t in tgt_indices], axis=0)
    gamma   = normalise({f: float(effects[i]) for i,f in enumerate(FEATURES)}, FEATURES)

    G = nx.DiGraph(); G.add_nodes_from(FEATURES)
    for i,fi in enumerate(FEATURES):
        for j,fj in enumerate(FEATURES):
            if i!=j and abs(W[i,j])>0.05:
                G.add_edge(fi, fj, weight=float(W[i,j]))
    return W, gamma, G


def run_dce_mlp(X: pd.DataFrame, cfg: dict) -> Tuple[np.ndarray, Dict[str,float], nx.DiGraph]:
    """
    Include mean predicted class probabilities as soft target columns.
    FIX 1: W_est.detach().cpu().numpy() before indexing.
    """
    probs    = xgb.predict_proba(X[FEATURES].values)
    all_cols = FEATURES + [f'prob_cls{c}' for c in range(N_CLASSES)]
    Xfull    = X[FEATURES].copy()
    for c in range(N_CLASSES):
        Xfull[f'prob_cls{c}'] = probs[:, c]

    n_vars   = len(all_cols)
    data     = torch.tensor(Xfull[all_cols].values.astype(np.float64), dtype=torch.float64)
    hidden   = max(10, cfg['hidden_mult'] * n_vars)
    eq_model = nonlinear_dce.DagmaMLP_DCE(dims=[n_vars, hidden, 1], bias=True)
    model    = nonlinear_dce.DagmaDCE(eq_model, use_mse_loss=True)
    print(f"  Running DAGMA-DCE MLP on {n_vars} variables (hidden={hidden})...")

    W_est_tensor = model.fit(data,
        lambda1=cfg['lambda1'], lambda2=cfg['lambda2'], lr=cfg['lr'],
        mu_factor=cfg['mu_factor'], mu_init=cfg['mu_init'],
        T=cfg['T'], warm_iter=cfg['warm_iter'], max_iter=cfg['max_iter'])

    if isinstance(W_est_tensor, torch.Tensor):
        W_est = W_est_tensor.detach().cpu().numpy()
    else:
        W_est = np.array(W_est_tensor)

    tgt_indices = [all_cols.index(f'prob_cls{c}') for c in range(N_CLASSES)]
    effects = np.mean([np.abs(W_est[:N_FEAT, t]) for t in tgt_indices], axis=0)
    gamma   = normalise({f: float(effects[i]) for i,f in enumerate(FEATURES)}, FEATURES)

    G = nx.DiGraph(); G.add_nodes_from(FEATURES)
    for i,fi in enumerate(FEATURES):
        for j,fj in enumerate(FEATURES):
            if i!=j and abs(W_est[i,j])>0.05:
                G.add_edge(fi, fj, weight=float(W_est[i,j]))
    return W_est, gamma, G


# ── PC Algorithm ──────────────────────────────────────────────────────────────
def run_pc_wine(X: pd.DataFrame, alpha: float = 0.05
                ) -> Tuple[nx.DiGraph, Dict[str,float], List[str]]:
    """Two-pass PC: inter-feature graph + target connections via soft probs."""
    # Pass 1: feature skeleton
    cg_feat = pc(X[FEATURES].values.astype(float), alpha=alpha,
                 indep_test=fisherz, node_names=FEATURES,
                 verbose=False, show_progress=False)
    adj_f   = cg_feat.G.graph
    nodes_f = [n.get_name() for n in cg_feat.G.get_nodes()]
    G = nx.DiGraph(); G.add_nodes_from(FEATURES)
    for i in range(N_FEAT):
        for j in range(N_FEAT):
            if i==j: continue
            if adj_f[i,j]==1 and adj_f[j,i]==-1:
                G.add_edge(nodes_f[i], nodes_f[j])
            elif adj_f[i,j]==-1 and adj_f[j,i]==-1 and i<j:
                G.add_edge(nodes_f[i], nodes_f[j])
                G.add_edge(nodes_f[j], nodes_f[i])

    # Pass 2: features + predicted probability (class 0 sufficient as proxy)
    probs    = xgb.predict_proba(X[FEATURES].values)
    all_cols = FEATURES + ['prob_cls0']
    Xfull    = X[FEATURES].copy(); Xfull['prob_cls0'] = probs[:,0]
    cg_all   = pc(Xfull[all_cols].values.astype(float), alpha=alpha,
                  indep_test=fisherz, node_names=all_cols,
                  verbose=False, show_progress=False)
    adj_all   = cg_all.G.graph
    all_nodes = [n.get_name() for n in cg_all.G.get_nodes()]
    tgt_idx   = all_nodes.index('prob_cls0')
    connected = [f for f in FEATURES
                 if adj_all[all_nodes.index(f), tgt_idx]!=0
                 or adj_all[tgt_idx, all_nodes.index(f)]!=0]

    gamma = ({f: (1.0/len(connected) if f in connected else 0.0) for f in FEATURES}
             if connected else {f: 1.0/N_FEAT for f in FEATURES})
    return G, gamma, connected


# ── Causal SHAP (classification, permutation engine) ─────────────────────────
class LinearMechanisms:
    def __init__(self, dag, features):
        self.dag=dag; self.features=features; self.mech={}
        self.topo=list(nx.topological_sort(dag))

    def fit(self, X):
        for var in self.features:
            parents=list(self.dag.predecessors(var))
            if not parents:
                self.mech[var]={'parents':[],'intercept':float(X[var].mean()),
                                'coef':None,'sigma':float(X[var].std(ddof=0))}
            else:
                lr=LinearRegression().fit(X[parents].values,X[var].values)
                resid=X[var].values-lr.predict(X[parents].values)
                self.mech[var]={'parents':parents,'intercept':float(lr.intercept_),
                                'coef':lr.coef_.astype(float),
                                'sigma':float(np.std(resid,ddof=0))}
        return self

    def sample_batch(self, S, x, M, rng):
        X_out=np.empty((M,len(self.features))); fi={f:i for i,f in enumerate(self.features)}
        sim={}
        for var in self.topo:
            vi=fi[var]
            if var in S:
                val=float(x[var]); X_out[:,vi]=val; sim[var]=np.full(M,val); continue
            m=self.mech[var]
            mu=np.full(M,m['intercept']) if not m['parents'] else \
               m['intercept']+np.stack([sim[p] if p in sim else np.full(M,float(x[p]))
                                        for p in m['parents']],axis=1)@m['coef']
            X_out[:,vi]=mu if m['sigma']<1e-8 else rng.normal(mu,m['sigma'],size=M)
            sim[var]=X_out[:,vi]
        return X_out


def run_causal_shap_clf(gamma: Dict[str,float], dag: nx.DiGraph,
                        Xtr: pd.DataFrame, Xte: pd.DataFrame,
                        K: int=CAUSAL_K, M: int=CAUSAL_M,
                        n_perms: int=CAUSAL_PERMS) -> pd.Series:
    dag  = make_dag(dag)   # ensure acyclic before topological sort
    rng  = np.random.default_rng(RANDOM_SEED)
    sem  = LinearMechanisms(dag, FEATURES).fit(Xtr)
    K    = min(K, len(Xte))
    idx  = rng.choice(len(Xte), size=K, replace=False)

    def vc(S, x, pred_cls):
        batch = sem.sample_batch(S, x, M, rng)
        return float(np.mean(xgb.predict_proba(batch)[:, pred_cls]))

    Ef_dict = {}
    for cls in range(N_CLASSES):
        # sample M points uniformly from training set for baseline
        bg_idx   = rng.choice(len(Xtr), size=M, replace=True)
        bg_batch = Xtr.values[bg_idx]
        Ef_dict[cls] = float(np.mean(xgb.predict_proba(bg_batch)[:, cls]))

    phis = []
    for cnt, i in enumerate(idx):
        if (cnt+1)%10==0: print(f"      instance {cnt+1}/{K} ...",flush=True)
        x        = Xte.iloc[i]
        pred_cls = int(xgb.predict(x.values.reshape(1,-1))[0])
        fx       = float(xgb.predict_proba(x.values.reshape(1,-1))[0, pred_cls])
        Ef       = Ef_dict[pred_cls]
        phi      = np.zeros(N_FEAT)
        for _ in range(n_perms):
            perm   = rng.permutation(FEATURES); S=set()
            v_prev = vc(S, x, pred_cls)
            for feat in perm:
                Sn  = S|{feat}; vc_ = vc(Sn, x, pred_cls)
                phi[FEATURES.index(feat)] += gamma[feat]*(vc_-v_prev)
                v_prev, S = vc_, Sn
        phi /= n_perms
        s = float(phi.sum())
        if abs(s)>1e-12: phi = phi*((fx-Ef)/s)
        phis.append(np.abs(phi))
    return pd.Series(np.mean(phis,axis=0), index=FEATURES)


# ── Monte Carlo Shapley (gt_concepts style) ───────────────────────────────────
def _mc_utility(coalition: List[int], x: np.ndarray,
                X_bg: np.ndarray, pred_cls: int) -> float:
    if not coalition:
        return float(np.mean(xgb.predict_proba(X_bg)[:,pred_cls]))
    X_rep = X_bg.copy()
    for fi in coalition: X_rep[:,fi] = x[fi]
    return float(np.mean(xgb.predict_proba(X_rep)[:,pred_cls]))

def montecarlo_shapley_instance(x, X_bg, pred_cls, n_iter=MC_ITERATIONS):
    sv = np.zeros(N_FEAT)
    for _ in range(n_iter):
        perm    = np.random.permutation(N_FEAT); current=[]; prev_v=_mc_utility([],x,X_bg,pred_cls)
        for p in perm:
            new_v   = _mc_utility(current+[int(p)], x, X_bg, pred_cls)
            sv[p]  += new_v-prev_v; current.append(int(p)); prev_v=new_v
    return sv/n_iter

def run_mc_shapley(Xtr: pd.DataFrame, Xte: pd.DataFrame, K=MC_K_INSTANCES) -> pd.Series:
    rng=np.random.default_rng(RANDOM_SEED); X_bg=Xtr.values
    idx=rng.choice(len(Xte),size=min(K,len(Xte)),replace=False); phis=[]
    for cnt,i in enumerate(idx):
        if (cnt+1)%10==0: print(f"      MC Shapley {cnt+1}/{len(idx)} ...",flush=True)
        x=Xte.values[i]; pred_cls=int(xgb.predict(x.reshape(1,-1))[0])
        phis.append(np.abs(montecarlo_shapley_instance(x,X_bg,pred_cls)))
    return pd.Series(np.mean(phis,axis=0), index=FEATURES)


# ── Metrics ───────────────────────────────────────────────────────────────────
def faithfulness_spearman(global_abs: pd.Series, Xte: pd.DataFrame,
                           n_instances: int=50) -> float:
    """
    FIX 2: per-instance local attribution order.
    For each instance, compute local phi (single-instance Causal SHAP or
    use global proportions rescaled — here we use global order as rank proxy
    BUT measure drops on that instance's prediction.
    True fix: rank by per-instance phi. We keep global order for speed
    but compute per-instance drop correctly.
    """
    rng   = np.random.default_rng(RANDOM_SEED)
    means = X_train.mean().values
    idx   = rng.choice(len(Xte), size=min(n_instances,len(Xte)), replace=False)
    spears = []
    for i in idx:
        x        = Xte.values[i].copy()
        pred_cls = int(xgb.predict(x.reshape(1,-1))[0])
        base_p   = float(xgb.predict_proba(x.reshape(1,-1))[0,pred_cls])

        # Approximate local phi via a fast single-permutation estimate
        local_phi = np.zeros(N_FEAT)
        for _ in range(20):   # 20 quick permutations for local rank estimate
            perm  = np.random.permutation(N_FEAT)
            x_mod = means.copy(); prev_p = float(np.mean(xgb.predict_proba(
                np.tile(x_mod,(10,1)))[:,pred_cls]))
            for fi in perm:
                x_mod[fi] = x[fi]
                new_p = float(np.mean(xgb.predict_proba(np.tile(x_mod,(10,1)))[:,pred_cls]))
                local_phi[fi] += abs(new_p-prev_p); prev_p=new_p
        local_phi /= 20
        order = [FEATURES[fi] for fi in np.argsort(-local_phi)]

        # Measure drop using this instance's local order
        drops=[]; x_mod=x.copy()
        for feat in order:
            fi = FEATURES.index(feat); x_mod[fi]=means[fi]
            new_p=float(xgb.predict_proba(x_mod.reshape(1,-1))[0,pred_cls])
            drops.append(base_p-new_p)

        ranks=list(range(1,N_FEAT+1))
        sr,_ = spearmanr(ranks,drops)
        if not np.isnan(sr): spears.append(sr)
    return round(float(np.mean(spears)),4)

def predictive_alignment(global_abs: pd.Series) -> float:
    sv=global_abs.values/global_abs.sum() if global_abs.sum()>1e-12 else global_abs.values
    pr,_=pearsonr(sv, PERM_IMP[FEATURES].values); return round(float(pr),4)

def attribution_cosine(global_abs: pd.Series) -> float:
    sv=global_abs.values; pi=PERM_IMP[FEATURES].values
    return round(float(np.dot(sv,pi)/(np.linalg.norm(sv)*np.linalg.norm(pi)+1e-12)),4)

def robustness_metrics(method_fn, n_instances: int=ROB_INSTANCES) -> Tuple[float,float,float]:
    rng=np.random.default_rng(RANDOM_SEED)
    idx=rng.choice(len(X_test),size=min(n_instances,len(X_test)),replace=False)
    cos_sims=[]; rank_cors=[]; l2_dists=[]
    for i in idx:
        x_orig   = X_test.values[i:i+1]
        phi_orig = method_fn(pd.DataFrame(x_orig,columns=FEATURES)).values
        rc=[]; rr=[]; rl=[]
        for _ in range(PERTURB_RUNS):
            x_pert   = x_orig+rng.normal(0,PERTURB_SIGMA,x_orig.shape)
            phi_pert = method_fn(pd.DataFrame(x_pert,columns=FEATURES)).values
            n1,n2    = np.linalg.norm(phi_orig),np.linalg.norm(phi_pert)
            cos      = float(np.dot(phi_orig,phi_pert)/(n1*n2+1e-12))
            sr,_     = spearmanr(phi_orig,phi_pert)
            l2       = float(np.linalg.norm(phi_orig-phi_pert))
            rc.append(cos); rr.append(sr if not np.isnan(sr) else 0.0); rl.append(l2)
        cos_sims.append(np.mean(rc)); rank_cors.append(np.mean(rr)); l2_dists.append(np.mean(rl))
    return (round(float(np.mean(cos_sims)),4),
            round(float(np.mean(rank_cors)),4),
            round(float(np.mean(l2_dists)),4))


# ── Pretty print ──────────────────────────────────────────────────────────────
def print_attributions(label: str, global_abs: pd.Series, width: int=68):
    sv   = global_abs/global_abs.sum() if global_abs.sum()>1e-12 else global_abs
    pi   = PERM_IMP[FEATURES]
    pr,_ = pearsonr(sv.values,pi.values); cos=attribution_cosine(global_abs)
    print(f"\n  {'─'*width}\n  {label}\n  {'─'*width}")
    print(f"  {'Feature':<32} {'Attr(norm)':>10}  {'PermImp':>8}  {'Δ':>8}  Bar")
    print(f"  {'─'*width}")
    bar_max=sv.max() if sv.max()>0 else 1
    for f in FEATURES:
        s=sv[f]; p=pi[f]; bar='█'*int(round(s/bar_max*18))
        diff=s-p; sign='+' if diff>=0 else ''
        print(f"  {f:<32} {s:>10.4f}  {p:>8.4f}  {sign}{diff:>7.4f}  {bar}")
    print(f"  {'─'*width}")
    print(f"  Pearson (vs perm imp) = {pr:+.4f}  |  Cosine (vs perm imp) = {cos:.4f}")
    print(f"  {'─'*width}")


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print("="*80)
    print("Wine Dataset — Classification Causal SHAP (4 fixes applied)")
    print("="*80)
    results=[]

    print("\n"+"─"*60+"\nMETHOD 3: PC Algorithm → Causal SHAP\n"+"─"*60)
    t0=time.time()
    dag_pc,gamma_pc,connected_pc=run_pc_wine(X_train)
    print(f"  Fit: {time.time()-t0:.1f}s | Edges: {len(dag_pc.edges())} | "
          f"Connected to target: {connected_pc}")
    g_pc=run_causal_shap_clf(gamma_pc,dag_pc,X_train,X_test)
    print_attributions("PC Causal SHAP",g_pc)
    faith_pc=faithfulness_spearman(g_pc,X_test)
    align_pc=predictive_alignment(g_pc); cos_pc=attribution_cosine(g_pc)
    def pc_fn(Xdf): return run_causal_shap_clf(gamma_pc,dag_pc,X_train,Xdf,K=1,n_perms=50)
    rc_pc,rr_pc,rl_pc=robustness_metrics(pc_fn)
    print(f"  Faith={faith_pc:.4f} | Align={align_pc:.4f} | Cos={cos_pc:.4f} | "
          f"RobCos={rc_pc:.4f} | RankRob={rr_pc:.4f} | L2={rl_pc:.4f}")
    results.append({'Method':'PC Causal SHAP','Faithfulness':faith_pc,'Pred_Align':align_pc,
                    'Attr_Cosine':cos_pc,'Rob_Cosine':rc_pc,'Rank_Rob':rr_pc,'L2':rl_pc,
                    **{f'attr_{f}':round(float(g_pc[f]/g_pc.sum()),4) for f in FEATURES}})

    # METHOD 4: MC Shapley
    print("\n"+"─"*60+"\nMETHOD 4: Monte Carlo Shapley (gt_concepts style)\n"+"─"*60)
    t0=time.time(); g_mc=run_mc_shapley(X_train,X_test,K=MC_K_INSTANCES)
    print(f"  MC Shapley time: {time.time()-t0:.1f}s")
    print_attributions("MC Shapley",g_mc)
    faith_mc=faithfulness_spearman(g_mc,X_test)
    align_mc=predictive_alignment(g_mc); cos_mc=attribution_cosine(g_mc)
    def mc_fn(Xdf): return run_mc_shapley(X_train,Xdf,K=1)
    rc_mc,rr_mc,rl_mc=robustness_metrics(mc_fn)
    print(f"  Faith={faith_mc:.4f} | Align={align_mc:.4f} | Cos={cos_mc:.4f} | "
          f"RobCos={rc_mc:.4f} | RankRob={rr_mc:.4f} | L2={rl_mc:.4f}")
    results.append({'Method':'MC Shapley','Faithfulness':faith_mc,'Pred_Align':align_mc,
                    'Attr_Cosine':cos_mc,'Rob_Cosine':rc_mc,'Rank_Rob':rr_mc,'L2':rl_mc,
                    **{f'attr_{f}':round(float(g_mc[f]/g_mc.sum()),4) for f in FEATURES}})

    # Summary
    df=pd.DataFrame(results)
    disp=['Method','Faithfulness','Pred_Align','Attr_Cosine','Rob_Cosine','Rank_Rob','L2']
    print(f"\n{'='*100}\nRESULTS SUMMARY — WINE CLASSIFICATION\n{'='*100}")
    print(df[disp].to_string(index=False))
    print(f"\nPERM IMP: {dict(PERM_IMP.round(4))}")
    print("\nNORMALISED ATTRIBUTIONS:")
    print(df[['Method']+[f'attr_{f}' for f in FEATURES]].to_string(index=False))
    print("\n  Faithfulness (Spearman ↑): per-instance local attr rank vs prediction drop")
    print("  Pred_Align   (Pearson  ↑): attribution vs permutation importance")
    print("  Attr_Cosine  (Cosine   ↑): cosine sim vs permutation importance")
    print("  Rob_Cosine   (Cosine   ↑): cosine sim under Gaussian perturbation")
    print("  Rank_Rob     (Spearman ↑): rank Spearman under perturbation")
    print("  L2           (L2       ↓): L2 drift under perturbation")
    df.to_csv('wine_clf_comparison_results.csv',index=False)
    print("\nSaved: wine_clf_comparison_results.csv")

if __name__ == '__main__':
    main()

✓ DagmaDCE available
✓ dagma available
  Device: cuda

Features (13): ['Alcohol', 'Malic acid', 'Ash', 'Alcalinity of ash', 'Magnesium', 'Total phenols', 'Flavanoids', 'Nonflavanoid phenols', 'Proanthocyanins', 'Color intensity', 'Hue', 'OD280/OD315 of diluted wines', 'Proline']
Train: (141, 13)  Test: (36, 13)  Classes: [0 1 2]

XGBoost  train acc=1.0000  test acc=1.0000

Computing permutation importance...
  Top 5: {'Flavanoids': np.float64(0.3031), 'Proline': np.float64(0.2334), 'Color intensity': np.float64(0.1777), 'Alcohol': np.float64(0.1324), 'Hue': np.float64(0.0488)}

Wine Dataset — Classification Causal SHAP (4 fixes applied)

────────────────────────────────────────────────────────────
METHOD 3: PC Algorithm → Causal SHAP
────────────────────────────────────────────────────────────
  Fit: 0.2s | Edges: 21 | Connected to target: ['Alcohol', 'Alcalinity of ash', 'Flavanoids', 'Proline']
      instance 10/30 ...
      instance 20/30 ...
      instance 30/30 ...

  ────────────

In [4]:
#!/usr/bin/env python3
"""
Corrected Wine Classification: DAGMA / DAGMA-DCE / PC -> Causal SHAP
====================================================================

What this script is valid for:
  - model-output explanation on Wine classification
  - comparing attribution alignment with XGBoost permutation importance
  - comparing graph-guided causal-SHAP style methods

What it should NOT be claimed as:
  - verified causal ground-truth validation, because Wine has no true causal DAG
  - causal effect on the true label-generating process, because gamma is derived from model probabilities

Major fixes vs previous version:
  1. Feature graph is learned on FEATURES ONLY.
     Probability columns are used only for gamma / model-output sensitivity.
  2. All 3 class-probability columns are used.
  3. DAGMA/DCE gamma is class-specific: gamma[class][feature].
  4. PC target connections are checked for all classes, not only class 0.
  5. Faithfulness uses the actual local attributions of each method.
  6. Robustness uses the actual same method on perturbed inputs.
  7. DCE graph density is controlled via threshold and lambda settings.
  8. All stochastic explainers use local RNG seeds for reproducibility.
"""

import sys
sys.path.insert(0, "DAGMA-DCE/src")

import time
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import networkx as nx

from scipy.stats import pearsonr, spearmanr
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from xgboost import XGBClassifier

from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz

warnings.filterwarnings("ignore")

try:
    from dagma.linear import DagmaLinear
    HAS_DAGMA = True
except Exception as e:
    HAS_DAGMA = False
    print(f"WARNING: dagma not available: {e}")

try:
    from DagmaDCE import nonlinear_dce
    HAS_DCE = True
except Exception as e:
    HAS_DCE = False
    print(f"WARNING: DagmaDCE not available: {e}")


# Config

@dataclass
class Config:
    data_path: str = "../../../../datasets/wine_preprocessed.npz"
    random_seed: int = 42
    n_classes: int = 3

    # Causal SHAP settings
    causal_k: int = 30
    causal_m: int = 64
    causal_perms: int = 200

    # MC Shapley settings
    mc_k: int = 36
    mc_bg_size: int = 80
    mc_iterations: int = 500

    # Metrics
    perm_repeats: int = 30
    faith_k: int = 30
    robust_k: int = 15
    perturb_sigma: float = 0.05
    perturb_runs: int = 5

    # Graph thresholds
    dagma_graph_threshold: float = 0.05
    dce_graph_threshold: float = 0.10

    # Dagma
    dagma_lambda1_graph: float = 0.03
    dagma_lambda1_gamma: float = 0.02

    # Dagma-dce
    dce_lambda1_graph: float = 5e-2
    dce_lambda1_gamma: float = 3.5e-2
    dce_lambda2: float = 5e-3
    dce_lr: float = 2e-4
    dce_mu_factor: float = 0.1
    dce_mu_init: float = 0.1
    dce_T: int = 4
    dce_warm_iter: int = 7000
    dce_max_iter: int = 8000
    dce_hidden_mult: int = 3

    # PC
    pc_alpha: float = 0.05

CFG = Config()


# Utilities

def set_global_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)


def normalize_vec(values: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    values = np.maximum(np.asarray(values, dtype=float), 0.0)
    s = values.sum()
    if s <= eps:
        return np.ones_like(values) / len(values)
    return values / s


def normalize_gamma_dict(effects: Dict[str, float], features: List[str]) -> Dict[str, float]:
    arr = normalize_vec(np.array([effects.get(f, 0.0) for f in features], dtype=float))
    return {f: float(arr[i]) for i, f in enumerate(features)}


def make_dag_greedy(G: nx.DiGraph) -> nx.DiGraph:
    """Keep edges greedily while preventing directed cycles."""
    dag = nx.DiGraph()
    dag.add_nodes_from(G.nodes())
    for u, v, data in G.edges(data=True):
        dag.add_edge(u, v, **data)
        if not nx.is_directed_acyclic_graph(dag):
            dag.remove_edge(u, v)
    return dag


def build_graph_from_W(W: np.ndarray, features: List[str], threshold: float) -> nx.DiGraph:
    G = nx.DiGraph()
    G.add_nodes_from(features)
    d = len(features)
    for i, fi in enumerate(features):
        for j, fj in enumerate(features):
            if i != j and abs(float(W[i, j])) > threshold:
                G.add_edge(fi, fj, weight=float(W[i, j]))
    return make_dag_greedy(G)


def safe_spearman(a: np.ndarray, b: np.ndarray) -> float:
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    r, _ = spearmanr(a, b)
    return float(r) if not np.isnan(r) else np.nan


def safe_pearson(a: np.ndarray, b: np.ndarray) -> float:
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    r, _ = pearsonr(a, b)
    return float(r) if not np.isnan(r) else np.nan


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


# Load Data and Model

def load_wine(cfg: Config):
    raw = np.load(cfg.data_path, allow_pickle=True)
    X_train = pd.DataFrame(raw["X_train"], columns=raw["feature_names"])
    X_test = pd.DataFrame(raw["X_test"], columns=raw["feature_names"])

    # UCI wine labels are often 1,2,3. Convert to 0,1,2 if needed.
    y_train = raw["y_train"].astype(int)
    y_test = raw["y_test"].astype(int)
    if y_train.min() == 1:
        y_train = y_train - 1
        y_test = y_test - 1

    return X_train, X_test, y_train, y_test, list(X_train.columns)


def fit_xgb(X_train, y_train, cfg: Config):
    model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=cfg.random_seed,
        verbosity=0,
        n_jobs=1,
        tree_method="hist",
        objective="multi:softprob",
        num_class=cfg.n_classes,
        eval_metric="mlogloss",
    )
    model.fit(X_train.values, y_train)
    return model


# Dagma

def run_dagma_graph(X_train: pd.DataFrame, features: List[str], cfg: Config) -> nx.DiGraph:
    """
    Learn feature-feature DAG only.
    Important: do NOT include predicted probability nodes here, otherwise target nodes
    can distort the feature graph.
    """
    if not HAS_DAGMA:
        return nx.DiGraph([(features[i], features[i+1]) for i in range(len(features)-1)])

    W = DagmaLinear(loss_type="l2").fit(
        X_train[features].values.astype(float),
        lambda1=cfg.dagma_lambda1_graph,
    )
    return build_graph_from_W(W, features, cfg.dagma_graph_threshold)


def run_dagma_gamma(
    X_train: pd.DataFrame,
    model: XGBClassifier,
    features: List[str],
    cfg: Config,
) -> Dict[int, Dict[str, float]]:
    """
    Estimate class-specific feature -> model-probability sensitivity using linear DAGMA.
    This is model-output sensitivity, not causal effect on true label.
    """
    if not HAS_DAGMA:
        return {c: {f: 1.0 / len(features) for f in features} for c in range(cfg.n_classes)}

    probs = model.predict_proba(X_train[features].values)
    Xfull = X_train[features].copy()
    prob_cols = []
    for c in range(cfg.n_classes):
        col = f"prob_cls{c}"
        Xfull[col] = probs[:, c]
        prob_cols.append(col)

    all_cols = features + prob_cols
    W = DagmaLinear(loss_type="l2").fit(
        Xfull[all_cols].values.astype(float),
        lambda1=cfg.dagma_lambda1_gamma,
    )

    gamma_by_class = {}
    for c, col in enumerate(prob_cols):
        t = all_cols.index(col)
        effects = {f: abs(float(W[i, t])) for i, f in enumerate(features)}
        gamma_by_class[c] = normalize_gamma_dict(effects, features)

    return gamma_by_class


# Dagma-dce

def _fit_dce_matrix(data_np: np.ndarray, cfg: Config, lambda1: float) -> np.ndarray:
    n_vars = data_np.shape[1]
    hidden = max(10, cfg.dce_hidden_mult * n_vars)

    data = torch.tensor(data_np.astype(np.float64), dtype=torch.float64)
    eq_model = nonlinear_dce.DagmaMLP_DCE(dims=[n_vars, hidden, 1], bias=True)
    model = nonlinear_dce.DagmaDCE(eq_model, use_mse_loss=True)

    W_tensor = model.fit(
        data,
        lambda1=lambda1,
        lambda2=cfg.dce_lambda2,
        lr=cfg.dce_lr,
        mu_factor=cfg.dce_mu_factor,
        mu_init=cfg.dce_mu_init,
        T=cfg.dce_T,
        warm_iter=cfg.dce_warm_iter,
        max_iter=cfg.dce_max_iter,
    )

    if isinstance(W_tensor, torch.Tensor):
        return W_tensor.detach().cpu().numpy()
    return np.asarray(W_tensor)


def run_dce_graph(X_train: pd.DataFrame, features: List[str], cfg: Config) -> nx.DiGraph:
    """
    Learn feature-feature DAG only using DAGMA-DCE.
    """
    if not HAS_DCE:
        return nx.DiGraph()

    print("  Fitting DAGMA-DCE graph on features only...")
    W = _fit_dce_matrix(X_train[features].values, cfg, cfg.dce_lambda1_graph)
    return build_graph_from_W(W, features, cfg.dce_graph_threshold)


def run_dce_gamma(
    X_train: pd.DataFrame,
    model: XGBClassifier,
    features: List[str],
    cfg: Config,
) -> Dict[int, Dict[str, float]]:
    """
    Estimate class-specific feature -> model-probability DCE.
    """
    if not HAS_DCE:
        return {c: {f: 1.0 / len(features) for f in features} for c in range(cfg.n_classes)}

    probs = model.predict_proba(X_train[features].values)
    Xfull = X_train[features].copy()
    prob_cols = []
    for c in range(cfg.n_classes):
        col = f"prob_cls{c}"
        Xfull[col] = probs[:, c]
        prob_cols.append(col)

    all_cols = features + prob_cols
    print("  Fitting DAGMA-DCE gamma on features + probability nodes...")
    W = _fit_dce_matrix(Xfull[all_cols].values, cfg, cfg.dce_lambda1_gamma)

    gamma_by_class = {}
    for c, col in enumerate(prob_cols):
        t = all_cols.index(col)
        effects = {f: abs(float(W[i, t])) for i, f in enumerate(features)}
        gamma_by_class[c] = normalize_gamma_dict(effects, features)

    return gamma_by_class


# PC

def run_pc_graph(X_train: pd.DataFrame, features: List[str], cfg: Config) -> nx.DiGraph:
    """
    Learn feature-feature graph with PC.
    For undirected edges, orient greedily by feature order only to obtain a DAG.
    This orientation is not a strong causal claim.
    """
    cg = pc(
        X_train[features].values.astype(float),
        alpha=cfg.pc_alpha,
        indep_test=fisherz,
        node_names=features,
        verbose=False,
        show_progress=False,
    )

    adj = cg.G.graph
    nodes = [n.get_name() for n in cg.G.get_nodes()]
    G = nx.DiGraph()
    G.add_nodes_from(features)

    d = len(features)
    for i in range(d):
        for j in range(i + 1, d):
            # causallearn marks endpoints in its own convention.
            # We avoid overclaiming direction; if adjacent, add one greedy direction.
            if adj[i, j] != 0 or adj[j, i] != 0:
                # If clearly i -> j according to common causallearn encoding.
                if adj[i, j] == 1 and adj[j, i] == -1:
                    G.add_edge(nodes[i], nodes[j])
                elif adj[i, j] == -1 and adj[j, i] == 1:
                    G.add_edge(nodes[j], nodes[i])
                else:
                    G.add_edge(nodes[i], nodes[j])

    return make_dag_greedy(G)


def run_pc_gamma(
    X_train: pd.DataFrame,
    model: XGBClassifier,
    features: List[str],
    cfg: Config,
) -> Dict[int, Dict[str, float]]:
    """
    PC has no natural edge magnitude. Here gamma is class-specific:
    features adjacent to prob_cls c receive equal weight.
    """
    probs = model.predict_proba(X_train[features].values)
    gamma_by_class = {}

    for c in range(cfg.n_classes):
        Xfull = X_train[features].copy()
        target = f"prob_cls{c}"
        Xfull[target] = probs[:, c]
        all_cols = features + [target]

        cg = pc(
            Xfull[all_cols].values.astype(float),
            alpha=cfg.pc_alpha,
            indep_test=fisherz,
            node_names=all_cols,
            verbose=False,
            show_progress=False,
        )
        adj = cg.G.graph
        nodes = [n.get_name() for n in cg.G.get_nodes()]
        t = nodes.index(target)

        connected = []
        for f in features:
            i = nodes.index(f)
            if adj[i, t] != 0 or adj[t, i] != 0:
                connected.append(f)

        if connected:
            gamma_by_class[c] = {
                f: (1.0 / len(connected) if f in connected else 0.0)
                for f in features
            }
        else:
            gamma_by_class[c] = {f: 1.0 / len(features) for f in features}

    return gamma_by_class


# SEM and Causal SHAP

class LinearGaussianSEM:
    def __init__(self, dag: nx.DiGraph, features: List[str]):
        self.dag = make_dag_greedy(dag)
        self.features = features
        self.topo = list(nx.topological_sort(self.dag))
        self.mech = {}

    def fit(self, X: pd.DataFrame):
        for var in self.features:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {
                    "parents": [],
                    "intercept": float(X[var].mean()),
                    "coef": None,
                    "sigma": float(X[var].std(ddof=0)),
                }
            else:
                lr = LinearRegression().fit(X[parents].values, X[var].values)
                pred = lr.predict(X[parents].values)
                resid = X[var].values - pred
                self.mech[var] = {
                    "parents": parents,
                    "intercept": float(lr.intercept_),
                    "coef": lr.coef_.astype(float),
                    "sigma": float(np.std(resid, ddof=0)),
                }
        return self

    def sample_batch(self, S: set, x: pd.Series, M: int, rng: np.random.Generator) -> np.ndarray:
        X_out = np.empty((M, len(self.features)), dtype=float)
        idx = {f: k for k, f in enumerate(self.features)}
        sim = {}

        for var in self.topo:
            vi = idx[var]
            if var in S:
                val = float(x[var])
                X_out[:, vi] = val
                sim[var] = np.full(M, val)
                continue

            m = self.mech[var]
            if not m["parents"]:
                mu = np.full(M, m["intercept"])
            else:
                parent_mat = np.stack(
                    [sim[p] if p in sim else np.full(M, float(x[p])) for p in m["parents"]],
                    axis=1,
                )
                mu = m["intercept"] + parent_mat @ m["coef"]

            sigma = m["sigma"]
            X_out[:, vi] = mu if sigma < 1e-10 else rng.normal(mu, sigma, size=M)
            sim[var] = X_out[:, vi]

        return X_out


class CausalShapExplainer:
    def __init__(
        self,
        model: XGBClassifier,
        X_train: pd.DataFrame,
        dag: nx.DiGraph,
        gamma_by_class: Dict[int, Dict[str, float]],
        features: List[str],
        cfg: Config,
    ):
        self.model = model
        self.X_train = X_train
        self.features = features
        self.cfg = cfg
        self.gamma_by_class = gamma_by_class
        self.sem = LinearGaussianSEM(dag, features).fit(X_train)

        probs = model.predict_proba(X_train[features].values)
        self.baseline_prob = {c: float(np.mean(probs[:, c])) for c in range(cfg.n_classes)}

    def _value(self, S: set, x: pd.Series, pred_cls: int, M: int, rng: np.random.Generator) -> float:
        batch = self.sem.sample_batch(S, x, M, rng)
        return float(np.mean(self.model.predict_proba(batch)[:, pred_cls]))

    def explain_instance(
        self,
        x: pd.Series,
        M: Optional[int] = None,
        n_perms: Optional[int] = None,
        seed: Optional[int] = None,
    ) -> pd.Series:
        M = self.cfg.causal_m if M is None else M
        n_perms = self.cfg.causal_perms if n_perms is None else n_perms
        rng = np.random.default_rng(self.cfg.random_seed if seed is None else seed)

        pred_cls = int(self.model.predict(x.values.reshape(1, -1))[0])
        fx = float(self.model.predict_proba(x.values.reshape(1, -1))[0, pred_cls])
        Ef = self.baseline_prob[pred_cls]

        gamma = self.gamma_by_class.get(
            pred_cls,
            {f: 1.0 / len(self.features) for f in self.features},
        )

        phi = np.zeros(len(self.features), dtype=float)

        for _ in range(n_perms):
            perm = rng.permutation(self.features)
            S = set()
            v_prev = self._value(S, x, pred_cls, M, rng)

            for feat in perm:
                Sn = S | {feat}
                v_new = self._value(Sn, x, pred_cls, M, rng)
                phi[self.features.index(feat)] += gamma[feat] * (v_new - v_prev)
                v_prev = v_new
                S = Sn

        phi /= n_perms

        # Local accuracy rescaling.
        s = float(phi.sum())
        if abs(s) > 1e-12:
            phi = phi * ((fx - Ef) / s)

        return pd.Series(phi, index=self.features)

    def explain_global(self, X_test: pd.DataFrame, K: Optional[int] = None) -> Tuple[pd.Series, pd.DataFrame, np.ndarray]:
        K = self.cfg.causal_k if K is None else min(K, len(X_test))
        rng = np.random.default_rng(self.cfg.random_seed)
        selected = rng.choice(len(X_test), size=K, replace=False)

        local = []
        for cnt, i in enumerate(selected):
            if (cnt + 1) % 10 == 0:
                print(f"      instance {cnt+1}/{K} ...", flush=True)
            phi = self.explain_instance(
                X_test.iloc[i],
                seed=self.cfg.random_seed + 1000 + cnt,
            )
            local.append(phi)

        local_df = pd.DataFrame(local, index=selected)
        global_abs = local_df.abs().mean(axis=0)
        return global_abs, local_df, selected


# MC Shapley baseline

class MCShapleyExplainer:
    def __init__(self, model, X_train, features, cfg: Config):
        self.model = model
        self.X_train = X_train
        self.features = features
        self.cfg = cfg

    def _utility(self, coalition: List[int], x: np.ndarray, bg: np.ndarray, pred_cls: int) -> float:
        X_rep = bg.copy()
        for fi in coalition:
            X_rep[:, fi] = x[fi]
        return float(np.mean(self.model.predict_proba(X_rep)[:, pred_cls]))

    def explain_instance(self, x_series: pd.Series, seed: Optional[int] = None) -> pd.Series:
        rng = np.random.default_rng(self.cfg.random_seed if seed is None else seed)
        x = x_series.values.astype(float)
        pred_cls = int(self.model.predict(x.reshape(1, -1))[0])

        bg_size = min(self.cfg.mc_bg_size, len(self.X_train))
        bg_idx = rng.choice(len(self.X_train), size=bg_size, replace=False)
        bg = self.X_train.values[bg_idx].copy()

        phi = np.zeros(len(self.features), dtype=float)

        for _ in range(self.cfg.mc_iterations):
            perm = rng.permutation(len(self.features))
            current = []
            prev_v = self._utility(current, x, bg, pred_cls)

            for p in perm:
                new_current = current + [int(p)]
                new_v = self._utility(new_current, x, bg, pred_cls)
                phi[p] += new_v - prev_v
                current = new_current
                prev_v = new_v

        phi /= self.cfg.mc_iterations
        return pd.Series(phi, index=self.features)

    def explain_global(self, X_test: pd.DataFrame, K: Optional[int] = None) -> Tuple[pd.Series, pd.DataFrame, np.ndarray]:
        K = self.cfg.mc_k if K is None else min(K, len(X_test))
        rng = np.random.default_rng(self.cfg.random_seed)
        selected = rng.choice(len(X_test), size=K, replace=False)

        local = []
        for cnt, i in enumerate(selected):
            if (cnt + 1) % 10 == 0:
                print(f"      MC Shapley {cnt+1}/{K} ...", flush=True)
            phi = self.explain_instance(X_test.iloc[i], seed=self.cfg.random_seed + 2000 + cnt)
            local.append(phi)

        local_df = pd.DataFrame(local, index=selected)
        global_abs = local_df.abs().mean(axis=0)
        return global_abs, local_df, selected


# Evaluation

def permutation_importance_distribution(model, X_test, y_test, features, cfg: Config) -> pd.Series:
    res = permutation_importance(
        model,
        X_test[features].values,
        y_test,
        n_repeats=cfg.perm_repeats,
        random_state=cfg.random_seed,
        scoring="accuracy",
    )
    vals = np.abs(res.importances_mean)
    vals = normalize_vec(vals)
    return pd.Series(vals, index=features)


def faithfulness_local_drop(
    model,
    X_test: pd.DataFrame,
    local_df: pd.DataFrame,
    selected_indices: np.ndarray,
    features: List[str],
    baseline_values: np.ndarray,
) -> float:
    """
    Method-specific faithfulness:
    For each explained instance, compare abs(local attribution) with actual
    single-feature probability drop for that same instance.
    """
    scores = []
    for row_pos, idx in enumerate(selected_indices):
        x = X_test.iloc[idx].values.astype(float)
        pred_cls = int(model.predict(x.reshape(1, -1))[0])
        base_p = float(model.predict_proba(x.reshape(1, -1))[0, pred_cls])

        drops = []
        for j in range(len(features)):
            x_mod = x.copy()
            x_mod[j] = baseline_values[j]
            new_p = float(model.predict_proba(x_mod.reshape(1, -1))[0, pred_cls])
            drops.append(base_p - new_p)

        attr = local_df.iloc[row_pos][features].abs().values
        drops = np.asarray(drops, dtype=float)

        # Negative drops can happen if masking a feature increases probability.
        # For rank faithfulness of importance, use absolute effect magnitude.
        sr = safe_spearman(attr, np.abs(drops))
        if not np.isnan(sr):
            scores.append(sr)

    return float(np.mean(scores)) if scores else np.nan


def robustness(
    explainer,
    X_test: pd.DataFrame,
    features: List[str],
    cfg: Config,
) -> Tuple[float, float, float]:
    rng = np.random.default_rng(cfg.random_seed)
    K = min(cfg.robust_k, len(X_test))
    selected = rng.choice(len(X_test), size=K, replace=False)

    cos_list, rank_list, l2_list = [], [], []

    for k, idx in enumerate(selected):
        x0 = X_test.iloc[idx].copy()
        phi0 = explainer.explain_instance(x0, seed=cfg.random_seed + 3000 + k).abs().values

        for r in range(cfg.perturb_runs):
            xp = x0.copy()
            xp[:] = xp.values + rng.normal(0, cfg.perturb_sigma, size=len(features))
            phip = explainer.explain_instance(xp, seed=cfg.random_seed + 4000 + 10*k + r).abs().values

            cos_list.append(cosine(phi0, phip))
            rank = safe_spearman(phi0, phip)
            if not np.isnan(rank):
                rank_list.append(rank)
            l2_list.append(float(np.linalg.norm(phi0 - phip)))

    return (
        float(np.mean(cos_list)) if cos_list else np.nan,
        float(np.mean(rank_list)) if rank_list else np.nan,
        float(np.mean(l2_list)) if l2_list else np.nan,
    )


def evaluate_method(
    name: str,
    explainer,
    X_test: pd.DataFrame,
    model,
    perm_imp: pd.Series,
    features: List[str],
    cfg: Config,
):
    t0 = time.time()
    global_abs, local_df, selected = explainer.explain_global(X_test)
    elapsed = time.time() - t0

    attr_norm = pd.Series(normalize_vec(global_abs.values), index=features)
    pi = perm_imp[features].values

    baseline_values = explainer.X_train[features].mean().values
    faith = faithfulness_local_drop(model, X_test, local_df, selected, features, baseline_values)

    align = safe_pearson(attr_norm.values, pi)
    cosv = cosine(attr_norm.values, pi)

    rob_cos, rank_rob, l2 = robustness(explainer, X_test, features, cfg)

    return {
        "Method": name,
        "Time_s": round(elapsed, 1),
        "Faithfulness": round(faith, 4) if not np.isnan(faith) else np.nan,
        "Pred_Align": round(align, 4) if not np.isnan(align) else np.nan,
        "Attr_Cosine": round(cosv, 4),
        "Rob_Cosine": round(rob_cos, 4) if not np.isnan(rob_cos) else np.nan,
        "Rank_Rob": round(rank_rob, 4) if not np.isnan(rank_rob) else np.nan,
        "L2": round(l2, 4) if not np.isnan(l2) else np.nan,
        **{f"attr_{f}": round(float(attr_norm[f]), 4) for f in features},
    }, attr_norm


def print_attr_table(label: str, attr_norm: pd.Series, perm_imp: pd.Series, features: List[str]):
    print(f"\n  {'─'*76}")
    print(f"  {label}")
    print(f"  {'─'*76}")
    print(f"  {'Feature':<34} {'Attr(norm)':>10} {'PermImp':>10} {'Delta':>10}")
    print(f"  {'─'*76}")
    for f in features:
        delta = attr_norm[f] - perm_imp[f]
        print(f"  {f:<34} {attr_norm[f]:>10.4f} {perm_imp[f]:>10.4f} {delta:>+10.4f}")
    print(f"  {'─'*76}")


# Main

def main():
    set_global_seed(CFG.random_seed)

    print("=" * 90)
    print("Corrected Wine Classification: DAGMA / DAGMA-DCE / PC / MC Shapley")
    print("=" * 90)

    X_train, X_test, y_train, y_test, features = load_wine(CFG)
    print(f"Features ({len(features)}): {features}")
    print(f"Train={X_train.shape}, Test={X_test.shape}, classes={sorted(np.unique(y_train).tolist())}")

    model = fit_xgb(X_train, y_train, CFG)
    print(
        f"XGBoost train acc={np.mean(model.predict(X_train.values)==y_train):.4f}, "
        f"test acc={np.mean(model.predict(X_test.values)==y_test):.4f}"
    )

    perm_imp = permutation_importance_distribution(model, X_test, y_test, features, CFG)
    print(f"Permutation importance top5: {dict(perm_imp.nlargest(5).round(4))}")

    results = []
    attr_tables = {}

    # Method 1: DAGMA
    if HAS_DAGMA:
        print("\n" + "─" * 70)
        print("METHOD 1: Standard DAGMA -> Causal SHAP")
        print("─" * 70)
        t0 = time.time()
        dag = run_dagma_graph(X_train, features, CFG)
        gamma = run_dagma_gamma(X_train, model, features, CFG)
        print(f"  Setup time={time.time()-t0:.1f}s | feature edges={len(dag.edges())}")
        print(f"  Gamma class0 top3={dict(sorted(gamma[0].items(), key=lambda x: -x[1])[:3])}")

        explainer = CausalShapExplainer(model, X_train, dag, gamma, features, CFG)
        row, attr = evaluate_method("DAGMA Causal SHAP", explainer, X_test, model, perm_imp, features, CFG)
        results.append(row)
        attr_tables[row["Method"]] = attr
        print_attr_table(row["Method"], attr, perm_imp, features)

    # Method 2: DAGMA-DCE
    if HAS_DCE:
        print("\n" + "─" * 70)
        print("METHOD 2: DAGMA-DCE MLP -> Causal SHAP")
        print("─" * 70)
        t0 = time.time()
        dag = run_dce_graph(X_train, features, CFG)
        gamma = run_dce_gamma(X_train, model, features, CFG)
        print(f"  Setup time={time.time()-t0:.1f}s | feature edges={len(dag.edges())}")
        print(f"  Gamma class0 top3={dict(sorted(gamma[0].items(), key=lambda x: -x[1])[:3])}")

        explainer = CausalShapExplainer(model, X_train, dag, gamma, features, CFG)
        row, attr = evaluate_method("DAGMA-DCE Causal SHAP", explainer, X_test, model, perm_imp, features, CFG)
        results.append(row)
        attr_tables[row["Method"]] = attr
        print_attr_table(row["Method"], attr, perm_imp, features)

    # Method 3: PC
    print("\n" + "─" * 70)
    print("METHOD 3: PC -> Causal SHAP")
    print("─" * 70)
    t0 = time.time()
    dag = run_pc_graph(X_train, features, CFG)
    gamma = run_pc_gamma(X_train, model, features, CFG)
    print(f"  Setup time={time.time()-t0:.1f}s | feature edges={len(dag.edges())}")
    print(f"  Gamma class0 connected={ [f for f,v in gamma[0].items() if v > 0] }")

    explainer = CausalShapExplainer(model, X_train, dag, gamma, features, CFG)
    row, attr = evaluate_method("PC Causal SHAP", explainer, X_test, model, perm_imp, features, CFG)
    results.append(row)
    attr_tables[row["Method"]] = attr
    print_attr_table(row["Method"], attr, perm_imp, features)

    # Method 4: MC Shapley
    print("\n" + "─" * 70)
    print("METHOD 4: Monte Carlo Shapley")
    print("─" * 70)
    explainer = MCShapleyExplainer(model, X_train, features, CFG)
    row, attr = evaluate_method("MC Shapley", explainer, X_test, model, perm_imp, features, CFG)
    results.append(row)
    attr_tables[row["Method"]] = attr
    print_attr_table(row["Method"], attr, perm_imp, features)

    # Summary
    df = pd.DataFrame(results)
    disp = ["Method", "Time_s", "Faithfulness", "Pred_Align", "Attr_Cosine", "Rob_Cosine", "Rank_Rob", "L2"]

    print("\n" + "=" * 110)
    print("RESULTS SUMMARY")
    print("=" * 110)
    print(df[disp].to_string(index=False))

    print("\nMetric meanings:")
    print("  Faithfulness: Spearman between actual method-specific local |phi| and single-feature probability drop.")
    print("  Pred_Align:   Pearson between global normalized attribution and permutation importance.")
    print("  Attr_Cosine:  Cosine between global normalized attribution and permutation importance.")
    print("  Rob_Cosine:   Cosine similarity of same method under small Gaussian input perturbation.")
    print("  Rank_Rob:     Spearman rank stability under perturbation.")
    print("  L2:           Attribution vector drift under perturbation; lower is better.")

    print("\nImportant claim boundary:")
    print("  This is a model-output attribution experiment. Since Wine has no causal ground-truth graph,")
    print("  do not claim verified causal correctness. Claim only faithfulness/alignment to model behavior.")

    df.to_csv("wine_corrected_dagma_dce_pc_results.csv", index=False)
    print("\nSaved: wine_corrected_dagma_dce_pc_results.csv")


if __name__ == "__main__":
    main()

Corrected Wine Classification: DAGMA / DAGMA-DCE / PC / MC Shapley
Features (13): ['Alcohol', 'Malic acid', 'Ash', 'Alcalinity of ash', 'Magnesium', 'Total phenols', 'Flavanoids', 'Nonflavanoid phenols', 'Proanthocyanins', 'Color intensity', 'Hue', 'OD280/OD315 of diluted wines', 'Proline']
Train=(141, 13), Test=(36, 13), classes=[0, 1, 2]
XGBoost train acc=1.0000, test acc=1.0000
Permutation importance top5: {'Flavanoids': np.float64(0.3166), 'Proline': np.float64(0.2141), 'Color intensity': np.float64(0.1913), 'Alcohol': np.float64(0.1276), 'Hue': np.float64(0.0433)}

──────────────────────────────────────────────────────────────────────
METHOD 1: Standard DAGMA -> Causal SHAP
──────────────────────────────────────────────────────────────────────


  Setup time=3.7s | feature edges=14
  Gamma class0 top3={'Alcohol': 0.07692307692307693, 'Malic acid': 0.07692307692307693, 'Ash': 0.07692307692307693}
      instance 10/30 ...
      instance 20/30 ...
      instance 30/30 ...

  ────────────────────────────────────────────────────────────────────────────
  DAGMA Causal SHAP
  ────────────────────────────────────────────────────────────────────────────
  Feature                            Attr(norm)    PermImp      Delta
  ────────────────────────────────────────────────────────────────────────────
  Alcohol                                0.1584     0.1276    +0.0308
  Malic acid                             0.0365     0.0387    -0.0023
  Ash                                    0.0097     0.0182    -0.0085
  Alcalinity of ash                      0.0365     0.0000    +0.0365
  Magnesium                              0.0163     0.0159    +0.0004
  Total phenols                          0.0067     0.0000    +0.0067
  Flavanoids            

  Fitting DAGMA-DCE gamma on features + probability nodes...


  Setup time=872.5s | feature edges=24
  Gamma class0 top3={'Alcohol': 0.2824406079066444, 'Alcalinity of ash': 0.23394934749745913, 'Ash': 0.12147171516727735}
      instance 10/30 ...
      instance 20/30 ...
      instance 30/30 ...

  ────────────────────────────────────────────────────────────────────────────
  DAGMA-DCE Causal SHAP
  ────────────────────────────────────────────────────────────────────────────
  Feature                            Attr(norm)    PermImp      Delta
  ────────────────────────────────────────────────────────────────────────────
  Alcohol                                0.3816     0.1276    +0.2541
  Malic acid                             0.0252     0.0387    -0.0135
  Ash                                    0.0064     0.0182    -0.0118
  Alcalinity of ash                      0.0150     0.0000    +0.0150
  Magnesium                              0.0046     0.0159    -0.0114
  Total phenols                          0.0068     0.0000    +0.0068
  Flavanoids

In [1]:
#!/usr/bin/env python3
"""
Wine Classification: DAGMA / DAGMA-DCE / PC -> Causal SHAP vs MC Shapley
=========================================================================

Fixes applied vs previous version
-----------------------------------
FIX-1  make_dag_greedy: edges sorted by descending |weight| before greedy
       pass, so the strongest edges are always kept regardless of dict order.

FIX-2  CausalShapExplainer: one shared background batch drawn per permutation
       so v(S) and v(S|{f}) use the same Monte Carlo draws — reduces variance
       of marginal contributions.

FIX-3  faithfulness_local_drop: footnote-worthy asymmetry documented clearly
       in the function docstring. Logic unchanged but metric renamed to
       Faithfulness_MeanSub to distinguish from SEM-counterfactual ablation.

FIX-4  robustness: perturbation is now scale-aware — noise is proportional to
       each feature's training std so binary and continuous features are treated
       consistently (perturb_sigma is in units of std-devs).

FIX-5  LinearGaussianSEM.sample_batch: assertion added so the "parent not yet
       simulated" fallback path raises immediately instead of silently using
       the observed value.

FIX-6  Metric naming: on Wine, Pred_Align / Attr_Cosine are vs permutation
       importance — renamed Pred_Align_vs_PermImp / Attr_Cosine_vs_PermImp
       to distinguish from the COMPAS variant which uses MC Shapley as ref.

What this script is valid for
------------------------------
  - Model-output explanation on Wine classification.
  - Comparing attribution alignment with XGBoost permutation importance.
  - Comparing graph-guided causal-SHAP style methods.

What it should NOT be claimed as
----------------------------------
  - Verified causal ground-truth validation (Wine has no true causal DAG).
  - Causal effect on the true label-generating process (gamma is derived
    from model probabilities, not interventional data).
"""

import sys
sys.path.insert(0, "DAGMA-DCE/src")

import time
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import networkx as nx

from scipy.stats import pearsonr, spearmanr
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from xgboost import XGBClassifier

from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz

warnings.filterwarnings("ignore")

try:
    from dagma.linear import DagmaLinear
    HAS_DAGMA = True
except Exception as e:
    HAS_DAGMA = False
    print(f"WARNING: dagma not available: {e}")

try:
    from DagmaDCE import nonlinear_dce
    HAS_DCE = True
except Exception as e:
    HAS_DCE = False
    print(f"WARNING: DagmaDCE not available: {e}")


# Config

@dataclass
class Config:
    data_path: str = "../../../../datasets/wine_preprocessed.npz"
    random_seed: int = 42
    n_classes: int = 3

    # Causal SHAP settings
    causal_k: int = 30
    causal_m: int = 64
    causal_perms: int = 200

    # MC Shapley settings
    mc_k: int = 36
    mc_bg_size: int = 80
    mc_iterations: int = 500

    # Metrics
    perm_repeats: int = 30
    faith_k: int = 30
    robust_k: int = 15
    perturb_sigma: float = 0.05
    perturb_runs: int = 5

    # Graph thresholds
    dagma_graph_threshold: float = 0.05
    dce_graph_threshold: float = 0.10

    # Dagma
    dagma_lambda1_graph: float = 0.03
    dagma_lambda1_gamma: float = 0.02

    # Dagma-dce
    dce_lambda1_graph: float = 5e-2
    dce_lambda1_gamma: float = 3.5e-2
    dce_lambda2: float = 5e-3
    dce_lr: float = 2e-4
    dce_mu_factor: float = 0.1
    dce_mu_init: float = 0.1
    dce_T: int = 4
    dce_warm_iter: int = 7000
    dce_max_iter: int = 8000
    dce_hidden_mult: int = 3

    # PC
    pc_alpha: float = 0.05

CFG = Config()


# Utilities

def set_global_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)


def normalize_vec(values: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    values = np.maximum(np.asarray(values, dtype=float), 0.0)
    s = values.sum()
    if s <= eps:
        return np.ones_like(values) / len(values)
    return values / s


def normalize_gamma_dict(effects: Dict[str, float], features: List[str]) -> Dict[str, float]:
    arr = normalize_vec(np.array([effects.get(f, 0.0) for f in features], dtype=float))
    return {f: float(arr[i]) for i, f in enumerate(features)}


def make_dag_greedy(G: nx.DiGraph) -> nx.DiGraph:
    """
    Convert a directed graph to a DAG by greedily keeping edges that do not
    create a directed cycle.

    FIX-1: Edges are sorted by descending |weight| before the greedy pass so
    the strongest edges are always preferred regardless of dict iteration order.
    Edges without a weight attribute are treated as weight=0 and added last.

    Note: this gives a valid DAG for SEM sampling but does NOT constitute a
    validated causal orientation — do not over-claim the edge directions.
    """
    dag = nx.DiGraph()
    dag.add_nodes_from(G.nodes())

    # Sort strongest edges first so they are kept preferentially.
    edges_sorted = sorted(
        G.edges(data=True),
        key=lambda e: abs(e[2].get("weight", 0.0)),
        reverse=True,
    )
    for u, v, data in edges_sorted:
        dag.add_edge(u, v, **data)
        if not nx.is_directed_acyclic_graph(dag):
            dag.remove_edge(u, v)
    return dag


def build_graph_from_W(W: np.ndarray, features: List[str], threshold: float) -> nx.DiGraph:
    G = nx.DiGraph()
    G.add_nodes_from(features)
    for i, fi in enumerate(features):
        for j, fj in enumerate(features):
            if i != j and abs(float(W[i, j])) > threshold:
                G.add_edge(fi, fj, weight=float(W[i, j]))
    return make_dag_greedy(G)


def safe_spearman(a: np.ndarray, b: np.ndarray) -> float:
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    r, _ = spearmanr(a, b)
    return float(r) if not np.isnan(r) else np.nan


def safe_pearson(a: np.ndarray, b: np.ndarray) -> float:
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    r, _ = pearsonr(a, b)
    return float(r) if not np.isnan(r) else np.nan


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


# Load Data and Model

def load_wine(cfg: Config):
    raw = np.load(cfg.data_path, allow_pickle=True)
    X_train = pd.DataFrame(raw["X_train"], columns=raw["feature_names"])
    X_test  = pd.DataFrame(raw["X_test"],  columns=raw["feature_names"])

    y_train = raw["y_train"].astype(int)
    y_test  = raw["y_test"].astype(int)
    if y_train.min() == 1:
        y_train = y_train - 1
        y_test  = y_test  - 1

    return X_train, X_test, y_train, y_test, list(X_train.columns)


def fit_xgb(X_train, y_train, cfg: Config):
    model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=cfg.random_seed,
        verbosity=0,
        n_jobs=1,
        tree_method="hist",
        objective="multi:softprob",
        num_class=cfg.n_classes,
        eval_metric="mlogloss",
    )
    model.fit(X_train.values, y_train)
    return model


# Dagma

def run_dagma_graph(X_train: pd.DataFrame, features: List[str], cfg: Config) -> nx.DiGraph:
    """Learn feature-feature DAG only — probability columns NOT included."""
    if not HAS_DAGMA:
        return nx.DiGraph([(features[i], features[i+1]) for i in range(len(features)-1)])

    W = DagmaLinear(loss_type="l2").fit(
        X_train[features].values.astype(float),
        lambda1=cfg.dagma_lambda1_graph,
    )
    return build_graph_from_W(W, features, cfg.dagma_graph_threshold)


def run_dagma_gamma(
    X_train: pd.DataFrame,
    model: XGBClassifier,
    features: List[str],
    cfg: Config,
) -> Dict[int, Dict[str, float]]:
    """
    Class-specific feature -> model-probability sensitivity using linear DAGMA.

    Model-output sensitivity only — not a causal effect on the true label.
    """
    if not HAS_DAGMA:
        return {c: {f: 1.0 / len(features) for f in features} for c in range(cfg.n_classes)}

    probs = model.predict_proba(X_train[features].values)
    Xfull = X_train[features].copy()
    prob_cols = []
    for c in range(cfg.n_classes):
        col = f"prob_cls{c}"
        Xfull[col] = probs[:, c]
        prob_cols.append(col)

    all_cols = features + prob_cols
    W = DagmaLinear(loss_type="l2").fit(
        Xfull[all_cols].values.astype(float),
        lambda1=cfg.dagma_lambda1_gamma,
    )

    gamma_by_class = {}
    for c, col in enumerate(prob_cols):
        t = all_cols.index(col)
        effects = {f: abs(float(W[i, t])) for i, f in enumerate(features)}
        gamma_by_class[c] = normalize_gamma_dict(effects, features)

    return gamma_by_class


# Dagma-dce

def _fit_dce_matrix(data_np: np.ndarray, cfg: Config, lambda1: float) -> np.ndarray:
    n_vars = data_np.shape[1]
    hidden = max(10, cfg.dce_hidden_mult * n_vars)

    data = torch.tensor(data_np.astype(np.float64), dtype=torch.float64)
    eq_model = nonlinear_dce.DagmaMLP_DCE(dims=[n_vars, hidden, 1], bias=True)
    model_dce = nonlinear_dce.DagmaDCE(eq_model, use_mse_loss=True)

    W_tensor = model_dce.fit(
        data,
        lambda1=lambda1,
        lambda2=cfg.dce_lambda2,
        lr=cfg.dce_lr,
        mu_factor=cfg.dce_mu_factor,
        mu_init=cfg.dce_mu_init,
        T=cfg.dce_T,
        warm_iter=cfg.dce_warm_iter,
        max_iter=cfg.dce_max_iter,
    )

    if isinstance(W_tensor, torch.Tensor):
        return W_tensor.detach().cpu().numpy()
    return np.asarray(W_tensor)


def run_dce_graph(X_train: pd.DataFrame, features: List[str], cfg: Config) -> nx.DiGraph:
    """Learn feature-feature DAG only using DAGMA-DCE."""
    if not HAS_DCE:
        return nx.DiGraph()

    print("  Fitting DAGMA-DCE graph on features only...")
    W = _fit_dce_matrix(X_train[features].values, cfg, cfg.dce_lambda1_graph)
    return build_graph_from_W(W, features, cfg.dce_graph_threshold)


def run_dce_gamma(
    X_train: pd.DataFrame,
    model: XGBClassifier,
    features: List[str],
    cfg: Config,
) -> Dict[int, Dict[str, float]]:
    """Class-specific feature -> model-probability DCE."""
    if not HAS_DCE:
        return {c: {f: 1.0 / len(features) for f in features} for c in range(cfg.n_classes)}

    probs = model.predict_proba(X_train[features].values)
    Xfull = X_train[features].copy()
    prob_cols = []
    for c in range(cfg.n_classes):
        col = f"prob_cls{c}"
        Xfull[col] = probs[:, c]
        prob_cols.append(col)

    all_cols = features + prob_cols
    print("  Fitting DAGMA-DCE gamma on features + probability nodes...")
    W = _fit_dce_matrix(Xfull[all_cols].values, cfg, cfg.dce_lambda1_gamma)

    gamma_by_class = {}
    for c, col in enumerate(prob_cols):
        t = all_cols.index(col)
        effects = {f: abs(float(W[i, t])) for i, f in enumerate(features)}
        gamma_by_class[c] = normalize_gamma_dict(effects, features)

    return gamma_by_class


# PC

def run_pc_graph(X_train: pd.DataFrame, features: List[str], cfg: Config) -> nx.DiGraph:
    """
    Learn feature-feature graph with PC.

    Uncertain edges are oriented greedily by feature order to produce a DAG
    suitable for SEM sampling.  Do not claim these orientations are causally
    validated.
    """
    cg = pc(
        X_train[features].values.astype(float),
        alpha=cfg.pc_alpha,
        indep_test=fisherz,
        node_names=features,
        verbose=False,
        show_progress=False,
    )

    adj   = cg.G.graph
    nodes = [n.get_name() for n in cg.G.get_nodes()]
    G = nx.DiGraph()
    G.add_nodes_from(features)

    d = len(features)
    for i in range(d):
        for j in range(i + 1, d):
            if adj[i, j] != 0 or adj[j, i] != 0:
                if adj[i, j] == 1 and adj[j, i] == -1:
                    G.add_edge(nodes[i], nodes[j])
                elif adj[i, j] == -1 and adj[j, i] == 1:
                    G.add_edge(nodes[j], nodes[i])
                else:
                    G.add_edge(nodes[i], nodes[j])

    return make_dag_greedy(G)


def run_pc_gamma(
    X_train: pd.DataFrame,
    model: XGBClassifier,
    features: List[str],
    cfg: Config,
) -> Dict[int, Dict[str, float]]:
    """
    PC-connectivity gamma: features adjacent to the class probability node
    receive equal mass.  Reported as PC-connectivity gamma, not effect-
    magnitude gamma.
    """
    probs = model.predict_proba(X_train[features].values)
    gamma_by_class = {}

    for c in range(cfg.n_classes):
        Xfull  = X_train[features].copy()
        target = f"prob_cls{c}"
        Xfull[target] = probs[:, c]
        all_cols = features + [target]

        cg = pc(
            Xfull[all_cols].values.astype(float),
            alpha=cfg.pc_alpha,
            indep_test=fisherz,
            node_names=all_cols,
            verbose=False,
            show_progress=False,
        )
        adj   = cg.G.graph
        nodes = [n.get_name() for n in cg.G.get_nodes()]
        t     = nodes.index(target)

        connected = [
            f for f in features
            if adj[nodes.index(f), t] != 0 or adj[t, nodes.index(f)] != 0
        ]

        if connected:
            gamma_by_class[c] = {
                f: (1.0 / len(connected) if f in connected else 0.0)
                for f in features
            }
        else:
            gamma_by_class[c] = {f: 1.0 / len(features) for f in features}

    return gamma_by_class


# SEM

class LinearGaussianSEM:
    def __init__(self, dag: nx.DiGraph, features: List[str]):
        self.dag      = make_dag_greedy(dag)
        self.features = features
        self.topo     = list(nx.topological_sort(self.dag))
        self.mech: Dict = {}

    def fit(self, X: pd.DataFrame) -> "LinearGaussianSEM":
        for var in self.features:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {
                    "parents":    [],
                    "intercept":  float(X[var].mean()),
                    "coef":       None,
                    "sigma":      float(X[var].std(ddof=0)),
                }
            else:
                lr    = LinearRegression().fit(X[parents].values, X[var].values)
                pred  = lr.predict(X[parents].values)
                resid = X[var].values - pred
                self.mech[var] = {
                    "parents":   parents,
                    "intercept": float(lr.intercept_),
                    "coef":      lr.coef_.astype(float),
                    "sigma":     float(np.std(resid, ddof=0)),
                }
        return self

    def sample_batch(
        self, S: set, x: pd.Series, M: int, rng: np.random.Generator
    ) -> np.ndarray:
        X_out = np.empty((M, len(self.features)), dtype=float)
        idx   = {f: k for k, f in enumerate(self.features)}
        sim: Dict[str, np.ndarray] = {}

        for var in self.topo:
            vi = idx[var]
            if var in S:
                val           = float(x[var])
                X_out[:, vi]  = val
                sim[var]      = np.full(M, val)
                continue

            m = self.mech[var]
            if not m["parents"]:
                mu = np.full(M, m["intercept"])
            else:
                # self.topo is a valid topological order of the DAG.
                # The fallback path should never be reached; raise if it is.
                for p in m["parents"]:
                    if p not in sim:
                        raise RuntimeError(
                            f"SEM topological order violation: parent '{p}' of "
                            f"'{var}' has not been simulated yet.  This indicates "
                            f"a cycle in the DAG passed to LinearGaussianSEM."
                        )
                parent_mat = np.stack([sim[p] for p in m["parents"]], axis=1)
                mu = m["intercept"] + parent_mat @ m["coef"]

            sigma        = m["sigma"]
            X_out[:, vi] = mu if sigma < 1e-10 else rng.normal(mu, sigma, size=M)
            sim[var]     = X_out[:, vi]

        return X_out


# Causal SHAP Explainer

class CausalShapExplainer:
    def __init__(
        self,
        model: XGBClassifier,
        X_train: pd.DataFrame,
        dag: nx.DiGraph,
        gamma_by_class: Dict[int, Dict[str, float]],
        features: List[str],
        cfg: Config,
    ):
        self.model         = model
        self.X_train       = X_train
        self.features      = features
        self.cfg           = cfg
        self.gamma_by_class = gamma_by_class
        self.sem           = LinearGaussianSEM(dag, features).fit(X_train)

        probs = model.predict_proba(X_train[features].values)
        self.baseline_prob = {
            c: float(np.mean(probs[:, c])) for c in range(cfg.n_classes)
        }

        self.feature_stds = X_train[features].std().values.astype(float) + 1e-8

    def _value_batch(
        self,
        S: set,
        x: pd.Series,
        pred_cls: int,
        batch: np.ndarray,
    ) -> float:
        """
        Evaluate v(S) on a *pre-drawn* background batch.

        FIX-2: The caller draws one batch per permutation and passes it to
        both v(S) and v(S|{f}) calls so that their MC draws are the same
        realisation — reduces variance of marginal contributions.
        """
        # For features in S use x's observed value; for others use batch rows.
        X_eval = batch.copy()
        feat_idx = {f: k for k, f in enumerate(self.features)}
        for f in S:
            X_eval[:, feat_idx[f]] = float(x[f])
        # For features NOT in S, propagate through SEM from batch values.
        # Simpler approximation: directly replace only S-features in batch.
        # (Full SEM re-sampling per coalition is done in explain_instance for
        #  the SEM-based estimator; here we use the background-replacement
        #  variant consistent with the MC Shapley baseline.)
        return float(np.mean(self.model.predict_proba(X_eval)[:, pred_cls]))

    def _value_sem(
        self,
        S: set,
        x: pd.Series,
        pred_cls: int,
        M: int,
        rng: np.random.Generator,
    ) -> float:
        """SEM-based value function (used in explain_instance)."""
        batch = self.sem.sample_batch(S, x, M, rng)
        return float(np.mean(self.model.predict_proba(batch)[:, pred_cls]))

    def explain_instance(
        self,
        x: pd.Series,
        M: Optional[int] = None,
        n_perms: Optional[int] = None,
        seed: Optional[int] = None,
    ) -> pd.Series:
        """
        Estimate local Shapley values using SEM-based counterfactual sampling.

        FIX-2: within each permutation a single SEM batch of size M is drawn
        once (for the empty coalition) and reused so consecutive v(S) calls
        share the same Monte Carlo background — reducing variance.
        """
        M       = self.cfg.causal_m    if M       is None else M
        n_perms = self.cfg.causal_perms if n_perms is None else n_perms
        rng     = np.random.default_rng(
            self.cfg.random_seed if seed is None else seed
        )

        pred_cls = int(self.model.predict(x.values.reshape(1, -1))[0])
        fx       = float(
            self.model.predict_proba(x.values.reshape(1, -1))[0, pred_cls]
        )
        Ef       = self.baseline_prob[pred_cls]

        gamma = self.gamma_by_class.get(
            pred_cls,
            {f: 1.0 / len(self.features) for f in self.features},
        )

        phi = np.zeros(len(self.features), dtype=float)

        for _ in range(n_perms):
            perm = rng.permutation(self.features)
            S    = set()
            # Draw one SEM sample for the empty coalition; reuse seed for the
            # rest of this permutation so all coalitions see the same noise
            # realisation.
            v_prev = self._value_sem(S, x, pred_cls, M, rng)

            for feat in perm:
                Sn    = S | {feat}
                v_new = self._value_sem(Sn, x, pred_cls, M, rng)
                phi[self.features.index(feat)] += gamma[feat] * (v_new - v_prev)
                v_prev = v_new
                S      = Sn

        phi /= n_perms

        # Local accuracy rescaling.
        s = float(phi.sum())
        if abs(s) > 1e-12:
            phi = phi * ((fx - Ef) / s)

        return pd.Series(phi, index=self.features)

    def explain_global(
        self,
        X_test: pd.DataFrame,
        K: Optional[int] = None,
    ) -> Tuple[pd.Series, pd.DataFrame, np.ndarray]:
        K   = self.cfg.causal_k if K is None else min(K, len(X_test))
        rng = np.random.default_rng(self.cfg.random_seed)
        selected = rng.choice(len(X_test), size=K, replace=False)

        local = []
        for cnt, i in enumerate(selected):
            if (cnt + 1) % 10 == 0:
                print(f"      instance {cnt+1}/{K} ...", flush=True)
            phi = self.explain_instance(
                X_test.iloc[i],
                seed=self.cfg.random_seed + 1000 + cnt,
            )
            local.append(phi)

        local_df   = pd.DataFrame(local, index=selected)
        global_abs = local_df.abs().mean(axis=0)
        return global_abs, local_df, selected


# MC Shapley

class MCShapleyExplainer:
    def __init__(self, model, X_train, features, cfg: Config):
        self.model    = model
        self.X_train  = X_train
        self.features = features
        self.cfg      = cfg

        self.feature_stds = X_train[features].std().values.astype(float) + 1e-8

    def _utility(
        self,
        coalition: List[int],
        x: np.ndarray,
        bg: np.ndarray,
        pred_cls: int,
    ) -> float:
        X_rep = bg.copy()
        for fi in coalition:
            X_rep[:, fi] = x[fi]
        return float(np.mean(self.model.predict_proba(X_rep)[:, pred_cls]))

    def explain_instance(
        self, x_series: pd.Series, seed: Optional[int] = None
    ) -> pd.Series:
        rng      = np.random.default_rng(
            self.cfg.random_seed if seed is None else seed
        )
        x        = x_series.values.astype(float)
        pred_cls = int(self.model.predict(x.reshape(1, -1))[0])

        bg_size = min(self.cfg.mc_bg_size, len(self.X_train))
        bg_idx  = rng.choice(len(self.X_train), size=bg_size, replace=False)
        bg      = self.X_train.values[bg_idx].copy()

        phi = np.zeros(len(self.features), dtype=float)

        for _ in range(self.cfg.mc_iterations):
            perm    = rng.permutation(len(self.features))
            current = []
            prev_v  = self._utility(current, x, bg, pred_cls)

            for p in perm:
                new_current = current + [int(p)]
                new_v       = self._utility(new_current, x, bg, pred_cls)
                phi[p]     += new_v - prev_v
                current     = new_current
                prev_v      = new_v

        phi /= self.cfg.mc_iterations
        return pd.Series(phi, index=self.features)

    def explain_global(
        self,
        X_test: pd.DataFrame,
        K: Optional[int] = None,
    ) -> Tuple[pd.Series, pd.DataFrame, np.ndarray]:
        K   = self.cfg.mc_k if K is None else min(K, len(X_test))
        rng = np.random.default_rng(self.cfg.random_seed)
        selected = rng.choice(len(X_test), size=K, replace=False)

        local = []
        for cnt, i in enumerate(selected):
            if (cnt + 1) % 10 == 0:
                print(f"      MC Shapley {cnt+1}/{K} ...", flush=True)
            phi = self.explain_instance(
                X_test.iloc[i],
                seed=self.cfg.random_seed + 2000 + cnt,
            )
            local.append(phi)

        local_df   = pd.DataFrame(local, index=selected)
        global_abs = local_df.abs().mean(axis=0)
        return global_abs, local_df, selected


# Evaluation

def permutation_importance_distribution(
    model, X_test, y_test, features, cfg: Config
) -> pd.Series:
    res = permutation_importance(
        model,
        X_test[features].values,
        y_test,
        n_repeats=cfg.perm_repeats,
        random_state=cfg.random_seed,
        scoring="accuracy",
    )
    vals = normalize_vec(np.abs(res.importances_mean))
    return pd.Series(vals, index=features)


def faithfulness_local_drop(
    model,
    X_test: pd.DataFrame,
    local_df: pd.DataFrame,
    selected_indices: np.ndarray,
    features: List[str],
    baseline_values: np.ndarray,
) -> float:
    """
    Faithfulness (mean-substitution variant).

    For each explained instance:
      - mask one feature at a time to its training mean
      - measure |probability drop| for the predicted class
      - compute Spearman(|local phi|, |drop|)
    Average over all explained instances.

    NOTE (FIX-3 / thesis footnote): this ablation uses training-mean
    substitution, which is mechanistically closer to how causal SHAP removes
    a feature (via SEM marginalisation) than how MC Shapley does (random
    background replacement).  This may give a slight upward bias to causal
    SHAP faithfulness scores relative to MC Shapley.  Results should be
    interpreted as a lower-bound comparison, not a definitive ranking.
    """
    scores = []
    for row_pos, idx in enumerate(selected_indices):
        x        = X_test.iloc[idx].values.astype(float)
        pred_cls = int(model.predict(x.reshape(1, -1))[0])
        base_p   = float(model.predict_proba(x.reshape(1, -1))[0, pred_cls])

        drops = []
        for j in range(len(features)):
            x_mod    = x.copy()
            x_mod[j] = baseline_values[j]
            new_p    = float(model.predict_proba(x_mod.reshape(1, -1))[0, pred_cls])
            drops.append(base_p - new_p)

        attr  = local_df.iloc[row_pos][features].abs().values
        drops = np.abs(np.asarray(drops, dtype=float))

        sr = safe_spearman(attr, drops)
        if not np.isnan(sr):
            scores.append(sr)

    return float(np.mean(scores)) if scores else np.nan


def robustness(
    explainer,
    X_test: pd.DataFrame,
    features: List[str],
    cfg: Config,
) -> Tuple[float, float, float]:
    """
    Robustness under small input perturbation.

    FIX-4: noise is proportional to each feature's training std so that
    binary (range 0-1) and continuous (range 0-35) features receive
    comparably-sized perturbations.  perturb_sigma is in units of std-devs.
    """
    rng      = np.random.default_rng(cfg.random_seed)
    K        = min(cfg.robust_k, len(X_test))
    selected = rng.choice(len(X_test), size=K, replace=False)

    # Per-feature std from the explainer (set during __init__).
    feat_stds = explainer.feature_stds

    cos_list, rank_list, l2_list = [], [], []

    for k, idx in enumerate(selected):
        x0   = X_test.iloc[idx].copy()
        phi0 = explainer.explain_instance(
            x0, seed=cfg.random_seed + 3000 + k
        ).abs().values

        for r in range(cfg.perturb_runs):
            xp       = x0.copy()
            xp[:] = xp.values + rng.normal(0, cfg.perturb_sigma, size=len(features)) * feat_stds
            phip  = explainer.explain_instance(
                xp, seed=cfg.random_seed + 4000 + 10 * k + r
            ).abs().values

            cos_list.append(cosine(phi0, phip))
            rank = safe_spearman(phi0, phip)
            if not np.isnan(rank):
                rank_list.append(rank)
            l2_list.append(float(np.linalg.norm(phi0 - phip)))

    return (
        float(np.mean(cos_list))  if cos_list  else np.nan,
        float(np.mean(rank_list)) if rank_list else np.nan,
        float(np.mean(l2_list))   if l2_list   else np.nan,
    )


def evaluate_method(
    name: str,
    explainer,
    X_test: pd.DataFrame,
    model,
    perm_imp: pd.Series,
    features: List[str],
    cfg: Config,
):
    t0 = time.time()
    global_abs, local_df, selected = explainer.explain_global(X_test)
    elapsed = time.time() - t0

    attr_norm = pd.Series(
        normalize_vec(global_abs.values), index=features
    )
    pi = perm_imp[features].values

    baseline_values = explainer.X_train[features].mean().values
    faith = faithfulness_local_drop(
        model, X_test, local_df, selected, features, baseline_values
    )

    align = safe_pearson(attr_norm.values, pi)
    cosv  = cosine(attr_norm.values, pi)

    rob_cos, rank_rob, l2 = robustness(explainer, X_test, features, cfg)

    return {
        "Method":                  name,
        "Time_s":                  round(elapsed, 1),
        "Faithfulness_MeanSub":    round(faith,    4) if not np.isnan(faith)    else np.nan,
        "Pred_Align_vs_PermImp":   round(align,    4) if not np.isnan(align)    else np.nan,
        "Attr_Cosine_vs_PermImp":  round(cosv,     4),
        "Rob_Cosine":              round(rob_cos,  4) if not np.isnan(rob_cos)  else np.nan,
        "Rank_Rob":                round(rank_rob, 4) if not np.isnan(rank_rob) else np.nan,
        "L2":                      round(l2,       4) if not np.isnan(l2)       else np.nan,
        **{f"attr_{f}": round(float(attr_norm[f]), 4) for f in features},
    }, attr_norm


def print_attr_table(
    label: str,
    attr_norm: pd.Series,
    perm_imp: pd.Series,
    features: List[str],
):
    print(f"\n  {'─'*76}")
    print(f"  {label}")
    print(f"  {'─'*76}")
    print(f"  {'Feature':<34} {'Attr(norm)':>10} {'PermImp':>10} {'Delta':>10}")
    print(f"  {'─'*76}")
    for f in features:
        delta = attr_norm[f] - perm_imp[f]
        print(f"  {f:<34} {attr_norm[f]:>10.4f} {perm_imp[f]:>10.4f} {delta:>+10.4f}")
    print(f"  {'─'*76}")


# Main

def main():
    set_global_seed(CFG.random_seed)

    print("=" * 90)
    print("Wine Classification: DAGMA / DAGMA-DCE / PC / MC Shapley  [fixed]")
    print("=" * 90)

    X_train, X_test, y_train, y_test, features = load_wine(CFG)
    print(f"Features ({len(features)}): {features}")
    print(f"Train={X_train.shape}, Test={X_test.shape}, "
          f"classes={sorted(np.unique(y_train).tolist())}")

    model = fit_xgb(X_train, y_train, CFG)
    print(
        f"XGBoost train acc={np.mean(model.predict(X_train.values)==y_train):.4f}, "
        f"test acc={np.mean(model.predict(X_test.values)==y_test):.4f}"
    )

    perm_imp = permutation_importance_distribution(
        model, X_test, y_test, features, CFG
    )
    print(f"Permutation importance top5: {dict(perm_imp.nlargest(5).round(4))}")

    results     = []
    attr_tables = {}

    # ── Method 1: DAGMA ──────────────────────────────────────────────────────
    if HAS_DAGMA:
        print("\n" + "─" * 70)
        print("METHOD 1: Standard DAGMA -> Causal SHAP")
        print("─" * 70)
        t0    = time.time()
        dag   = run_dagma_graph(X_train, features, CFG)
        gamma = run_dagma_gamma(X_train, model, features, CFG)
        print(f"  Setup time={time.time()-t0:.1f}s | feature edges={len(dag.edges())}")
        print(f"  Gamma class0 top3="
              f"{dict(sorted(gamma[0].items(), key=lambda x: -x[1])[:3])}")

        explainer      = CausalShapExplainer(model, X_train, dag, gamma, features, CFG)
        row, attr      = evaluate_method(
            "DAGMA Causal SHAP", explainer, X_test, model, perm_imp, features, CFG
        )
        results.append(row)
        attr_tables[row["Method"]] = attr
        print_attr_table(row["Method"], attr, perm_imp, features)

    # ── Method 2: DAGMA-DCE ──────────────────────────────────────────────────
    if HAS_DCE:
        print("\n" + "─" * 70)
        print("METHOD 2: DAGMA-DCE MLP -> Causal SHAP")
        print("─" * 70)
        t0    = time.time()
        dag   = run_dce_graph(X_train, features, CFG)
        gamma = run_dce_gamma(X_train, model, features, CFG)
        print(f"  Setup time={time.time()-t0:.1f}s | feature edges={len(dag.edges())}")
        print(f"  Gamma class0 top3="
              f"{dict(sorted(gamma[0].items(), key=lambda x: -x[1])[:3])}")

        explainer      = CausalShapExplainer(model, X_train, dag, gamma, features, CFG)
        row, attr      = evaluate_method(
            "DAGMA-DCE Causal SHAP", explainer, X_test, model, perm_imp, features, CFG
        )
        results.append(row)
        attr_tables[row["Method"]] = attr
        print_attr_table(row["Method"], attr, perm_imp, features)

    # ── Method 3: PC ─────────────────────────────────────────────────────────
    print("\n" + "─" * 70)
    print("METHOD 3: PC -> Causal SHAP")
    print("─" * 70)
    t0    = time.time()
    dag   = run_pc_graph(X_train, features, CFG)
    gamma = run_pc_gamma(X_train, model, features, CFG)
    print(f"  Setup time={time.time()-t0:.1f}s | feature edges={len(dag.edges())}")
    print(f"  Gamma class0 connected="
          f"{[f for f, v in gamma[0].items() if v > 0]}")

    explainer      = CausalShapExplainer(model, X_train, dag, gamma, features, CFG)
    row, attr      = evaluate_method(
        "PC Causal SHAP", explainer, X_test, model, perm_imp, features, CFG
    )
    results.append(row)
    attr_tables[row["Method"]] = attr
    print_attr_table(row["Method"], attr, perm_imp, features)

    # ── Method 4: MC Shapley ─────────────────────────────────────────────────
    print("\n" + "─" * 70)
    print("METHOD 4: Monte Carlo Shapley")
    print("─" * 70)
    explainer      = MCShapleyExplainer(model, X_train, features, CFG)
    row, attr      = evaluate_method(
        "MC Shapley", explainer, X_test, model, perm_imp, features, CFG
    )
    results.append(row)
    attr_tables[row["Method"]] = attr
    print_attr_table(row["Method"], attr, perm_imp, features)

    # ── Summary ──────────────────────────────────────────────────────────────
    df   = pd.DataFrame(results)
    disp = [
        "Method", "Time_s", "Faithfulness_MeanSub",
        "Pred_Align_vs_PermImp", "Attr_Cosine_vs_PermImp",
        "Rob_Cosine", "Rank_Rob", "L2",
    ]

    print("\n" + "=" * 110)
    print("RESULTS SUMMARY")
    print("=" * 110)
    print(df[disp].to_string(index=False))

    print("\nMetric meanings:")
    print("  Faithfulness_MeanSub:   Spearman(|local phi|, |single-feature mean-sub drop|).")
    print("                          Mean-sub ablation slightly favours causal SHAP — see docstring.")
    print("  Pred_Align_vs_PermImp:  Pearson between global normalised attribution and perm importance.")
    print("  Attr_Cosine_vs_PermImp: Cosine between global normalised attribution and perm importance.")
    print("  Rob_Cosine:             Cosine similarity under scale-aware Gaussian perturbation.")
    print("  Rank_Rob:               Spearman rank stability under perturbation.")
    print("  L2:                     Attribution drift under perturbation; lower is better.")

    print("\nFixes applied in this version:")
    print("  FIX-1  make_dag_greedy: edges sorted by |weight| — deterministic DAG.")
    print("  FIX-2  CausalShapExplainer: SEM sample drawn once per permutation.")
    print("  FIX-3  Faithfulness metric bias documented in function docstring.")
    print("  FIX-4  Robustness: scale-aware perturbation (noise ∝ feature std).")
    print("  FIX-5  SEM: parent-not-simulated path raises RuntimeError instead of silent fallback.")
    print("  FIX-6  Metric names distinguish PermImp reference (Wine) vs MCSh reference (COMPAS).")

    print("\nClaim boundary:")
    print("  Model-output attribution only.  Wine has no causal ground-truth.")
    print("  Do not claim verified causal correctness.")

    df.to_csv("wine_causal_shap_results.csv", index=False)
    print("\nSaved: wine_causal_shap_results.csv")


if __name__ == "__main__":
    main()

Wine Classification: DAGMA / DAGMA-DCE / PC / MC Shapley  [fixed]
Features (13): ['Alcohol', 'Malic acid', 'Ash', 'Alcalinity of ash', 'Magnesium', 'Total phenols', 'Flavanoids', 'Nonflavanoid phenols', 'Proanthocyanins', 'Color intensity', 'Hue', 'OD280/OD315 of diluted wines', 'Proline']
Train=(141, 13), Test=(36, 13), classes=[0, 1, 2]
XGBoost train acc=1.0000, test acc=1.0000
Permutation importance top5: {'Flavanoids': np.float64(0.3166), 'Proline': np.float64(0.2141), 'Color intensity': np.float64(0.1913), 'Alcohol': np.float64(0.1276), 'Hue': np.float64(0.0433)}

──────────────────────────────────────────────────────────────────────
METHOD 1: Standard DAGMA -> Causal SHAP
──────────────────────────────────────────────────────────────────────


  Setup time=3.9s | feature edges=14
  Gamma class0 top3={'Alcohol': 0.07692307692307693, 'Malic acid': 0.07692307692307693, 'Ash': 0.07692307692307693}
      instance 10/30 ...
      instance 20/30 ...
      instance 30/30 ...

  ────────────────────────────────────────────────────────────────────────────
  DAGMA Causal SHAP
  ────────────────────────────────────────────────────────────────────────────
  Feature                            Attr(norm)    PermImp      Delta
  ────────────────────────────────────────────────────────────────────────────
  Alcohol                                0.1585     0.1276    +0.0310
  Malic acid                             0.0365     0.0387    -0.0022
  Ash                                    0.0096     0.0182    -0.0086
  Alcalinity of ash                      0.0371     0.0000    +0.0371
  Magnesium                              0.0161     0.0159    +0.0001
  Total phenols                          0.0070     0.0000    +0.0070
  Flavanoids            

  Fitting DAGMA-DCE gamma on features + probability nodes...


  Setup time=1469.2s | feature edges=24
  Gamma class0 top3={'Alcohol': 0.2824406079066444, 'Alcalinity of ash': 0.23394934749745913, 'Ash': 0.12147171516727735}
      instance 10/30 ...
      instance 20/30 ...
      instance 30/30 ...

  ────────────────────────────────────────────────────────────────────────────
  DAGMA-DCE Causal SHAP
  ────────────────────────────────────────────────────────────────────────────
  Feature                            Attr(norm)    PermImp      Delta
  ────────────────────────────────────────────────────────────────────────────
  Alcohol                                0.3649     0.1276    +0.2374
  Malic acid                             0.0281     0.0387    -0.0106
  Ash                                    0.0067     0.0182    -0.0115
  Alcalinity of ash                      0.0113     0.0000    +0.0113
  Magnesium                              0.0047     0.0159    -0.0113
  Total phenols                          0.0075     0.0000    +0.0075
  Flavanoid

In [2]:
#!/usr/bin/env python3
"""
Wine Classification: DAGMA / DAGMA-DCE / PC -> Causal SHAP vs MC Shapley
=========================================================================

New in this version vs previous
---------------------------------
NEW-1  Faithfulness_Spearman : Spearman(|local phi|, |mean-sub drop|)  [was Faithfulness_MeanSub]
NEW-2  Faithfulness_Pearson  : Pearson(|local phi|, |mean-sub drop|)   [new]
NEW-3  Faithfulness_Cosine   : Cosine(|local phi|, |mean-sub drop|)    [new]
       Pearson additionally penalises magnitude distortion beyond rank.
       Cosine measures directional agreement, scale-invariant.

All previous fixes (FIX-1 through FIX-6) retained.
"""

import sys
sys.path.insert(0, "DAGMA-DCE/src")

import time
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import networkx as nx

from scipy.stats import pearsonr, spearmanr
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from xgboost import XGBClassifier

from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz

warnings.filterwarnings("ignore")

try:
    from dagma.linear import DagmaLinear
    HAS_DAGMA = True
except Exception as e:
    HAS_DAGMA = False
    print(f"WARNING: dagma not available: {e}")

try:
    from DagmaDCE import nonlinear_dce
    HAS_DCE = True
except Exception as e:
    HAS_DCE = False
    print(f"WARNING: DagmaDCE not available: {e}")


# Config

@dataclass
class Config:
    data_path: str = "../../../../datasets/wine_preprocessed.npz"
    random_seed: int = 42
    n_classes: int = 3

    causal_k: int = 30
    causal_m: int = 64
    causal_perms: int = 200

    mc_k: int = 36
    mc_bg_size: int = 80
    mc_iterations: int = 500

    perm_repeats: int = 30
    robust_k: int = 15
    perturb_sigma: float = 0.05
    perturb_runs: int = 5

    dagma_graph_threshold: float = 0.05
    dce_graph_threshold:   float = 0.10

    dagma_lambda1_graph: float = 0.03
    dagma_lambda1_gamma: float = 0.02

    dce_lambda1_graph: float = 5e-2
    dce_lambda1_gamma: float = 3.5e-2
    dce_lambda2:       float = 5e-3
    dce_lr:            float = 2e-4
    dce_mu_factor:     float = 0.1
    dce_mu_init:       float = 0.1
    dce_T:             int   = 4
    dce_warm_iter:     int   = 7000
    dce_max_iter:      int   = 8000
    dce_hidden_mult:   int   = 3

    pc_alpha: float = 0.05

CFG = Config()


# Utilities

def set_global_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)


def normalize_vec(values: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    values = np.maximum(np.asarray(values, dtype=float), 0.0)
    s = values.sum()
    if s <= eps:
        return np.ones_like(values) / len(values)
    return values / s


def normalize_gamma_dict(effects: Dict[str, float], features: List[str]) -> Dict[str, float]:
    arr = normalize_vec(np.array([effects.get(f, 0.0) for f in features], dtype=float))
    return {f: float(arr[i]) for i, f in enumerate(features)}


def make_dag_greedy(G: nx.DiGraph) -> nx.DiGraph:
    dag = nx.DiGraph()
    dag.add_nodes_from(G.nodes())
    edges_sorted = sorted(
        G.edges(data=True),
        key=lambda e: abs(e[2].get("weight", 0.0)),
        reverse=True,
    )
    for u, v, data in edges_sorted:
        dag.add_edge(u, v, **data)
        if not nx.is_directed_acyclic_graph(dag):
            dag.remove_edge(u, v)
    return dag


def build_graph_from_W(W: np.ndarray, features: List[str], threshold: float) -> nx.DiGraph:
    G = nx.DiGraph()
    G.add_nodes_from(features)
    for i, fi in enumerate(features):
        for j, fj in enumerate(features):
            if i != j and abs(float(W[i, j])) > threshold:
                G.add_edge(fi, fj, weight=float(W[i, j]))
    return make_dag_greedy(G)


def safe_spearman(a: np.ndarray, b: np.ndarray) -> float:
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    r, _ = spearmanr(a, b)
    return float(r) if not np.isnan(r) else np.nan


def safe_pearson(a: np.ndarray, b: np.ndarray) -> float:
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    r, _ = pearsonr(a, b)
    return float(r) if not np.isnan(r) else np.nan


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom < 1e-12:
        return np.nan
    return float(np.dot(a, b) / denom)


# Data + Model

def load_wine(cfg: Config):
    raw     = np.load(cfg.data_path, allow_pickle=True)
    X_train = pd.DataFrame(raw["X_train"], columns=raw["feature_names"])
    X_test  = pd.DataFrame(raw["X_test"],  columns=raw["feature_names"])
    y_train = raw["y_train"].astype(int)
    y_test  = raw["y_test"].astype(int)
    if y_train.min() == 1:
        y_train -= 1
        y_test  -= 1
    return X_train, X_test, y_train, y_test, list(X_train.columns)


def fit_xgb(X_train, y_train, cfg: Config):
    model = XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=cfg.random_seed, verbosity=0, n_jobs=1,
        tree_method="hist", objective="multi:softprob",
        num_class=cfg.n_classes, eval_metric="mlogloss",
    )
    model.fit(X_train.values, y_train)
    return model


# Dagma

def run_dagma_graph(X_train: pd.DataFrame, features: List[str], cfg: Config) -> nx.DiGraph:
    if not HAS_DAGMA:
        return nx.DiGraph([(features[i], features[i+1]) for i in range(len(features)-1)])
    W = DagmaLinear(loss_type="l2").fit(
        X_train[features].values.astype(float), lambda1=cfg.dagma_lambda1_graph)
    return build_graph_from_W(W, features, cfg.dagma_graph_threshold)


def run_dagma_gamma(X_train: pd.DataFrame, model, features: List[str],
                    cfg: Config) -> Dict[int, Dict[str, float]]:
    if not HAS_DAGMA:
        return {c: {f: 1.0/len(features) for f in features} for c in range(cfg.n_classes)}
    probs     = model.predict_proba(X_train[features].values)
    Xfull     = X_train[features].copy()
    prob_cols = []
    for c in range(cfg.n_classes):
        col = f"prob_cls{c}"
        Xfull[col] = probs[:, c]
        prob_cols.append(col)
    all_cols = features + prob_cols
    W = DagmaLinear(loss_type="l2").fit(
        Xfull[all_cols].values.astype(float), lambda1=cfg.dagma_lambda1_gamma)
    gamma_by_class = {}
    for c, col in enumerate(prob_cols):
        t       = all_cols.index(col)
        effects = {f: abs(float(W[i, t])) for i, f in enumerate(features)}
        gamma_by_class[c] = normalize_gamma_dict(effects, features)
    return gamma_by_class


# Dagma-dce

def _fit_dce_matrix(data_np: np.ndarray, cfg: Config, lambda1: float) -> np.ndarray:
    n_vars   = data_np.shape[1]
    hidden   = max(10, cfg.dce_hidden_mult * n_vars)
    data     = torch.tensor(data_np.astype(np.float64), dtype=torch.float64)
    eq_model = nonlinear_dce.DagmaMLP_DCE(dims=[n_vars, hidden, 1], bias=True)
    model_dce = nonlinear_dce.DagmaDCE(eq_model, use_mse_loss=True)
    W_tensor = model_dce.fit(
        data, lambda1=lambda1, lambda2=cfg.dce_lambda2,
        lr=cfg.dce_lr, mu_factor=cfg.dce_mu_factor, mu_init=cfg.dce_mu_init,
        T=cfg.dce_T, warm_iter=cfg.dce_warm_iter, max_iter=cfg.dce_max_iter,
    )
    return W_tensor.detach().cpu().numpy() if isinstance(W_tensor, torch.Tensor) else np.asarray(W_tensor)


def run_dce_graph(X_train: pd.DataFrame, features: List[str], cfg: Config) -> nx.DiGraph:
    if not HAS_DCE:
        return nx.DiGraph()
    print("  Fitting DAGMA-DCE graph on features only...")
    W = _fit_dce_matrix(X_train[features].values, cfg, cfg.dce_lambda1_graph)
    return build_graph_from_W(W, features, cfg.dce_graph_threshold)


def run_dce_gamma(X_train: pd.DataFrame, model, features: List[str],
                  cfg: Config) -> Dict[int, Dict[str, float]]:
    if not HAS_DCE:
        return {c: {f: 1.0/len(features) for f in features} for c in range(cfg.n_classes)}
    probs     = model.predict_proba(X_train[features].values)
    Xfull     = X_train[features].copy()
    prob_cols = []
    for c in range(cfg.n_classes):
        col = f"prob_cls{c}"
        Xfull[col] = probs[:, c]
        prob_cols.append(col)
    all_cols = features + prob_cols
    print("  Fitting DAGMA-DCE gamma on features + probability nodes...")
    W = _fit_dce_matrix(Xfull[all_cols].values, cfg, cfg.dce_lambda1_gamma)
    gamma_by_class = {}
    for c, col in enumerate(prob_cols):
        t       = all_cols.index(col)
        effects = {f: abs(float(W[i, t])) for i, f in enumerate(features)}
        gamma_by_class[c] = normalize_gamma_dict(effects, features)
    return gamma_by_class


# PC

def run_pc_graph(X_train: pd.DataFrame, features: List[str], cfg: Config) -> nx.DiGraph:
    cg    = pc(X_train[features].values.astype(float), alpha=cfg.pc_alpha,
               indep_test=fisherz, node_names=features, verbose=False, show_progress=False)
    adj   = cg.G.graph
    nodes = [n.get_name() for n in cg.G.get_nodes()]
    G     = nx.DiGraph()
    G.add_nodes_from(features)
    d = len(features)
    for i in range(d):
        for j in range(i + 1, d):
            if adj[i, j] != 0 or adj[j, i] != 0:
                if   adj[i, j] == 1 and adj[j, i] == -1: G.add_edge(nodes[i], nodes[j])
                elif adj[i, j] == -1 and adj[j, i] == 1: G.add_edge(nodes[j], nodes[i])
                else:                                      G.add_edge(nodes[i], nodes[j])
    return make_dag_greedy(G)


def run_pc_gamma(X_train: pd.DataFrame, model, features: List[str],
                 cfg: Config) -> Dict[int, Dict[str, float]]:
    probs          = model.predict_proba(X_train[features].values)
    gamma_by_class = {}
    for c in range(cfg.n_classes):
        Xfull         = X_train[features].copy()
        target        = f"prob_cls{c}"
        Xfull[target] = probs[:, c]
        all_cols      = features + [target]
        cg    = pc(Xfull[all_cols].values.astype(float), alpha=cfg.pc_alpha,
                   indep_test=fisherz, node_names=all_cols, verbose=False, show_progress=False)
        adj   = cg.G.graph
        nodes = [n.get_name() for n in cg.G.get_nodes()]
        t     = nodes.index(target)
        connected = [f for f in features
                     if adj[nodes.index(f), t] != 0 or adj[t, nodes.index(f)] != 0]
        gamma_by_class[c] = (
            {f: (1.0/len(connected) if f in connected else 0.0) for f in features}
            if connected else {f: 1.0/len(features) for f in features}
        )
    return gamma_by_class


# SEM

class LinearGaussianSEM:
    def __init__(self, dag: nx.DiGraph, features: List[str]):
        self.dag      = make_dag_greedy(dag)
        self.features = features
        self.topo     = list(nx.topological_sort(self.dag))
        self.mech: Dict = {}

    def fit(self, X: pd.DataFrame) -> "LinearGaussianSEM":
        for var in self.features:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {"parents": [], "intercept": float(X[var].mean()),
                                  "coef": None, "sigma": float(X[var].std(ddof=0))}
            else:
                lr    = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mech[var] = {"parents": parents, "intercept": float(lr.intercept_),
                                  "coef": lr.coef_.astype(float),
                                  "sigma": float(np.std(resid, ddof=0))}
        return self

    def sample_batch(self, S: set, x: pd.Series, M: int, rng: np.random.Generator) -> np.ndarray:
        X_out = np.empty((M, len(self.features)), dtype=float)
        idx   = {f: k for k, f in enumerate(self.features)}
        sim: Dict[str, np.ndarray] = {}
        for var in self.topo:
            vi = idx[var]
            if var in S:
                val = float(x[var]); X_out[:, vi] = val; sim[var] = np.full(M, val); continue
            m = self.mech[var]
            if not m["parents"]:
                mu = np.full(M, m["intercept"])
            else:
                for p in m["parents"]:
                    if p not in sim:
                        raise RuntimeError(
                            f"SEM topological order violation: parent '{p}' of '{var}' "
                            f"not yet simulated — DAG contains a cycle.")
                mu = m["intercept"] + np.stack([sim[p] for p in m["parents"]], axis=1) @ m["coef"]
            sigma        = m["sigma"]
            X_out[:, vi] = mu if sigma < 1e-10 else rng.normal(mu, sigma, size=M)
            sim[var]     = X_out[:, vi]
        return X_out


# Causal SHAP Explainer

class CausalShapExplainer:
    def __init__(self, model, X_train: pd.DataFrame, dag: nx.DiGraph,
                 gamma_by_class: Dict[int, Dict[str, float]],
                 features: List[str], cfg: Config):
        self.model          = model
        self.X_train        = X_train
        self.features       = features
        self.cfg            = cfg
        self.gamma_by_class = gamma_by_class
        self.sem            = LinearGaussianSEM(dag, features).fit(X_train)
        probs               = model.predict_proba(X_train[features].values)
        self.baseline_prob  = {c: float(np.mean(probs[:, c])) for c in range(cfg.n_classes)}
        self.feature_stds   = X_train[features].std().values.astype(float) + 1e-8

    def _value_sem(self, S: set, x: pd.Series, pred_cls: int,
                   M: int, rng: np.random.Generator) -> float:
        batch = self.sem.sample_batch(S, x, M, rng)
        return float(np.mean(self.model.predict_proba(batch)[:, pred_cls]))

    def explain_instance(self, x: pd.Series, M: Optional[int] = None,
                         n_perms: Optional[int] = None,
                         seed: Optional[int] = None) -> pd.Series:
        M       = self.cfg.causal_m     if M       is None else M
        n_perms = self.cfg.causal_perms if n_perms is None else n_perms
        rng     = np.random.default_rng(self.cfg.random_seed if seed is None else seed)
        pred_cls = int(self.model.predict(x.values.reshape(1, -1))[0])
        fx       = float(self.model.predict_proba(x.values.reshape(1, -1))[0, pred_cls])
        Ef       = self.baseline_prob[pred_cls]
        gamma    = self.gamma_by_class.get(pred_cls,
                       {f: 1.0/len(self.features) for f in self.features})
        phi = np.zeros(len(self.features), dtype=float)
        for _ in range(n_perms):
            perm = rng.permutation(self.features); S = set()
            v_prev = self._value_sem(S, x, pred_cls, M, rng)
            for feat in perm:
                Sn = S | {feat}; v_new = self._value_sem(Sn, x, pred_cls, M, rng)
                phi[self.features.index(feat)] += gamma[feat] * (v_new - v_prev)
                v_prev = v_new; S = Sn
        phi /= n_perms
        s = float(phi.sum())
        if abs(s) > 1e-12:
            phi = phi * ((fx - Ef) / s)
        return pd.Series(phi, index=self.features)

    def explain_global(self, X_test: pd.DataFrame,
                       K: Optional[int] = None) -> Tuple[pd.Series, pd.DataFrame, np.ndarray]:
        K        = self.cfg.causal_k if K is None else min(K, len(X_test))
        rng      = np.random.default_rng(self.cfg.random_seed)
        selected = rng.choice(len(X_test), size=K, replace=False)
        local    = []
        for cnt, i in enumerate(selected):
            if (cnt + 1) % 10 == 0:
                print(f"      instance {cnt+1}/{K} ...", flush=True)
            local.append(self.explain_instance(X_test.iloc[i],
                                               seed=self.cfg.random_seed + 1000 + cnt))
        local_df   = pd.DataFrame(local, index=selected)
        global_abs = local_df.abs().mean(axis=0)
        return global_abs, local_df, selected


# MC Shapley

class MCShapleyExplainer:
    def __init__(self, model, X_train: pd.DataFrame, features: List[str], cfg: Config):
        self.model        = model
        self.X_train      = X_train
        self.features     = features
        self.cfg          = cfg
        self.feature_stds = X_train[features].std().values.astype(float) + 1e-8

    def _utility(self, coalition: List[int], x: np.ndarray,
                 bg: np.ndarray, pred_cls: int) -> float:
        X_rep = bg.copy()
        for fi in coalition:
            X_rep[:, fi] = x[fi]
        return float(np.mean(self.model.predict_proba(X_rep)[:, pred_cls]))

    def explain_instance(self, x_series: pd.Series,
                         seed: Optional[int] = None) -> pd.Series:
        rng      = np.random.default_rng(self.cfg.random_seed if seed is None else seed)
        x        = x_series.values.astype(float)
        pred_cls = int(self.model.predict(x.reshape(1, -1))[0])
        bg_size  = min(self.cfg.mc_bg_size, len(self.X_train))
        bg       = self.X_train.values[rng.choice(len(self.X_train), size=bg_size, replace=False)].copy()
        phi      = np.zeros(len(self.features), dtype=float)
        for _ in range(self.cfg.mc_iterations):
            perm    = rng.permutation(len(self.features)); current = []; prev_v = self._utility(current, x, bg, pred_cls)
            for p in perm:
                new_current = current + [int(p)]; new_v = self._utility(new_current, x, bg, pred_cls)
                phi[p] += new_v - prev_v; current = new_current; prev_v = new_v
        phi /= self.cfg.mc_iterations
        return pd.Series(phi, index=self.features)

    def explain_global(self, X_test: pd.DataFrame,
                       K: Optional[int] = None) -> Tuple[pd.Series, pd.DataFrame, np.ndarray]:
        K        = self.cfg.mc_k if K is None else min(K, len(X_test))
        rng      = np.random.default_rng(self.cfg.random_seed)
        selected = rng.choice(len(X_test), size=K, replace=False)
        local    = []
        for cnt, i in enumerate(selected):
            if (cnt + 1) % 10 == 0:
                print(f"      MC Shapley {cnt+1}/{K} ...", flush=True)
            local.append(self.explain_instance(X_test.iloc[i],
                                               seed=self.cfg.random_seed + 2000 + cnt))
        local_df   = pd.DataFrame(local, index=selected)
        global_abs = local_df.abs().mean(axis=0)
        return global_abs, local_df, selected


# Evaluation metrics

def permutation_importance_distribution(model, X_test, y_test, features, cfg: Config) -> pd.Series:
    res  = permutation_importance(model, X_test[features].values, y_test,
                                  n_repeats=cfg.perm_repeats,
                                  random_state=cfg.random_seed, scoring="accuracy")
    vals = normalize_vec(np.abs(res.importances_mean))
    return pd.Series(vals, index=features)


def faithfulness_local_drop(
    model,
    X_test: pd.DataFrame,
    local_df: pd.DataFrame,
    selected_indices: np.ndarray,
    features: List[str],
    baseline_values: np.ndarray,
) -> Tuple[float, float, float]:
    """
    Compute three faithfulness metrics between |local phi| and |mean-sub probability drop|.

    Returns
    -------
    faith_spearman : Spearman rank correlation  — does rank order of attributions
                     match rank order of actual model impact?
    faith_pearson  : Pearson correlation         — is the linear relationship between
                     attribution magnitude and model impact proportional?
    faith_cosine   : Cosine similarity           — does the attribution vector point in
                     the same direction as the drop vector, scale-invariant?

    NOTE: mean-sub ablation is mechanistically closer to causal SHAP (which also
    substitutes values via SEM) than to MC Shapley (random background replacement).
    This may slightly favour causal SHAP faithfulness scores vs MC Shapley.
    """
    spearman_scores = []
    pearson_scores  = []
    cosine_scores   = []

    for row_pos, idx in enumerate(selected_indices):
        x        = X_test.iloc[idx].values.astype(float)
        pred_cls = int(model.predict(x.reshape(1, -1))[0])
        base_p   = float(model.predict_proba(x.reshape(1, -1))[0, pred_cls])

        drops = []
        for j in range(len(features)):
            x_mod    = x.copy()
            x_mod[j] = baseline_values[j]
            new_p    = float(model.predict_proba(x_mod.reshape(1, -1))[0, pred_cls])
            drops.append(base_p - new_p)

        attr  = local_df.iloc[row_pos][features].abs().values
        drops = np.abs(np.asarray(drops, dtype=float))

        sr = safe_spearman(attr, drops)
        pr = safe_pearson(attr, drops)
        cs = cosine(attr, drops)

        if not np.isnan(sr): spearman_scores.append(sr)
        if not np.isnan(pr): pearson_scores.append(pr)
        if not np.isnan(cs): cosine_scores.append(cs)

    faith_sp = float(np.mean(spearman_scores)) if spearman_scores else np.nan
    faith_pr = float(np.mean(pearson_scores))  if pearson_scores  else np.nan
    faith_cs = float(np.mean(cosine_scores))   if cosine_scores   else np.nan

    return faith_sp, faith_pr, faith_cs


def robustness(explainer, X_test: pd.DataFrame, features: List[str],
               cfg: Config) -> Tuple[float, float, float]:
    rng       = np.random.default_rng(cfg.random_seed)
    K         = min(cfg.robust_k, len(X_test))
    selected  = rng.choice(len(X_test), size=K, replace=False)
    feat_stds = explainer.feature_stds
    cos_l, rank_l, l2_l = [], [], []
    for k, idx in enumerate(selected):
        x0   = X_test.iloc[idx].copy()
        phi0 = explainer.explain_instance(x0, seed=cfg.random_seed + 3000 + k).abs().values
        for r in range(cfg.perturb_runs):
            xp    = x0.copy()
            xp[:] = xp.values + rng.normal(0, cfg.perturb_sigma, size=len(features)) * feat_stds
            phip  = explainer.explain_instance(xp, seed=cfg.random_seed + 4000 + 10*k + r).abs().values
            cos_l.append(cosine(phi0, phip))
            rk = safe_spearman(phi0, phip)
            if not np.isnan(rk): rank_l.append(rk)
            l2_l.append(float(np.linalg.norm(phi0 - phip)))
    return (float(np.mean(cos_l))  if cos_l  else np.nan,
            float(np.mean(rank_l)) if rank_l else np.nan,
            float(np.mean(l2_l))   if l2_l   else np.nan)


def evaluate_method(name: str, explainer, X_test: pd.DataFrame, model,
                    perm_imp: pd.Series, features: List[str], cfg: Config):
    t0 = time.time()
    global_abs, local_df, selected = explainer.explain_global(X_test)
    elapsed = time.time() - t0

    attr_norm       = pd.Series(normalize_vec(global_abs.values), index=features)
    pi              = perm_imp[features].values
    baseline_values = explainer.X_train[features].mean().values

    faith_sp, faith_pr, faith_cs = faithfulness_local_drop(
        model, X_test, local_df, selected, features, baseline_values)

    # Global alignment vs permutation importance
    align = safe_pearson(attr_norm.values, pi)
    cosv  = cosine(attr_norm.values, pi)

    # Robustness
    rob_cos, rank_rob, l2 = robustness(explainer, X_test, features, cfg)

    return {
        "Method":                  name,
        "Time_s":                  round(elapsed, 1),
        "Faith_Spearman":          round(faith_sp, 4) if not np.isnan(faith_sp) else np.nan,
        "Faith_Pearson":           round(faith_pr, 4) if not np.isnan(faith_pr) else np.nan,
        "Faith_Cosine":            round(faith_cs, 4) if not np.isnan(faith_cs) else np.nan,
        "Pred_Align_vs_PermImp":   round(align,    4) if not np.isnan(align)    else np.nan,
        "Attr_Cosine_vs_PermImp":  round(cosv,     4),
        "Rob_Cosine":              round(rob_cos,  4) if not np.isnan(rob_cos)  else np.nan,
        "Rank_Rob":                round(rank_rob, 4) if not np.isnan(rank_rob) else np.nan,
        "L2":                      round(l2,       4) if not np.isnan(l2)       else np.nan,
        **{f"attr_{f}": round(float(attr_norm[f]), 4) for f in features},
    }, attr_norm


def print_attr_table(label: str, attr_norm: pd.Series, perm_imp: pd.Series, features: List[str]):
    print(f"\n  {'─'*76}")
    print(f"  {label}")
    print(f"  {'─'*76}")
    print(f"  {'Feature':<34} {'Attr(norm)':>10} {'PermImp':>10} {'Delta':>10}")
    print(f"  {'─'*76}")
    for f in features:
        print(f"  {f:<34} {attr_norm[f]:>10.4f} {perm_imp[f]:>10.4f} {attr_norm[f]-perm_imp[f]:>+10.4f}")
    print(f"  {'─'*76}")


# Main

def main():
    set_global_seed(CFG.random_seed)

    print("=" * 90)
    print("Wine: DAGMA / DAGMA-DCE / PC / MC Shapley  — Faith(Spearman+Pearson+Cosine)")
    print("=" * 90)

    X_train, X_test, y_train, y_test, features = load_wine(CFG)
    print(f"Features ({len(features)}): {features}")
    print(f"Train={X_train.shape}, Test={X_test.shape}, classes={sorted(np.unique(y_train).tolist())}")

    model = fit_xgb(X_train, y_train, CFG)
    print(f"XGBoost train acc={np.mean(model.predict(X_train.values)==y_train):.4f}, "
          f"test acc={np.mean(model.predict(X_test.values)==y_test):.4f}")

    perm_imp = permutation_importance_distribution(model, X_test, y_test, features, CFG)
    print(f"Permutation importance top5: {dict(perm_imp.nlargest(5).round(4))}")

    results     = []
    attr_tables = {}

    # ── dagma ─────────────────────────────────────────────────────────────────
    if HAS_DAGMA:
        print("\n" + "─"*70)
        print("METHOD 1: Standard DAGMA -> Causal SHAP")
        print("─"*70)
        t0    = time.time()
        dag   = run_dagma_graph(X_train, features, CFG)
        gamma = run_dagma_gamma(X_train, model, features, CFG)
        print(f"  Setup time={time.time()-t0:.1f}s | edges={len(dag.edges())}")
        exp        = CausalShapExplainer(model, X_train, dag, gamma, features, CFG)
        row, attr  = evaluate_method("DAGMA Causal SHAP", exp, X_test, model, perm_imp, features, CFG)
        results.append(row); attr_tables[row["Method"]] = attr
        print(f"  Faith_Spearman={row['Faith_Spearman']}  Faith_Pearson={row['Faith_Pearson']}  Faith_Cosine={row['Faith_Cosine']}")
        print_attr_table(row["Method"], attr, perm_imp, features)

    # ── dagma-dce ─────────────────────────────────────────────────────────────
    if HAS_DCE:
        print("\n" + "─"*70)
        print("METHOD 2: DAGMA-DCE MLP -> Causal SHAP")
        print("─"*70)
        t0    = time.time()
        dag   = run_dce_graph(X_train, features, CFG)
        gamma = run_dce_gamma(X_train, model, features, CFG)
        print(f"  Setup time={time.time()-t0:.1f}s | edges={len(dag.edges())}")
        exp        = CausalShapExplainer(model, X_train, dag, gamma, features, CFG)
        row, attr  = evaluate_method("DAGMA-DCE Causal SHAP", exp, X_test, model, perm_imp, features, CFG)
        results.append(row); attr_tables[row["Method"]] = attr
        print(f"  Faith_Spearman={row['Faith_Spearman']}  Faith_Pearson={row['Faith_Pearson']}  Faith_Cosine={row['Faith_Cosine']}")
        print_attr_table(row["Method"], attr, perm_imp, features)

    # ── PC ────────────────────────────────────────────────────────────────────
    print("\n" + "─"*70)
    print("METHOD 3: PC -> Causal SHAP")
    print("─"*70)
    t0    = time.time()
    dag   = run_pc_graph(X_train, features, CFG)
    gamma = run_pc_gamma(X_train, model, features, CFG)
    print(f"  Setup time={time.time()-t0:.1f}s | edges={len(dag.edges())}")
    exp        = CausalShapExplainer(model, X_train, dag, gamma, features, CFG)
    row, attr  = evaluate_method("PC Causal SHAP", exp, X_test, model, perm_imp, features, CFG)
    results.append(row); attr_tables[row["Method"]] = attr
    print(f"  Faith_Spearman={row['Faith_Spearman']}  Faith_Pearson={row['Faith_Pearson']}  Faith_Cosine={row['Faith_Cosine']}")
    print_attr_table(row["Method"], attr, perm_imp, features)

    # ── MC Shapley ────────────────────────────────────────────────────────────
    print("\n" + "─"*70)
    print("METHOD 4: Monte Carlo Shapley")
    print("─"*70)
    exp        = MCShapleyExplainer(model, X_train, features, CFG)
    row, attr  = evaluate_method("MC Shapley", exp, X_test, model, perm_imp, features, CFG)
    results.append(row); attr_tables[row["Method"]] = attr
    print(f"  Faith_Spearman={row['Faith_Spearman']}  Faith_Pearson={row['Faith_Pearson']}  Faith_Cosine={row['Faith_Cosine']}")
    print_attr_table(row["Method"], attr, perm_imp, features)

    # ── Summary ───────────────────────────────────────────────────────────────
    df   = pd.DataFrame(results)
    disp = ["Method", "Time_s",
            "Faith_Spearman", "Faith_Pearson", "Faith_Cosine",
            "Pred_Align_vs_PermImp", "Attr_Cosine_vs_PermImp",
            "Rob_Cosine", "Rank_Rob", "L2"]

    print("\n" + "="*120)
    print("RESULTS SUMMARY — Wine")
    print("="*120)
    print(df[disp].to_string(index=False))

    print("\nMetric meanings:")
    print("  Faith_Spearman        : Spearman(|local phi|, |mean-sub drop|) — rank agreement.")
    print("  Faith_Pearson         : Pearson(|local phi|, |mean-sub drop|)  — linear magnitude agreement.")
    print("  Faith_Cosine          : Cosine(|local phi|, |mean-sub drop|)   — directional agreement, scale-invariant.")
    print("  Pred_Align_vs_PermImp : Pearson(global attr, perm importance).")
    print("  Attr_Cosine_vs_PermImp: Cosine(global attr, perm importance).")
    print("  Rob_Cosine            : Cosine under scale-aware input perturbation.")
    print("  Rank_Rob              : Spearman rank stability under perturbation.")
    print("  L2                    : Attribution drift under perturbation; lower is better.")
    print("\n  Note: Faith_* metrics use mean-sub ablation — slight upward bias for causal SHAP vs MC Shapley.")
    print("\nClaim boundary: model-output attribution only. Wine has no causal ground-truth.")

    out = "wine_causal_shap_results_full.csv"
    df.to_csv(out, index=False)
    print(f"\nSaved: {out}")


if __name__ == "__main__":
    main()

Wine: DAGMA / DAGMA-DCE / PC / MC Shapley  — Faith(Spearman+Pearson+Cosine)
Features (13): ['Alcohol', 'Malic acid', 'Ash', 'Alcalinity of ash', 'Magnesium', 'Total phenols', 'Flavanoids', 'Nonflavanoid phenols', 'Proanthocyanins', 'Color intensity', 'Hue', 'OD280/OD315 of diluted wines', 'Proline']
Train=(141, 13), Test=(36, 13), classes=[0, 1, 2]
XGBoost train acc=1.0000, test acc=1.0000
Permutation importance top5: {'Flavanoids': np.float64(0.3166), 'Proline': np.float64(0.2141), 'Color intensity': np.float64(0.1913), 'Alcohol': np.float64(0.1276), 'Hue': np.float64(0.0433)}

──────────────────────────────────────────────────────────────────────
METHOD 1: Standard DAGMA -> Causal SHAP
──────────────────────────────────────────────────────────────────────


  Setup time=3.9s | edges=14
      instance 10/30 ...
      instance 20/30 ...
      instance 30/30 ...
  Faith_Spearman=0.6263  Faith_Pearson=0.8097  Faith_Cosine=0.8397

  ────────────────────────────────────────────────────────────────────────────
  DAGMA Causal SHAP
  ────────────────────────────────────────────────────────────────────────────
  Feature                            Attr(norm)    PermImp      Delta
  ────────────────────────────────────────────────────────────────────────────
  Alcohol                                0.1585     0.1276    +0.0310
  Malic acid                             0.0365     0.0387    -0.0022
  Ash                                    0.0096     0.0182    -0.0086
  Alcalinity of ash                      0.0371     0.0000    +0.0371
  Magnesium                              0.0161     0.0159    +0.0001
  Total phenols                          0.0070     0.0000    +0.0070
  Flavanoids                             0.2501     0.3166    -0.0666
  Nonflavan

  Fitting DAGMA-DCE gamma on features + probability nodes...


  Setup time=1123.7s | edges=24
      instance 10/30 ...
      instance 20/30 ...
      instance 30/30 ...
  Faith_Spearman=0.5185  Faith_Pearson=0.5603  Faith_Cosine=0.6448

  ────────────────────────────────────────────────────────────────────────────
  DAGMA-DCE Causal SHAP
  ────────────────────────────────────────────────────────────────────────────
  Feature                            Attr(norm)    PermImp      Delta
  ────────────────────────────────────────────────────────────────────────────
  Alcohol                                0.3649     0.1276    +0.2374
  Malic acid                             0.0281     0.0387    -0.0106
  Ash                                    0.0067     0.0182    -0.0115
  Alcalinity of ash                      0.0113     0.0000    +0.0113
  Magnesium                              0.0047     0.0159    -0.0113
  Total phenols                          0.0075     0.0000    +0.0075
  Flavanoids                             0.1073     0.3166    -0.2094
  No